# FACE-04 — Hiệu chỉnh điểm GenD (calibration, KHÔNG dạy lại mô hình)

Notebook này chấm điểm bằng **GenD gốc** (đúng `face/gend.py`) rồi fit **calibrator** để điểm thô
thành xác suất, nhằm giảm chuyện người Việt thật bị đẩy sang bước đọc câu oan.

- **Lớp thật (dùng để fit):** người Việt thật FAS-VN `live` (Kaggle `maihongtng/face-anti-spoofing`,
  CC BY 4.0, ảnh tĩnh → GenD chấm 1 khung) + video gốc RTFS-10k (cùng nguồn với lớp giả).
- **VLR** (Zenodo 19705466, video bài giảng đã căn mặt) mặc định TẮT (`USE_VLR = False`). Bật thì
  chỉ báo phân bố điểm để THAM KHẢO, KHÔNG dùng để fit hay chọn gì (quyết định trưởng nhóm 20/10).
- **Lớp giả:** video ghép mặt RTFS-10k (Kaggle `stplusplus/rtfs-10k`, CC BY-SA 4.0). Nhóm KHÔNG tự
  tạo deepfake. Không có deepfake người Việt — giới hạn này được ghi vào `card.md`.
- Mọi mẫu đi cùng một đường: cắt khung vuông quanh mặt → 320×320 → H.264 CRF 23, 25 fps.
- Chia theo nhóm (người/video gốc/ảnh donor) thành `calib_fit` / `calib_ltt` / `test`.
  Calibrator chỉ fit trên `calib_fit`. `calib_ltt` chỉ lưu điểm (ROUT-03 chọn ngưỡng sau).
  **Tập test KHÔNG được đọc** (hàm test tắt mặc định).

Hướng dẫn chạy cho người không chuyên: `ml/notebooks/README-gend.md`.
**Chạy SMOKE = True trước (≈ 10–15 phút), xong mới đặt SMOKE = False và Run All.**

In [ ]:
# ===================== CẤU HÌNH (chỉ sửa ô này) =====================
SMOKE = True  # True: chạy thử nhỏ để kiểm. False: chạy thật.
SEED = 20261019
DEVICE = "cuda"  # "cuda" hoặc "cpu"

# VLR: chỉ tham khảo, không dùng để fit. Mặc định TẮT cho nhanh (bật thì tải thêm từ Zenodo).
USE_VLR = False

# Số mẫu khi chạy THẬT (SMOKE = False). None = lấy hết.
VLR_PARTS = ["part1", "part2", "part3", "part4", "part5", "part6"]  # bỏ part7 theo trưởng nhóm
VLR_VIDEOS_PER_PART = 40
VLR_CLIPS_PER_VIDEO = 3
N_FAS_LIVE = 800
N_RTFS_REAL = 400
N_RTFS_FAKE = 1200
ARCFACE_THRESHOLD = 0.35  # gom người lỏng tay (thà gộp nhầm còn hơn tách sót)

if SMOKE:
    VLR_PARTS = ["part1"]
    VLR_VIDEOS_PER_PART = 8
    VLR_CLIPS_PER_VIDEO = 2
    N_FAS_LIVE = 24
    N_RTFS_REAL = 12
    N_RTFS_FAKE = 24

# Tập test: chỉ trưởng nhóm bật, MỘT lần, sau khi đã chốt calibrator.
RUN_TEST = False
TEST_CONFIRMATION = ""

In [ ]:
# ===================== CODE NHÚNG TỪ REPO (KHÔNG SỬA TAY) =====================
# Sinh bởi scripts/build_gend_notebooks.py. Tiền xử lý GenD lấy NGUYÊN VĂN từ
# ml/src/vericlue_ml/face/{gend,pad,frames}.py để giống hệt hệ thống đang chạy.
SOURCE_COMMIT = "d03b855ac76cc91764e4145986c5085ab99dc7bf"
SOURCE_DIRTY = False  # True = lúc sinh notebook có file nhúng chưa commit
import hashlib as _hashlib
import os as _os
import sys as _sys
from pathlib import Path as _Path

_EMBEDDED = {
    'models/manifest.yaml': ('d67a1280cc0cad9bba597dcd1dbbb12a2fffb125374a5e03be4dbb346452d12f', '# DATA-02: pin nguồn; sha256 ghi sau tải/kiểm byte thực. Không commit weights.\nschema_version: 1\nmodels:\n- id: minifasnet\n  purpose: PAD (ảnh in, màn hình) — chạy nhanh\n  source:\n    type: hf\n    repo: yakhyo/uniface-weights\n    revision: bfcc7d48ea8412d55de406cecf96b65b3ca720b2\n  license: Apache-2.0\n  license_url: https://huggingface.co/yakhyo/uniface-weights/blob/bfcc7d48ea8412d55de406cecf96b65b3ca720b2/README.md\n  license_status: verified\n  owner: M3\n  status: ready\n  files:\n  - path: minifasnet_v2.onnx\n    url: https://huggingface.co/yakhyo/uniface-weights/resolve/bfcc7d48ea8412d55de406cecf96b65b3ca720b2/minifasnet_v2.onnx?download=true\n    size_bytes: 1743581\n    sha256: b32929adc2d9c34b9486f8c4c7bc97c1b69bc0ea9befefc380e4faae4e463907\n    upstream_sha256: b32929adc2d9c34b9486f8c4c7bc97c1b69bc0ea9befefc380e4faae4e463907\n- id: gend-clip-l14\n  purpose: Deepfake khuôn mặt — mô hình chính\n  source:\n    type: hf\n    repo: yermandy/GenD_CLIP_L_14\n    revision: 891ce014a0308386c4d7d25b3dcf436a22db5504\n  license: MIT\n  license_url: https://huggingface.co/yermandy/GenD_CLIP_L_14/blob/891ce014a0308386c4d7d25b3dcf436a22db5504/README.md\n  license_status: verified\n  owner: M3\n  status: ready\n  files:\n  - path: config.json\n    url: https://huggingface.co/yermandy/GenD_CLIP_L_14/resolve/891ce014a0308386c4d7d25b3dcf436a22db5504/config.json?download=true\n    size_bytes: 193\n    sha256: 7f2761e13678191774a152121f1630d19baa33a74216dd264d54445d96b16229\n    upstream_sha256: null\n  - path: model.safetensors\n    url: https://huggingface.co/yermandy/GenD_CLIP_L_14/resolve/891ce014a0308386c4d7d25b3dcf436a22db5504/model.safetensors?download=true\n    size_bytes: 1215928160\n    sha256: d76f0bdfd74a29fe1b1c1b84a80ac92486993e426878e8c7a3944281fbb96833\n    upstream_sha256: d76f0bdfd74a29fe1b1c1b84a80ac92486993e426878e8c7a3944281fbb96833\n  dependencies:\n  - clip-vit-large-patch14\n  - scrfd-10g\n  note: Chỉ weights/config; loader/tiền xử lý repo gốc thực hiện ở FACE-02, không\n    tải remote Python code.\n- id: scrfd-10g\n  purpose: Dò và căn chỉnh khuôn mặt cho GenD\n  source:\n    type: hf\n    repo: yakhyo/uniface-weights\n    revision: bfcc7d48ea8412d55de406cecf96b65b3ca720b2\n  license: nghiên cứu phi thương mại\n  license_url: https://huggingface.co/yakhyo/uniface-weights/blob/bfcc7d48ea8412d55de406cecf96b65b3ca720b2/README.md\n  license_status: accepted-research\n  owner: M3\n  status: ready\n  files:\n  - path: det_10g.onnx\n    url: https://huggingface.co/yakhyo/uniface-weights/resolve/bfcc7d48ea8412d55de406cecf96b65b3ca720b2/scrfd_10g.onnx?download=true\n    size_bytes: 16923827\n    sha256: 5838f7fe053675b1c7a08b633df49e7af5495cee0493c7dcf6697200b85b5b91\n    upstream_sha256: 5838f7fe053675b1c7a08b633df49e7af5495cee0493c7dcf6697200b85b5b91\n  artifact: SCRFD buffalo_l det_10g.onnx\n  note: UniFace scrfd_10g.onnx byte-identical det_10g.onnx gốc; chỉ nghiên cứu phi\n    thương mại.\n  identity_evidence:\n    gend_source: https://github.com/yermandy/GenD/blob/387a42266dd385fbe8f7626c5d3aa03eea3bfaab/src/retinaface.py\n    official_archive: https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip\n    original_size_bytes: 16923827\n    original_sha256: 5838f7fe053675b1c7a08b633df49e7af5495cee0493c7dcf6697200b85b5b91\n- id: antideepfake-mms300m\n  purpose: Giọng giả — mô hình nền\n  source:\n    type: hf\n    repo: nii-yamagishilab/mms-300m-anti-deepfake\n    revision: 7928e119e09f616b2ce698892e00ca22dbaa40bf\n  license: CC-BY-NC-SA-4.0\n  license_url: https://github.com/nii-yamagishilab/AntiDeepfake/blob/0dea622bde8f064c8ee5a557f2598643123fc6b6/LICENSE-CHECKPOINT\n  license_status: accepted-research\n  owner: M4\n  status: ready\n  files:\n  - path: config.json\n    url: https://huggingface.co/nii-yamagishilab/mms-300m-anti-deepfake/resolve/7928e119e09f616b2ce698892e00ca22dbaa40bf/config.json?download=true\n    size_bytes: 2046\n    sha256: be5a16e7bf332832f0a5583357ed37dd67cc4a39abd24832cf6a74a8217b046c\n    upstream_sha256: null\n  - path: model.safetensors\n    url: https://huggingface.co/nii-yamagishilab/mms-300m-anti-deepfake/resolve/7928e119e09f616b2ce698892e00ca22dbaa40bf/model.safetensors?download=true\n    size_bytes: 1269622944\n    sha256: 9bd5d9785bf72f91dfca8508651354946c4b307921cb358505e1b94035102224\n    upstream_sha256: 9bd5d9785bf72f91dfca8508651354946c4b307921cb358505e1b94035102224\n- id: phowhisper\n  purpose: Nhận dạng câu đọc tiếng Việt\n  source:\n    type: hf\n    repo: vinai/PhoWhisper-small\n    revision: a86b604c346caf7148c37512eafe783a16420adb\n  license: BSD-3-Clause\n  license_url: https://huggingface.co/vinai/PhoWhisper-small/blob/a86b604c346caf7148c37512eafe783a16420adb/README.md\n  license_status: verified\n  owner: M4\n  status: ready\n  files:\n  - path: config.json\n    url: https://huggingface.co/vinai/PhoWhisper-small/resolve/a86b604c346caf7148c37512eafe783a16420adb/config.json?download=true\n    size_bytes: 1329\n    sha256: c9d7e721876727d437cfb145229936926b439d1ab483db572e5f9b5fb6b6d0bf\n    upstream_sha256: null\n  - path: generation_config.json\n    url: https://huggingface.co/vinai/PhoWhisper-small/resolve/a86b604c346caf7148c37512eafe783a16420adb/generation_config.json?download=true\n    size_bytes: 3832\n    sha256: ba8dbcc6416b036d6f596ab70619bb713f0e996a0bbd142cc8e8eea80b93ebd7\n    upstream_sha256: null\n  - path: preprocessor_config.json\n    url: https://huggingface.co/vinai/PhoWhisper-small/resolve/a86b604c346caf7148c37512eafe783a16420adb/preprocessor_config.json?download=true\n    size_bytes: 339\n    sha256: a6a76d28c93edb273669eb9e0b0636a2bddbb1272c3261e47b7ca6dfdbac1b8d\n    upstream_sha256: null\n  - path: tokenizer.json\n    url: https://huggingface.co/vinai/PhoWhisper-small/resolve/a86b604c346caf7148c37512eafe783a16420adb/tokenizer.json?download=true\n    size_bytes: 2204003\n    sha256: c218b12acbc2168cad951ae8a8c8d345df09dfa63b787214d4ac27762170fd7b\n    upstream_sha256: null\n  - path: tokenizer_config.json\n    url: https://huggingface.co/vinai/PhoWhisper-small/resolve/a86b604c346caf7148c37512eafe783a16420adb/tokenizer_config.json?download=true\n    size_bytes: 805\n    sha256: 14f84bdf4b9ecdbd4738ddc81c17a1baedfc02bb93c6e049c951e15a1b40b70d\n    upstream_sha256: null\n  - path: special_tokens_map.json\n    url: https://huggingface.co/vinai/PhoWhisper-small/resolve/a86b604c346caf7148c37512eafe783a16420adb/special_tokens_map.json?download=true\n    size_bytes: 2077\n    sha256: b0aac6950381c4ae2f60b3f0fab04bcfe399a6a320cb3e4c7aee2d52924d733e\n    upstream_sha256: null\n  - path: added_tokens.json\n    url: https://huggingface.co/vinai/PhoWhisper-small/resolve/a86b604c346caf7148c37512eafe783a16420adb/added_tokens.json?download=true\n    size_bytes: 2082\n    sha256: ce949fe720c14311cb6c446e69cfe340dc669d7b006077a6feed6ae571dd7e88\n    upstream_sha256: null\n  - path: merges.txt\n    url: https://huggingface.co/vinai/PhoWhisper-small/resolve/a86b604c346caf7148c37512eafe783a16420adb/merges.txt?download=true\n    size_bytes: 493864\n    sha256: 78f27f801feb4283ae969d4cadacb28fc051b655647402b1a252c7079ea11c46\n    upstream_sha256: null\n  - path: vocab.json\n    url: https://huggingface.co/vinai/PhoWhisper-small/resolve/a86b604c346caf7148c37512eafe783a16420adb/vocab.json?download=true\n    size_bytes: 835550\n    sha256: 8f680bba319e01a653d2e8a5dbc17a9157179e0576e6ce74ce0c06356c6e24f9\n    upstream_sha256: null\n  - path: normalizer.json\n    url: https://huggingface.co/vinai/PhoWhisper-small/resolve/a86b604c346caf7148c37512eafe783a16420adb/normalizer.json?download=true\n    size_bytes: 52666\n    sha256: bf1c507dc8724ca9cf9903640dacfb69dae2f00edee4f21ceba106a7392f26dd\n    upstream_sha256: null\n  - path: pytorch_model.bin\n    url: https://huggingface.co/vinai/PhoWhisper-small/resolve/a86b604c346caf7148c37512eafe783a16420adb/pytorch_model.bin?download=true\n    size_bytes: 967102729\n    sha256: 9ec6f07c5bd321fc8f477cf778c43c3fd94b0e33a9f20283fc61cc6d22cbfded\n    upstream_sha256: 9ec6f07c5bd321fc8f477cf778c43c3fd94b0e33a9f20283fc61cc6d22cbfded\n- id: ctc-aligner\n  purpose: Căn chỉnh câu đề bài theo thời gian\n  source:\n    type: hf\n    repo: MahmoudAshraf/mms-300m-1130-forced-aligner\n    revision: 49402e9577b1158620820667c218cd494cc44486\n  license: CC-BY-NC-4.0\n  license_url: https://huggingface.co/MahmoudAshraf/mms-300m-1130-forced-aligner/blob/49402e9577b1158620820667c218cd494cc44486/README.md\n  license_status: accepted-research\n  owner: M4\n  status: ready\n  files:\n  - path: config.json\n    url: https://huggingface.co/MahmoudAshraf/mms-300m-1130-forced-aligner/resolve/49402e9577b1158620820667c218cd494cc44486/config.json?download=true\n    size_bytes: 2076\n    sha256: 2af7f73073d212b0ef91d3c73839355ca504c24a6b361c58a9e4e64210376ba9\n    upstream_sha256: null\n  - path: preprocessor_config.json\n    url: https://huggingface.co/MahmoudAshraf/mms-300m-1130-forced-aligner/resolve/49402e9577b1158620820667c218cd494cc44486/preprocessor_config.json?download=true\n    size_bytes: 211\n    sha256: 6334d6e0c5f2084c9a99b85ddff243cbc79dbaa4aa790bcddf8c41c496fab6fb\n    upstream_sha256: null\n  - path: special_tokens_map.json\n    url: https://huggingface.co/MahmoudAshraf/mms-300m-1130-forced-aligner/resolve/49402e9577b1158620820667c218cd494cc44486/special_tokens_map.json?download=true\n    size_bytes: 74\n    sha256: 5e4d1f5e759d74cb1c2fe1d165cfc62b5237aa904de759380cd6f43042eec723\n    upstream_sha256: null\n  - path: tokenizer_config.json\n    url: https://huggingface.co/MahmoudAshraf/mms-300m-1130-forced-aligner/resolve/49402e9577b1158620820667c218cd494cc44486/tokenizer_config.json?download=true\n    size_bytes: 1047\n    sha256: e0fe11ce3a3231f3147b4539a93b30a9aefaca0c2717fbd35881695610417497\n    upstream_sha256: null\n  - path: vocab.json\n    url: https://huggingface.co/MahmoudAshraf/mms-300m-1130-forced-aligner/resolve/49402e9577b1158620820667c218cd494cc44486/vocab.json?download=true\n    size_bytes: 286\n    sha256: 6b6f1632d08b489faf2c954eb70e71f39d0861053feb9ae40da59cfa8d0ef5bb\n    upstream_sha256: null\n  - path: model.safetensors\n    url: https://huggingface.co/MahmoudAshraf/mms-300m-1130-forced-aligner/resolve/49402e9577b1158620820667c218cd494cc44486/model.safetensors?download=true\n    size_bytes: 1261930388\n    sha256: 9aa5229a1af4d7714285cfa14bcaab6af2a39a9e810b475722b24975d5cb1dab\n    upstream_sha256: 9aa5229a1af4d7714285cfa14bcaab6af2a39a9e810b475722b24975d5cb1dab\n- id: syncnet\n  purpose: Khớp môi–âm\n  source:\n    type: hf\n    repo: ByteDance/LatentSync-1.6\n    revision: c42c7e6c8e9c213626389fa7d9a3c444b8536353\n  license: OpenRAIL++\n  license_url: https://huggingface.co/ByteDance/LatentSync-1.6/blob/c42c7e6c8e9c213626389fa7d9a3c444b8536353/README.md\n  license_status: accepted-research\n  owner: M4\n  status: ready\n  files:\n  - path: stable_syncnet.pt\n    url: https://huggingface.co/ByteDance/LatentSync-1.6/resolve/c42c7e6c8e9c213626389fa7d9a3c444b8536353/stable_syncnet.pt?download=true\n    size_bytes: 1605328746\n    sha256: 77678d13861a02d6e83d0b169962175b6266be773f1f6f85fda2b0182e225118\n    upstream_sha256: 77678d13861a02d6e83d0b169962175b6266be773f1f6f85fda2b0182e225118\n- id: audioseal-detector\n  purpose: Dò thủy vân AudioSeal\n  source:\n    type: hf\n    repo: facebook/audioseal\n    revision: 3c19eba53390776cf2cc9ed5f6c9ac67ce72ecba\n  license: MIT\n  license_url: https://huggingface.co/facebook/audioseal/blob/3c19eba53390776cf2cc9ed5f6c9ac67ce72ecba/README.md\n  license_status: verified\n  owner: M4\n  status: ready\n  files:\n  - path: detector_base.pth\n    url: https://huggingface.co/facebook/audioseal/resolve/3c19eba53390776cf2cc9ed5f6c9ac67ce72ecba/detector_base.pth?download=true\n    size_bytes: 34667641\n    sha256: 8a78e8a83584113523e161fc599fcab10fd0e94c04d2eb9d2fa1e9ec91ab69d9\n    upstream_sha256: 8a78e8a83584113523e161fc599fcab10fd0e94c04d2eb9d2fa1e9ec91ab69d9\n- id: clip-vit-large-patch14\n  purpose: CLIP nền/processor cho GenD — dependency offline\n  source:\n    type: hf\n    repo: openai/clip-vit-large-patch14\n    revision: 32bd64288804d66eefd0ccbe215aa642df71cc41\n  license: MIT\n  license_url: https://github.com/openai/CLIP/blob/d05afc436d78f1c48dc0dbf8e5980a9d471f35f6/LICENSE\n  license_status: accepted-research\n  owner: M3\n  status: ready\n  files:\n  - path: config.json\n    url: https://huggingface.co/openai/clip-vit-large-patch14/resolve/32bd64288804d66eefd0ccbe215aa642df71cc41/config.json?download=true\n    size_bytes: 4519\n    sha256: 8a09b467700c58138c29d53c605b34ebc69beaadd13274a8a2af8ad2c2f4032a\n    upstream_sha256: null\n  - path: preprocessor_config.json\n    url: https://huggingface.co/openai/clip-vit-large-patch14/resolve/32bd64288804d66eefd0ccbe215aa642df71cc41/preprocessor_config.json?download=true\n    size_bytes: 316\n    sha256: 910e70b3956ac9879ebc90b22fb3bc8a75b6a0677814500101a4c072bd7857bd\n    upstream_sha256: null\n  - path: tokenizer.json\n    url: https://huggingface.co/openai/clip-vit-large-patch14/resolve/32bd64288804d66eefd0ccbe215aa642df71cc41/tokenizer.json?download=true\n    size_bytes: 2224003\n    sha256: a83e0809aa4c3af7208b2df632a7a69668c6d48775b3c3fe4e1b1199d1f8b8f4\n    upstream_sha256: null\n  - path: tokenizer_config.json\n    url: https://huggingface.co/openai/clip-vit-large-patch14/resolve/32bd64288804d66eefd0ccbe215aa642df71cc41/tokenizer_config.json?download=true\n    size_bytes: 905\n    sha256: deef455e52fa5e8151e339add0582e4235f066009601360999d3a9cda83b1129\n    upstream_sha256: null\n  - path: special_tokens_map.json\n    url: https://huggingface.co/openai/clip-vit-large-patch14/resolve/32bd64288804d66eefd0ccbe215aa642df71cc41/special_tokens_map.json?download=true\n    size_bytes: 389\n    sha256: f8c0d6c39aee3f8431078ef6646567b0aba7f2246e9c54b8b99d55c22b707cbf\n    upstream_sha256: null\n  - path: merges.txt\n    url: https://huggingface.co/openai/clip-vit-large-patch14/resolve/32bd64288804d66eefd0ccbe215aa642df71cc41/merges.txt?download=true\n    size_bytes: 524619\n    sha256: 9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a\n    upstream_sha256: null\n  - path: vocab.json\n    url: https://huggingface.co/openai/clip-vit-large-patch14/resolve/32bd64288804d66eefd0ccbe215aa642df71cc41/vocab.json?download=true\n    size_bytes: 961143\n    sha256: 3f0c4f7d2086b61b38487075278ea9ed04edb53a03cbb045b86c27190fa8fb69\n    upstream_sha256: null\n  - path: model.safetensors\n    url: https://huggingface.co/openai/clip-vit-large-patch14/resolve/32bd64288804d66eefd0ccbe215aa642df71cc41/model.safetensors?download=true\n    size_bytes: 1710540580\n    sha256: a2bf730a0c7debf160f7a6b50b3aaf3703e7e88ac73de7a314903141db026dcb\n    upstream_sha256: a2bf730a0c7debf160f7a6b50b3aaf3703e7e88ac73de7a314903141db026dcb\n  license_scope: MIT của repo/software openai/CLIP; HF card không khai báo riêng weights\n  note: Trưởng nhóm 15/10 chốt nguồn giấy phép openai/CLIP, không chặn; giữ rõ phạm\n    vi chứng cứ.\n- id: sbi-ffc23\n  purpose: Deepfake khuôn mặt — dự phòng, nguồn heatmap\n  source:\n    type: gdrive\n    file_id: 1X0-NYT8KPursLZZdxduRQju6E52hauV0\n    revision: chưa pin\n    reference_repo_commit: d6d9a351f32334ae9a3a250872c05836e0dd0f2c\n  license: research-only (nc)\n  license_url: https://github.com/mapooon/SelfBlendedImages/blob/d6d9a351f32334ae9a3a250872c05836e0dd0f2c/LICENSE\n  license_status: accepted-research\n  owner: M3\n  status: deferred\n  note: Chưa pin file Drive/size/hash; mỗi máy tải nguồn gốc, KHÔNG mirror; theo dõi\n    FACE-03, không chặn DATA-02. --only sbi-ffc23 chỉ thử nguồn gốc, lỗi HTML/quota\n    báo rõ.\n  files:\n  - path: FFc23.tar\n    url: https://drive.google.com/uc?export=download&id=1X0-NYT8KPursLZZdxduRQju6E52hauV0\n    size_bytes: null\n    sha256: null\n    upstream_sha256: null\n- id: perth\n  purpose: Dò thủy vân PerTh (Resemble / Chatterbox)\n  source:\n    type: http\n    repo: resemble-ai/Perth\n    revision: ff1c8ac55a976971245cdd53c18d6131ca00d993\n  license: chờ xác minh quyền trọng số — FOR-03\n  license_url: https://github.com/resemble-ai/Perth/blob/ff1c8ac55a976971245cdd53c18d6131ca00d993/LICENSE\n  license_status: pending\n  owner: M4\n  status: blocked\n  note: Chờ xác minh quyền trọng số — FOR-03; không tải.\n  files:\n  - path: perth_net_250000.pth.tar\n    url: https://raw.githubusercontent.com/resemble-ai/Perth/ff1c8ac55a976971245cdd53c18d6131ca00d993/src/perth/perth_net/pretrained/implicit/perth_net_250000.pth.tar\n    size_bytes: 37429684\n    sha256: null\n    upstream_sha256: null\n'),
    'nb_common.py': ('ed22e8867830b0eb49b0fae282e0d87dc8c4a480374009f1fd7ab0a03c2f9f2f', '"""Tiện ích dùng chung cho notebook GenD (Kaggle / Colab / máy Linux). Được nhúng vào notebook.\n\nKhông phải code sản phẩm: chỉ nhận diện nền tảng, cài gói, chạy lệnh có ghi log, đo VRAM, tải file có\nkiểm SHA-256 và quản lý ngân sách thời gian của phiên.\n"""\n\nfrom __future__ import annotations\n\nimport glob\nimport hashlib\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport tarfile\nimport threading\nimport time\nfrom pathlib import Path\n\nGB = 1024**3\n\n\ndef detect_platform() -> str:\n    """\'kaggle\' | \'colab\' | \'local\'."""\n    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").is_dir():\n        return "kaggle"\n    if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):\n        return "colab"\n    return "local"\n\n\ndef default_dirs(platform: str, name: str) -> tuple[Path, Path]:\n    """(OUT: kết quả cần GIỮ, WORK: nháp có thể mất). Biến môi trường VC_OUT/VC_WORK ghi đè."""\n    if os.environ.get("VC_OUT"):\n        out = Path(os.environ["VC_OUT"])\n    elif platform == "kaggle":\n        out = Path("/kaggle/working") / name\n    elif platform == "colab":\n        out = Path("/content/drive/MyDrive/vericlue") / name\n    else:\n        out = Path.cwd() / "vc_out" / name\n    if os.environ.get("VC_WORK"):\n        work = Path(os.environ["VC_WORK"])\n    elif platform == "kaggle":\n        work = Path("/kaggle/tmp/vc_work") if Path("/kaggle/tmp").is_dir() else Path("/tmp/vc_work")\n    elif platform == "colab":\n        work = Path("/content/vc_work")\n    else:\n        work = Path.cwd() / "vc_work"\n    out.mkdir(parents=True, exist_ok=True)\n    work.mkdir(parents=True, exist_ok=True)\n    return out, work\n\n\ndef mount_colab_drive() -> None:\n    from google.colab import drive  # type: ignore[import-not-found]\n\n    drive.mount("/content/drive")\n\n\ndef _has_pip(python: str) -> bool:\n    return subprocess.run([python, "-m", "pip", "--version"], capture_output=True).returncode == 0\n\n\ndef pip_install(packages: list[str], python: str | None = None, extra: list[str] | None = None):\n    """Cài gói vào môi trường `python` (mặc định kernel). Có pip thì dùng pip, không thì `uv pip`."""\n    python = python or sys.executable\n    if _has_pip(python):\n        cmd = [python, "-m", "pip", "install", "-q", *packages, *(extra or [])]\n    else:\n        cmd = [ensure_uv(), "pip", "install", "-q", "-p", python, *packages, *(extra or [])]\n    run(cmd)\n\n\ndef ensure_import(module: str, package: str | None = None) -> None:\n    """Thiếu module trong kernel thì cài (không đụng gói đã có sẵn của Kaggle/Colab)."""\n    import importlib\n\n    try:\n        importlib.import_module(module)\n    except ImportError:\n        pip_install([package or module])\n\n\ndef ensure_uv() -> str:\n    found = shutil.which("uv")\n    if found:\n        return found\n    local = Path.home() / ".local" / "bin" / "uv"\n    if local.is_file():\n        return str(local)\n    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)\n    found = shutil.which("uv") or str(Path(sys.executable).parent / "uv")\n    return found\n\n\ndef run(cmd, *, cwd=None, env=None, log: Path | None = None, timeout: float | None = None):\n    """Chạy lệnh; lỗi thì ném RuntimeError kèm 40 dòng cuối log (tiếng Việt)."""\n    cmd = [str(c) for c in cmd]\n    if log is not None:\n        log.parent.mkdir(parents=True, exist_ok=True)\n    started = time.time()\n    with open(log, "a", encoding="utf-8") if log else open(os.devnull, "w") as handle:\n        handle.write(f"\\n$ {\' \'.join(cmd)}\\n")\n        handle.flush()\n        process = subprocess.run(\n            cmd,\n            cwd=cwd,\n            env=env,\n            stdout=subprocess.PIPE,\n            stderr=subprocess.STDOUT,\n            text=True,\n            errors="replace",\n            timeout=timeout,\n        )\n        handle.write(process.stdout or "")\n    if process.returncode != 0:\n        tail = "\\n".join((process.stdout or "").splitlines()[-40:])\n        raise RuntimeError(\n            f"Lệnh lỗi (mã {process.returncode}, {time.time() - started:.0f} s): {\' \'.join(cmd[:6])} …\\n{tail}"\n        )\n    return process\n\n\ndef gpu_info() -> dict:\n    try:\n        out = (\n            subprocess.run(\n                [\n                    "nvidia-smi",\n                    "--query-gpu=name,memory.total,driver_version",\n                    "--format=csv,noheader,nounits",\n                ],\n                capture_output=True,\n                text=True,\n                check=True,\n            )\n            .stdout.strip()\n            .splitlines()[0]\n        )\n        name, total, driver = (x.strip() for x in out.split(","))\n        return {"name": name, "total_mb": int(float(total)), "driver": driver}\n    except Exception:  # noqa: BLE001\n        return {"name": "none", "total_mb": 0, "driver": ""}\n\n\ndef gpu_used_mb() -> int:\n    try:\n        out = subprocess.run(\n            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],\n            capture_output=True,\n            text=True,\n            check=True,\n        ).stdout\n        return int(float(out.strip().splitlines()[0]))\n    except Exception:  # noqa: BLE001\n        return 0\n\n\nclass GpuMonitor:\n    """Đo đỉnh VRAM toàn GPU (nvidia-smi) trong khi một lệnh chạy; `peak_mb` trừ mức nền lúc bắt đầu."""\n\n    def __init__(self, interval: float = 0.5) -> None:\n        self.interval = interval\n        self.baseline = 0\n        self.peak = 0\n        self._stop = threading.Event()\n        self._thread: threading.Thread | None = None\n\n    def __enter__(self) -> GpuMonitor:\n        self.baseline = gpu_used_mb()\n        self.peak = self.baseline\n        self.started = time.time()\n        self._thread = threading.Thread(target=self._poll, daemon=True)\n        self._thread.start()\n        return self\n\n    def _poll(self) -> None:\n        while not self._stop.is_set():\n            self.peak = max(self.peak, gpu_used_mb())\n            self._stop.wait(self.interval)\n\n    def __exit__(self, *exc) -> None:\n        self._stop.set()\n        if self._thread:\n            self._thread.join(timeout=5)\n        self.seconds = time.time() - self.started\n\n    @property\n    def peak_mb(self) -> int:\n        return max(0, self.peak - self.baseline)\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with open(path, "rb") as handle:\n        for block in iter(lambda: handle.read(1 << 20), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef download(url: str, dst: Path, sha256: str | None = None, retries: int = 4) -> Path:\n    """Tải về `dst` (bỏ qua nếu đã có và đúng SHA-256). Sai SHA-256 thì DỪNG."""\n    import requests\n\n    dst = Path(dst)\n    if dst.is_file() and (sha256 is None or sha256_file(dst) == sha256):\n        return dst\n    dst.parent.mkdir(parents=True, exist_ok=True)\n    tmp = dst.with_suffix(dst.suffix + ".part")\n    for attempt in range(retries):\n        try:\n            with requests.get(url, stream=True, timeout=120) as response:\n                response.raise_for_status()\n                with open(tmp, "wb") as handle:\n                    for chunk in response.iter_content(1 << 20):\n                        handle.write(chunk)\n            break\n        except Exception as exc:  # noqa: BLE001\n            if attempt == retries - 1:\n                raise RuntimeError(f"Không tải được {url}: {exc}") from exc\n            time.sleep(5 * (attempt + 1))\n    if sha256 is not None:\n        actual = sha256_file(tmp)\n        if actual != sha256:\n            tmp.unlink(missing_ok=True)\n            raise RuntimeError(f"SHA-256 sai cho {dst.name}: {actual} ≠ {sha256}")\n    os.replace(tmp, dst)\n    return dst\n\n\ndef manifest_files(manifest: dict, model_id: str) -> tuple[dict, list[dict]]:\n    entry = next(m for m in manifest["models"] if m["id"] == model_id)\n    return entry, entry["files"]\n\n\ndef fetch_manifest_model(\n    manifest: dict, model_id: str, models_dir: Path, only: list[str] | None = None\n):\n    """Tải file của một mô hình theo `models/manifest.yaml` (đúng revision) và kiểm SHA-256."""\n    entry, files = manifest_files(manifest, model_id)\n    for item in files:\n        if only and item["path"] not in only:\n            continue\n        download(item["url"], Path(models_dir) / model_id / item["path"], item["sha256"])\n    return entry\n\n\ndef ensure_ffmpeg(work: Path) -> tuple[str, str]:\n    """(ffmpeg, ffprobe). Kaggle/Colab có sẵn; máy thiếu thì tải bản tĩnh (johnvansickle, GPL) vào WORK/bin.\n    ffmpeg chỉ chạy như chương trình rời, không nhúng vào sản phẩm."""\n    if os.environ.get("VC_FFMPEG"):\n        return os.environ["VC_FFMPEG"], os.environ.get("VC_FFPROBE", "")\n    ffmpeg, ffprobe = shutil.which("ffmpeg"), shutil.which("ffprobe")\n    if ffmpeg and ffprobe:\n        return ffmpeg, ffprobe\n    bin_dir = Path(work) / "bin"\n    if not (bin_dir / "ffmpeg").is_file():\n        archive = download(\n            "https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz",\n            Path(work) / "ffmpeg-static.tar.xz",\n        )\n        bin_dir.mkdir(parents=True, exist_ok=True)\n        with tarfile.open(archive) as tar:\n            for member in tar.getmembers():\n                if member.name.endswith(("/ffmpeg", "/ffprobe")):\n                    member.name = Path(member.name).name\n                    tar.extract(member, bin_dir)\n    os.environ["PATH"] = f"{bin_dir}{os.pathsep}{os.environ[\'PATH\']}"\n    return str(bin_dir / "ffmpeg"), str(bin_dir / "ffprobe")\n\n\ndef ffmpeg_version(ffmpeg: str) -> str:\n    return subprocess.run([ffmpeg, "-version"], capture_output=True, text=True).stdout.splitlines()[\n        0\n    ]\n\n\ndef find_inputs(pattern: str, roots: list[str] | None = None) -> list[Path]:\n    """Tìm file khớp mẫu trong các thư mục input (Kaggle: /kaggle/input; Colab: Drive)."""\n    roots = roots or ["/kaggle/input", "/content/drive/MyDrive/vericlue", str(Path.cwd())]\n    found: list[Path] = []\n    for root in roots:\n        if Path(root).is_dir():\n            found.extend(Path(p) for p in glob.glob(os.path.join(root, pattern), recursive=True))\n    return sorted(set(found))\n\n\nclass Budget:\n    """Ngân sách thời gian của phiên (Kaggle tối đa 12 giờ): dừng sạch trước khi hết."""\n\n    def __init__(self, hours: float) -> None:\n        self.deadline = time.time() + hours * 3600\n\n    def left_h(self) -> float:\n        return (self.deadline - time.time()) / 3600\n\n    def ok(self, need_minutes: float = 5) -> bool:\n        return time.time() + need_minutes * 60 < self.deadline\n\n\ndef write_json(path: Path, data) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=1, default=str), encoding="utf-8")\n    os.replace(tmp, path)\n\n\ndef read_json(path: Path, default=None):\n    path = Path(path)\n    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else default\n\n\ndef disk_free_gb(path: Path) -> float:\n    return shutil.disk_usage(path).free / GB\n\n\ndef environment_report() -> dict:\n    """Phiên bản Python/thư viện, GPU, RAM, ổ đĩa — in ở ô đầu để gửi kèm khi báo lỗi."""\n    import importlib\n    import platform\n\n    info: dict = {"python": sys.version.split()[0], "os": platform.platform()}\n    for module in ("torch", "transformers", "onnxruntime", "cv2", "sklearn", "pydantic", "numpy"):\n        try:\n            info[module] = getattr(importlib.import_module(module), "__version__", "?")\n        except Exception as exc:  # noqa: BLE001 - chỉ để báo cáo\n            info[module] = f"chưa cài ({type(exc).__name__})"\n    try:\n        import torch\n\n        info["torch_cuda"] = torch.version.cuda\n        info["cuda_available"] = torch.cuda.is_available()\n    except Exception:  # noqa: BLE001\n        pass\n    try:\n        import onnxruntime\n\n        info["onnx_providers"] = onnxruntime.get_available_providers()\n    except Exception:  # noqa: BLE001\n        pass\n    info["gpu"] = gpu_info()\n    meminfo = Path("/proc/meminfo")\n    if meminfo.is_file():\n        fields = dict(\n            line.split(":", 1) for line in meminfo.read_text().splitlines() if ":" in line\n        )\n        for key in ("MemTotal", "MemAvailable"):\n            if key in fields:\n                info[f"ram_{key}_gb"] = round(int(fields[key].split()[0]) / 1024**2, 1)\n    for path in ("/kaggle/working", "/content", str(Path.cwd())):\n        if Path(path).is_dir():\n            info[f"disk_free_gb[{path}]"] = round(disk_free_gb(Path(path)), 1)\n    return info\n'),
    'vericlue_ml/__init__.py': ('43e8ba761f24224931ef9c34b87f08c1fc776933aacd1750362e12f9681bb547', '"""VeriClue — bộ não phát hiện giả mạo dùng chung cho chế độ eKYC và chế độ giám định."""\n\n__version__ = "0.1.0"\n'),
    'vericlue_ml/common/__init__.py': ('05d5cae1a0eee19f79de447fdbeb55caf7127f47e1c78239a647fa94a86ea6fb', '"""Thành phần dùng chung: định dạng kết quả, giao diện bộ phát hiện."""\n\nfrom vericlue_ml.common.detector import Detector, DetectorInput\nfrom vericlue_ml.common.result import (\n    ErrorCode,\n    Evidence,\n    EvidenceType,\n    Gate,\n    Module,\n    ModuleResult,\n    Status,\n)\n\n__all__ = [\n    "Detector",\n    "DetectorInput",\n    "ErrorCode",\n    "Evidence",\n    "EvidenceType",\n    "Gate",\n    "Module",\n    "ModuleResult",\n    "Status",\n]\n'),
    'vericlue_ml/common/calibration.py': ('d2ad7ad1458dd4f7b8bad5f41906e6b6d8fbef1fd8dcd9db3b0b4f3ee8b67beb', '"""Hiệu chỉnh trên calib_fit, tách người khi chọn Platt/isotonic (ROUT-01).\n\nNguồn thuật toán: scikit-learn GroupKFold, LogisticRegression, IsotonicRegression.\nKhông fit/chọn bằng calib_ltt hay test. Nhãn 1 = tấn công, score tăng = tấn công.\n"""\n\nfrom __future__ import annotations\n\nimport os\nfrom collections.abc import Sequence\nfrom datetime import UTC, datetime\nfrom pathlib import Path\nfrom typing import Annotated, Literal\nfrom urllib.parse import quote\n\nimport numpy as np\nfrom numpy.typing import ArrayLike, NDArray\nfrom pydantic import BaseModel, ConfigDict, Field, TypeAdapter, model_validator\nfrom sklearn.isotonic import IsotonicRegression\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.preprocessing import StandardScaler\n\nfrom vericlue_ml.common.result import Module, ModuleResult, Status\n\nEPSILON = 1e-4\nMIN_ISOTONIC_CLASS = 100\nMIN_RELATIVE_IMPROVEMENT = 0.02\nPLATT_C = 1e6\nPLATT_MAX_ITER = 2000\nPLATT_TOL = 1e-8\nREQUIRED_MODULES = frozenset(\n    {Module.FACE_PAD, Module.FACE_DEEPFAKE, Module.VOICE_SPOOF, Module.LIPSYNC}\n)\nFloatArray = NDArray[np.float64]\n\n\nclass CalibrationError(ValueError):\n    """Không được dùng điểm thô thay cho calibrator thiếu/hỏng/sai phiên bản."""\n\n\nclass _Record(BaseModel):\n    model_config = ConfigDict(extra="forbid", frozen=True, allow_inf_nan=False)\n\n\ndef _vector(values: ArrayLike, name: str) -> FloatArray:\n    result = np.asarray(values, dtype=float)\n    if result.ndim != 1 or not result.size or not np.isfinite(result).all():\n        raise CalibrationError(f"{name} phải là vector không rỗng, hữu hạn.")\n    return result\n\n\ndef _pairs(labels: ArrayLike, probabilities: ArrayLike) -> tuple[FloatArray, FloatArray]:\n    y, p = _vector(labels, "labels"), _vector(probabilities, "probabilities")\n    if y.shape != p.shape or not np.isin(y, [0, 1]).all():\n        raise CalibrationError("Nhãn phải là 0/1 và cùng chiều với xác suất.")\n    if np.any((p < 0) | (p > 1)):\n        raise CalibrationError("Xác suất phải nằm trong [0, 1].")\n    return y, p\n\n\nclass ReliabilityBin(_Record):\n    lower: float\n    upper: float\n    count: int\n    mean_probability: float | None\n    attack_rate: float | None\n\n\ndef reliability_table(\n    labels: ArrayLike, probabilities: ArrayLike, *, n_bins: int = 10\n) -> list[ReliabilityBin]:\n    """Bin đều [lower, upper); bin cuối gồm 1, giữ bin rỗng với giá trị None."""\n    y, p = _pairs(labels, probabilities)\n    if isinstance(n_bins, bool) or not isinstance(n_bins, int) or n_bins < 1:\n        raise CalibrationError("n_bins phải là số nguyên dương.")\n    indices = np.minimum((p * n_bins).astype(int), n_bins - 1)\n    rows = []\n    for i in range(n_bins):\n        mask = indices == i\n        count = int(mask.sum())\n        rows.append(\n            ReliabilityBin(\n                lower=i / n_bins,\n                upper=(i + 1) / n_bins,\n                count=count,\n                mean_probability=float(p[mask].mean()) if count else None,\n                attack_rate=float(y[mask].mean()) if count else None,\n            )\n        )\n    return rows\n\n\ndef ece(labels: ArrayLike, probabilities: ArrayLike, *, n_bins: int = 10) -> float:\n    """ECE nhị phân: trung bình có trọng số |P(tấn công) - tỷ lệ tấn công|."""\n    rows = reliability_table(labels, probabilities, n_bins=n_bins)\n    total = sum(row.count for row in rows)\n    return (\n        sum(\n            row.count * abs(row.mean_probability - row.attack_rate)\n            for row in rows\n            if row.count and row.mean_probability is not None and row.attack_rate is not None\n        )\n        / total\n    )\n\n\ndef brier_score(labels: ArrayLike, probabilities: ArrayLike) -> float:\n    """Brier nhị phân, không chia thêm cho hai."""\n    y, p = _pairs(labels, probabilities)\n    return float(np.mean((p - y) ** 2))\n\n\ndef log_loss(labels: ArrayLike, probabilities: ArrayLike, *, epsilon: float = EPSILON) -> float:\n    """Log-loss nhị phân với cắt xác suất hữu hạn."""\n    y, p = _pairs(labels, probabilities)\n    if not 0 < epsilon < 0.5:\n        raise CalibrationError("epsilon phải nằm trong (0, 0.5).")\n    p = np.clip(p, epsilon, 1 - epsilon)\n    return float(-np.mean(y * np.log(p) + (1 - y) * np.log1p(-p)))\n\n\nclass CalibrationMetrics(_Record):\n    log_loss: float = Field(ge=0)\n    ece: float = Field(ge=0, le=1)\n    brier: float = Field(ge=0, le=1)\n\n\ndef _metrics(y: FloatArray, p: FloatArray, epsilon: float) -> CalibrationMetrics:\n    return CalibrationMetrics(\n        log_loss=log_loss(y, p, epsilon=epsilon), ece=ece(y, p), brier=brier_score(y, p)\n    )\n\n\nclass CalibFit(_Record):\n    """Dữ liệu một module/phiên bản; caller chịu trách nhiệm chọn đúng calib_fit."""\n\n    scores: list[float]\n    labels: list[int]\n    subject_ids: list[str]\n    dataset: str = Field(min_length=1)\n    split: Literal["calib_fit"] = "calib_fit"\n\n\nclass Parameters(_Record):\n    slope: float | None = None\n    intercept: float | None = None\n    x: list[float] = Field(default_factory=list)\n    y: list[float] = Field(default_factory=list)\n\n\nclass Calibrator(_Record):\n    """Artifact JSON, tham số thuần số; không pickle hay tải trọng số mô hình."""\n\n    module: Module\n    model_version: str = Field(min_length=1)\n    provisional: bool = False\n    method: Literal["platt", "isotonic"]\n    parameters: Parameters\n    epsilon: Literal[0.0001] = EPSILON\n    n_pos: int = Field(gt=0)\n    n_neg: int = Field(gt=0)\n    before: CalibrationMetrics\n    after: CalibrationMetrics\n    before_reliability: list[ReliabilityBin]\n    after_reliability: list[ReliabilityBin]\n    date: datetime\n    commit: str = Field(min_length=1)\n    dataset: str = Field(min_length=1)\n    split: Literal["calib_fit"] = "calib_fit"\n    cv_folds: int = Field(ge=2)\n    cv_log_loss: dict[Literal["platt", "isotonic"], float]\n    metrics_scope: Literal["group_oof"] = "group_oof"\n    baseline: Literal["clipped_raw_score"] = "clipped_raw_score"\n\n    @model_validator(mode="after")\n    def validate_parameters(self) -> Calibrator:\n        p = self.parameters\n        if self.method == "platt":\n            if p.slope is None or p.intercept is None or p.x or p.y or p.slope < 0:\n                raise ValueError("Tham số Platt không hợp lệ hoặc ngược chiều.")\n        elif (\n            p.slope is not None\n            or p.intercept is not None\n            or not p.x\n            or len(p.x) != len(p.y)\n            or np.any(np.diff(p.x) <= 0)\n            or np.any(np.diff(p.y) < 0)\n            or np.any((np.asarray(p.y) < 0) | (np.asarray(p.y) > 1))\n        ):\n            raise ValueError("Tham số isotonic không hợp lệ.")\n        if "platt" not in self.cv_log_loss or any(v < 0 for v in self.cv_log_loss.values()):\n            raise ValueError("Thiếu/sai log-loss kiểm định chéo.")\n        if self.method == "isotonic" and not _choose_isotonic(\n            self.n_pos, self.n_neg, self.cv_log_loss\n        ):\n            raise ValueError("Isotonic không đạt điều kiện chọn.")\n        return self\n\n    @property\n    def prior_attack(self) -> float:\n        """Tiền nghiệm π trên calib_fit cho FOR-04; không phải tỷ lệ ngoài thực tế."""\n        return self.n_pos / (self.n_pos + self.n_neg)\n\n    def predict(self, scores: ArrayLike) -> FloatArray:\n        """P(tấn công), luôn được cắt theo epsilon của artifact."""\n        return _predict(self.method, self.parameters, _vector(scores, "scores"), self.epsilon)\n\n\nclass IdentityCalibrator(_Record):\n    """Explicit untrained MVP artifact; no fitted statistics or report prior."""\n\n    module: Module\n    model_version: str = Field(min_length=1)\n    method: Literal["identity"] = "identity"\n    provisional: Literal[True] = True\n    note: Literal["MVP — không dùng cho báo cáo"] = "MVP — không dùng cho báo cáo"\n    parameters: Parameters = Field(default_factory=Parameters)\n    epsilon: Literal[0.0001] = EPSILON\n    n_pos: Literal[0] = 0\n    n_neg: Literal[0] = 0\n    dataset: Literal["not_fitted"] = "not_fitted"\n    split: Literal["none"] = "none"\n    metrics_scope: Literal["not_evaluated"] = "not_evaluated"\n    date: datetime\n    commit: str = Field(min_length=1)\n\n    @model_validator(mode="after")\n    def validate_identity(self) -> IdentityCalibrator:\n        if self.module not in REQUIRED_MODULES:\n            raise ValueError("Identity is limited to the ADR-0012 score modules")\n        if self.parameters != Parameters():\n            raise ValueError("Identity cannot contain fitted parameters")\n        return self\n\n    @property\n    def prior_attack(self) -> float:\n        """No empirical prior exists for an artifact that has never been fitted."""\n        raise CalibrationError("Identity MVP has no fitted prior; not for reports")\n\n    def predict(self, scores: ArrayLike) -> FloatArray:\n        """Keep finite raw scores and clip to the ordinary artifact epsilon."""\n        return np.clip(_vector(scores, "scores"), self.epsilon, 1 - self.epsilon)\n\n\ndef identity(*, module: str | Module, model_version: str, commit: str) -> IdentityCalibrator:\n    """Create the explicitly labelled, data-free MVP calibrator from ADR-0012."""\n    if not model_version.strip() or not commit.strip():\n        raise CalibrationError("Model version and source commit are required")\n    return IdentityCalibrator(\n        module=Module(module), model_version=model_version, commit=commit, date=datetime.now(UTC)\n    )\n\n\nCalibrationArtifact = Calibrator | IdentityCalibrator\n_ARTIFACT_ADAPTER = TypeAdapter(Annotated[CalibrationArtifact, Field(discriminator="method")])\n\n\ndef _fit_parameters(method: str, x: FloatArray, y: FloatArray) -> Parameters:\n    if method == "isotonic":\n        model = IsotonicRegression(increasing=True, out_of_bounds="clip").fit(x, y)\n        return Parameters(x=model.X_thresholds_.tolist(), y=model.y_thresholds_.tolist())\n    scaler = StandardScaler().fit(x.reshape(-1, 1))\n    model = LogisticRegression(C=PLATT_C, solver="lbfgs", max_iter=PLATT_MAX_ITER, tol=PLATT_TOL)\n    model.fit(scaler.transform(x.reshape(-1, 1)), y)\n    slope = float(model.coef_[0, 0] / scaler.scale_[0])\n    return Parameters(slope=slope, intercept=float(model.intercept_[0] - slope * scaler.mean_[0]))\n\n\ndef _predict(method: str, p: Parameters, scores: FloatArray, epsilon: float) -> FloatArray:\n    if method == "isotonic":\n        probabilities = np.interp(scores, p.x, p.y)\n    else:\n        z = p.slope * scores + p.intercept\n        # Stable sigmoid without overflow for extreme raw scores.\n        probabilities = np.exp(-np.logaddexp(0, -z))\n    return np.clip(probabilities, epsilon, 1 - epsilon)\n\n\ndef _choose_isotonic(n_pos: int, n_neg: int, losses: dict[str, float]) -> bool:\n    return (\n        min(n_pos, n_neg) >= MIN_ISOTONIC_CLASS\n        and "isotonic" in losses\n        and losses["isotonic"] <= losses["platt"] * (1 - MIN_RELATIVE_IMPROVEMENT)\n    )\n\n\ndef fit(\n    calib_fit: CalibFit,\n    *,\n    module: str | Module,\n    model_version: str,\n    commit: str,\n    n_splits: int = 5,\n    epsilon: float = EPSILON,\n) -> Calibrator:\n    """Chọn bằng GroupKFold OOF log-loss, rồi fit phương pháp thắng trên toàn calib_fit."""\n    x = _vector(calib_fit.scores, "scores")\n    y, _ = _pairs(calib_fit.labels, np.clip(x, 0, 1))\n    groups = np.asarray(calib_fit.subject_ids)\n    if groups.shape != x.shape or any(not s.strip() for s in calib_fit.subject_ids):\n        raise CalibrationError("subject_ids phải không rỗng và cùng chiều với scores.")\n    if np.unique(y).size != 2:\n        raise CalibrationError("calib_fit phải có cả hai lớp.")\n    if not isinstance(n_splits, int) or isinstance(n_splits, bool) or n_splits < 2:\n        raise CalibrationError("GroupKFold cần ít nhất hai phần.")\n    if epsilon != EPSILON:\n        raise CalibrationError("Fit phải dùng epsilon=1e-4.")\n    if roc_auc_score(y, x) < 0.5:\n        raise CalibrationError("AUC < 0.5: nghi score ngược chiều tấn công.")\n    folds = min(n_splits, np.unique(groups).size)\n    if folds < 2:\n        raise CalibrationError("GroupKFold cần ít nhất hai subject.")\n    splits = list(GroupKFold(n_splits=folds).split(x, y, groups))\n    if any(np.unique(y[train]).size != 2 for train, _ in splits):\n        raise CalibrationError("Một fold fit thiếu lớp; cần thêm subject hoặc đổi n_splits.")\n    n_pos, n_neg = int(y.sum()), int((1 - y).sum())\n    methods = ["platt", "isotonic"] if min(n_pos, n_neg) >= MIN_ISOTONIC_CLASS else ["platt"]\n    losses: dict[str, float] = {}\n    predictions: dict[str, FloatArray] = {}\n    for method in methods:\n        oof = np.empty_like(x)\n        for train, valid in splits:\n            parameters = _fit_parameters(method, x[train], y[train])\n            oof[valid] = _predict(method, parameters, x[valid], epsilon)\n        predictions[method] = oof\n        losses[method] = log_loss(y, oof, epsilon=epsilon)\n    method = "isotonic" if _choose_isotonic(n_pos, n_neg, losses) else "platt"\n    return Calibrator(\n        module=Module(module),\n        model_version=model_version,\n        method=method,\n        parameters=_fit_parameters(method, x, y),\n        epsilon=epsilon,\n        n_pos=n_pos,\n        n_neg=n_neg,\n        before=_metrics(y, np.clip(x, epsilon, 1 - epsilon), epsilon),\n        after=_metrics(y, predictions[method], epsilon),\n        before_reliability=reliability_table(y, np.clip(x, epsilon, 1 - epsilon)),\n        after_reliability=reliability_table(y, predictions[method]),\n        date=datetime.now(UTC),\n        commit=commit,\n        dataset=calib_fit.dataset,\n        cv_folds=folds,\n        cv_log_loss=losses,\n    )\n\n\ndef reliability_markdown(rows: Sequence[ReliabilityBin]) -> str:\n    """Bảng reliability Markdown từ dữ liệu bin, không tự tạo số liệu."""\n    lines = ["| Bin | Số mẫu | Xác suất trung bình | Tỷ lệ tấn công |", "|---|---:|---:|---:|"]\n    for row in rows:\n        mean = "—" if row.mean_probability is None else f"{row.mean_probability:.6f}"\n        rate = "—" if row.attack_rate is None else f"{row.attack_rate:.6f}"\n        closing = "]" if row.upper == 1 else ")"\n        lines.append(\n            f"| [{row.lower:.3f}, {row.upper:.3f}{closing} | {row.count} | {mean} | {rate} |"\n        )\n    return "\\n".join(lines) + "\\n"\n\n\ndef plot_reliability(rows: Sequence[ReliabilityBin], destination: Path) -> None:\n    """Xuất PNG khi đã cài extra eval; matplotlib chỉ được import lúc gọi."""\n    try:\n        import matplotlib.pyplot as plt\n    except ImportError as exc:\n        raise CalibrationError("Xuất PNG cần cài extra eval của vericlue-ml.") from exc\n    populated = [r for r in rows if r.count]\n    fig, ax = plt.subplots()\n    try:\n        ax.plot([0, 1], [0, 1], linestyle="--", color="gray")\n        ax.plot([r.mean_probability for r in populated], [r.attack_rate for r in populated], "o-")\n        ax.set(\n            xlim=(0, 1),\n            ylim=(0, 1),\n            xlabel="Xác suất tấn công trung bình",\n            ylabel="Tỷ lệ tấn công",\n            title="Reliability",\n        )\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        fig.savefig(destination, format="png", bbox_inches="tight")\n    finally:\n        plt.close(fig)\n\n\nclass CalibrationStore:\n    """Nạp một lần theo module/version; root truyền vào hoặc CALIBRATORS_DIR."""\n\n    def __init__(self, root: Path | None = None) -> None:\n        self.root = (\n            root\n            if root is not None\n            else Path(\n                os.getenv(\n                    "CALIBRATORS_DIR",\n                    str(Path(__file__).resolve().parents[3] / "configs" / "calibrators"),\n                )\n            )\n        )\n        self._cache: dict[tuple[Module, str], CalibrationArtifact] = {}\n\n    def path(self, module: str | Module, model_version: str) -> Path:\n        """Đường dẫn portable: percent-encode ký tự không hợp lệ, không cho traversal."""\n        if not model_version.strip() or model_version in {".", ".."}:\n            raise CalibrationError("model_version không hợp lệ.")\n        return self.root / Module(module).value / f"{quote(model_version, safe=\'@-_.\')}.json"\n\n    def save(self, calibrator: CalibrationArtifact) -> Path:\n        """Lưu JSON đã kiểm tra, rồi cập nhật cache của store này."""\n        calibrator = _ARTIFACT_ADAPTER.validate_json(calibrator.model_dump_json())\n        path = self.path(calibrator.module, calibrator.model_version)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        path.write_text(calibrator.model_dump_json(indent=2) + "\\n", encoding="utf-8")\n        self._cache[(calibrator.module, calibrator.model_version)] = calibrator\n        return path\n\n    def load(self, module: str | Module, model_version: str) -> CalibrationArtifact:\n        """Thiếu/hỏng/sai phiên bản => CalibrationError, tuyệt đối không fallback."""\n        key = (Module(module), model_version)\n        if key not in self._cache:\n            path = self.path(*key)\n            try:\n                artifact = _ARTIFACT_ADAPTER.validate_json(path.read_text(encoding="utf-8"))\n            except (OSError, ValueError) as exc:\n                raise CalibrationError(\n                    f"Không nạp được calibrator {key[0].value}/{model_version}: {path}"\n                ) from exc\n            if (artifact.module, artifact.model_version) != key:\n                raise CalibrationError(f"Calibrator sai module/model_version: {path}")\n            self._cache[key] = artifact\n        return self._cache[key]\n\n    def apply(self, result: ModuleResult) -> ModuleResult:\n        """Caller bắt CalibrationError để trả status=error; không sửa result gốc."""\n        if (\n            result.model_version.startswith("mock@")\n            or Module(result.module) not in REQUIRED_MODULES\n        ):\n            return result\n        if result.status != Status.OK:\n            # Do not manufacture usable probability from error/degraded/not_applicable.\n            return result.model_copy(update={"prob_attack": None})\n        calibrator = self.load(result.module, result.model_version)\n        if result.score is None:\n            raise CalibrationError(f"Thiếu score cho {result.module}/{result.model_version}.")\n        probability = float(calibrator.predict([result.score])[0])\n        return result.model_copy(update={"prob_attack": probability})\n'),
    'vericlue_ml/common/detector.py': ('78a49bd741d646ffd57a40b3c1af97bbeaf6a33ec1d45396f852b8a1de2605ff', '"""Giao diện chung của mọi bộ phát hiện (`docs/architecture/architecture.md` §3)."""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Protocol, runtime_checkable\n\nfrom pydantic import BaseModel, ConfigDict, Field\n\nfrom vericlue_ml.common.result import ModuleResult\n\n\nclass DetectorInput(BaseModel):\n    """Đầu vào cho một bộ phát hiện. Bộ nào dùng trường nào thì đọc trường đó."""\n\n    model_config = ConfigDict(extra="forbid", arbitrary_types_allowed=True)\n\n    session_id: str | None = None\n    images: list[Path] = Field(default_factory=list, description="Ảnh mặt chất lượng cao")\n    video: Path | None = None\n    audio: Path | None = None\n    signals: dict[str, Any] = Field(\n        default_factory=dict, description="Tín hiệu trình duyệt đo (schema Signals)"\n    )\n    metadata: dict[str, Any] = Field(\n        default_factory=dict, description="Ví dụ: scenario (cho mock), challenge, câu đề bài"\n    )\n\n\n@runtime_checkable\nclass Detector(Protocol):\n    """Mọi bộ phát hiện (thật hay mock) đều có dạng này."""\n\n    name: str\n    version: str\n\n    def predict(self, inp: DetectorInput) -> ModuleResult: ...\n'),
    'vericlue_ml/common/device.py': ('9ab2d636ec4c68cc13c6ea27b2b76c8f99de3e64915b1bfc885f6512f204af68', '"""Chọn thiết bị chạy mô hình (CPU hay GPU) một cách thống nhất cho mọi detector.\n\n`DEVICE` (biến môi trường hoặc `Settings.device`) nhận `cpu` | `cuda` | `cuda:N` | `auto`:\n- `cpu`: luôn CPU.\n- `cuda` / `cuda:N` / `auto`: dùng GPU nếu có; KHÔNG có GPU (hoặc thiếu runtime) thì tự lùi về CPU và\n  ghi log cảnh báo — không bao giờ làm phiên hỏng chỉ vì thiếu GPU.\n\nDùng chung cho torch (GenD, AntiDeepfake, PhoWhisper) và onnxruntime (SCRFD, MiniFASNet). Bản mặc định\nvà CI cài torch/onnxruntime CPU; GPU cần extra `cuda` (xem `ml/README.md`).\n"""\n\nfrom __future__ import annotations\n\nimport logging\nimport os\nimport re\n\nlogger = logging.getLogger("vericlue_ml.device")\n\n_VALID = re.compile(r"cpu|auto|cuda(?::\\d+)?")\n\n\ndef requested_device(device: str | None = None, default: str = "cpu") -> str:\n    """Giá trị đã chuẩn hóa: tham số > biến `DEVICE` > `default`. Sai định dạng thì ValueError."""\n    value = (device or os.environ.get("DEVICE") or default).strip().lower()\n    if not _VALID.fullmatch(value):\n        raise ValueError("Thiết bị hợp lệ: cpu, cuda, cuda:N hoặc auto.")\n    return value\n\n\ndef resolve_torch_device(\n    device: str | None = None, *, component: str = "", default: str = "cpu"\n) -> str:\n    """`cpu` hoặc `cuda[:N]` cho torch; yêu cầu GPU mà không có thì lùi về CPU và ghi log."""\n    wanted = requested_device(device, default)\n    if wanted == "cpu":\n        return "cpu"\n    try:\n        import torch\n\n        available = torch.cuda.is_available()\n    except Exception:  # noqa: BLE001 - thiếu torch hay lỗi driver đều coi như không có GPU\n        available = False\n    if not available:\n        logger.warning(\n            "device_fallback component=%s requested=%s used=cpu reason=torch_cuda_unavailable",\n            component,\n            wanted,\n        )\n        return "cpu"\n    return "cuda" if wanted == "auto" else wanted\n\n\ndef preload_onnx_cuda_dlls() -> None:\n    """Nạp DLL CUDA/cuDNN cho onnxruntime-gpu (gói pip `nvidia-*` hoặc đi kèm torch). Lỗi thì bỏ qua."""\n    try:\n        import onnxruntime as ort\n\n        preload = getattr(ort, "preload_dlls", None)\n        if preload is not None:\n            preload()\n    except Exception as exc:  # noqa: BLE001\n        logger.warning("onnx_preload_dlls_failed error=%s", type(exc).__name__)\n\n\ndef onnx_providers(device: str | None = None, *, component: str = "") -> list[str]:\n    """Danh sách execution provider cho onnxruntime; CPU luôn có để lùi về."""\n    import onnxruntime as ort\n\n    wanted = requested_device(device)\n    if wanted == "cpu":\n        return ["CPUExecutionProvider"]\n    preload_onnx_cuda_dlls()\n    if "CUDAExecutionProvider" in ort.get_available_providers():\n        return ["CUDAExecutionProvider", "CPUExecutionProvider"]\n    logger.warning(\n        "device_fallback component=%s requested=%s used=cpu reason=onnxruntime_without_cuda",\n        component,\n        wanted,\n    )\n    return ["CPUExecutionProvider"]\n\n\ndef check_onnx_session(session, device: str | None, *, component: str = "") -> str:\n    """Provider thực sự đang chạy; ghi log nếu yêu cầu GPU mà phiên đã lùi về CPU."""\n    active = session.get_providers()[0]\n    if requested_device(device) != "cpu" and active != "CUDAExecutionProvider":\n        logger.warning(\n            "device_fallback component=%s requested=%s used=cpu reason=cuda_provider_not_loaded",\n            component,\n            requested_device(device),\n        )\n    return active\n'),
    'vericlue_ml/common/result.py': ('110ebae42697f2dfd6a810157ad685d6e8f8cd7aa3ecff5d6c977e822b178def', '"""ModuleResult — định dạng kết quả chuẩn của MỌI bộ phát hiện.\n\nKhớp hợp đồng `packages/contracts/schemas/module_result.schema.json`.\nTest `ml/tests/test_contract_sync.py` bảo đảm hai bên không lệch nhau.\n"""\n\nfrom __future__ import annotations\n\nfrom enum import StrEnum\nfrom typing import Any\n\nfrom pydantic import BaseModel, ConfigDict, Field\n\n\nclass Module(StrEnum):\n    QUALITY = "quality"\n    FACE_PAD = "face_pad"\n    FACE_DEEPFAKE = "face_deepfake"\n    CHROMAPULSE = "chromapulse"\n    CAMPROBE = "camprobe"\n    VOICE_SPOOF = "voice_spoof"\n    VOICE_CHALLENGE = "voice_challenge"\n    LIPSYNC = "lipsync"\n    PROVENANCE = "provenance"\n\n\nclass Status(StrEnum):\n    OK = "ok"\n    NOT_APPLICABLE = "not_applicable"  # không có mặt, im lặng, nhiều mặt...\n    DEGRADED = "degraded"  # chạy được nhưng đầu vào kém — KHÔNG được coi là đạt\n    ERROR = "error"  # lỗi — KHÔNG BAO GIỜ được coi là đạt\n\n\nclass ErrorCode(StrEnum):\n    E_NO_FACE = "E_NO_FACE"\n    E_MULTI_FACE = "E_MULTI_FACE"\n    E_LOW_QUALITY = "E_LOW_QUALITY"\n    E_TIMEOUT = "E_TIMEOUT"\n    E_DECODE = "E_DECODE"\n    E_CHALLENGE_MISMATCH = "E_CHALLENGE_MISMATCH"\n    E_INTEGRITY = "E_INTEGRITY"\n    E_INTERNAL = "E_INTERNAL"\n\n\nclass Gate(StrEnum):\n    """Kết quả của cổng luật (quality, chromapulse, camprobe)."""\n\n    PASS = "pass"\n    FAIL = "fail"\n    UNCLEAR = "unclear"\n\n\nclass EvidenceType(StrEnum):\n    SCALAR = "scalar"\n    TIMESERIES = "timeseries"\n    IMAGE = "image"\n    SEGMENTS = "segments"\n    TEXT = "text"\n\n\nclass Evidence(BaseModel):\n    """Một mẩu bằng chứng hiển thị cho người duyệt hoặc điều tra viên."""\n\n    model_config = ConfigDict(extra="forbid")\n\n    type: EvidenceType\n    name: str\n    value: Any = None\n    uri: str | None = None\n    unit: str | None = None\n    label_vi: str | None = Field(default=None, description="Nhãn tiếng Việt cho người duyệt")\n\n\nclass ModuleResult(BaseModel):\n    """Kết quả chuẩn của một bộ phát hiện. TrustRouter chỉ dùng `prob_attack`."""\n\n    model_config = ConfigDict(extra="forbid", use_enum_values=True)\n\n    module: Module\n    model_version: str\n    status: Status\n    error_code: ErrorCode | None = None\n    score: float | None = Field(default=None, description="Điểm thô, chưa hiệu chỉnh")\n    prob_attack: float | None = Field(default=None, ge=0.0, le=1.0)\n    gate: Gate | None = None\n    quality: dict[str, float | str | bool | None] = Field(default_factory=dict)\n    evidence: list[Evidence] = Field(default_factory=list)\n    latency_ms: float = Field(ge=0.0)\n\n    @property\n    def is_usable(self) -> bool:\n        """Có thể dùng làm bằng chứng cho phép \'đạt\' hay không.\n\n        error và degraded KHÔNG BAO GIỜ được coi là đạt (AGENTS.md §6 luật 4).\n        """\n        return self.status == Status.OK\n\n    def to_contract(self) -> dict[str, Any]:\n        """Dạng JSON đúng hợp đồng (bỏ trường None ngoài các trường bắt buộc)."""\n        return self.model_dump(mode="json", exclude_none=True)\n'),
    'vericlue_ml/eval/__init__.py': ('70179ec7606b703a4115e7197ea86cf852d288568983299ccf57d23c22c44058', '"""Khung đánh giá chung VeriClue (EVAL-01), chỉ đo với cấu hình đã khóa."""\n'),
    'vericlue_ml/eval/metrics.py': ('19fcf605857c0fe5f8936525d582663ce616ef345ab26c4b59901fd716c819b8', '"""Chỉ số nhị phân: 1 = tấn công; probability >= threshold bị từ chối.\n\nROC mô tả không xuất ngưỡng. Nguồn: sklearn.metrics.roc_curve/roc_auc_score;\nClopper-Pearson dùng scipy.stats.binomtest(...).proportion_ci(method="exact").\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Sequence\n\nimport numpy as np\nfrom numpy.typing import ArrayLike, NDArray\nfrom scipy.stats import binomtest\nfrom sklearn.metrics import roc_auc_score, roc_curve\n\nfrom vericlue_ml.common.calibration import brier_score, ece\n\n__all__ = [\n    "apcer_by_attack",\n    "auc",\n    "bpcer",\n    "brier_score",\n    "clopper_pearson",\n    "descriptive_bpcer_at_apcer",\n    "ece",\n    "eer",\n]\n\n\ndef _vectors(labels: ArrayLike, probabilities: ArrayLike) -> tuple[NDArray, NDArray]:\n    y, p = np.asarray(labels), np.asarray(probabilities, dtype=float)\n    if y.ndim != 1 or p.ndim != 1 or y.shape != p.shape or not y.size:\n        raise ValueError("Nhãn và xác suất phải là vector không rỗng, cùng chiều.")\n    if not np.isin(y, [0, 1]).all() or not np.isfinite(p).all() or np.any((p < 0) | (p > 1)):\n        raise ValueError("Nhãn phải là 0/1, xác suất hữu hạn trong [0,1].")\n    return y, p\n\n\ndef _threshold(threshold: float) -> None:\n    if not np.isfinite(threshold) or not 0 <= threshold <= 1:\n        raise ValueError("Ngưỡng xác suất phải nằm trong [0,1].")\n\n\ndef apcer_by_attack(\n    labels: ArrayLike,\n    probabilities: ArrayLike,\n    attack_types: Sequence[str],\n    threshold: float,\n) -> dict[str, float]:\n    """Tỷ lệ tấn công được chấp nhận theo từng attack_type; bỏ lớp thật khỏi mẫu số."""\n    y, p = _vectors(labels, probabilities)\n    _threshold(threshold)\n    attacks = np.asarray(attack_types)\n    if attacks.shape != y.shape or any(not isinstance(a, str) for a in attack_types):\n        raise ValueError("attack_types phải là vector chuỗi cùng chiều.")\n    if any(not str(a).strip() for a in attacks[y == 1]):\n        raise ValueError("Mẫu tấn công phải có attack_type.")\n    attacks = np.array([str(a).strip() for a in attacks])\n    return {\n        str(kind): float(np.mean(p[(y == 1) & (attacks == kind)] < threshold))\n        for kind in sorted(set(attacks[y == 1]))\n    }\n\n\ndef bpcer(labels: ArrayLike, probabilities: ArrayLike, threshold: float) -> float | None:\n    """Tỷ lệ người thật bị từ chối; None khi không có mẫu thật."""\n    y, p = _vectors(labels, probabilities)\n    _threshold(threshold)\n    return float(np.mean(p[y == 0] >= threshold)) if np.any(y == 0) else None\n\n\ndef auc(labels: ArrayLike, probabilities: ArrayLike) -> float | None:\n    """ROC AUC mô tả; None khi thiếu một lớp."""\n    y, p = _vectors(labels, probabilities)\n    return float(roc_auc_score(y, p)) if np.unique(y).size == 2 else None\n\n\ndef eer(labels: ArrayLike, probabilities: ArrayLike) -> float | None:\n    """EER nội suy tuyến tính trên ROC, chỉ trả chỉ số; không chọn/xuất ngưỡng."""\n    y, p = _vectors(labels, probabilities)\n    if np.unique(y).size < 2:\n        return None\n    fpr, tpr = roc_curve(y, p, drop_intermediate=False)[:2]\n    difference = 1 - tpr - fpr\n    right = int(np.flatnonzero(difference <= 0)[0])\n    if difference[right] == 0:\n        return float(fpr[right])\n    left = right - 1\n    fraction = difference[left] / (difference[left] - difference[right])\n    return float(fpr[left] + fraction * (fpr[right] - fpr[left]))\n\n\ndef descriptive_bpcer_at_apcer(\n    labels: ArrayLike,\n    probabilities: ArrayLike,\n    target: float = 0.05,\n) -> float | None:\n    """BPCER nội suy ở APCER gộp target trên ROC đánh giá; chỉ trả scalar.\n\n    Nội suy là mô tả, có thể tương ứng trộn quyết định ở điểm đồng hạng, không\n    phải ngưỡng triển khai. Không lưu hoặc trả bất cứ ngưỡng nào của đường cong.\n    """\n    y, p = _vectors(labels, probabilities)\n    if not np.isfinite(target) or not 0 <= target <= 1:\n        raise ValueError("target phải nằm trong [0,1].")\n    if np.unique(y).size < 2:\n        return None\n    fpr, tpr = roc_curve(y, p, drop_intermediate=False)[:2]\n    fnr = 1 - tpr\n    exact = np.flatnonzero(np.isclose(fnr, target, rtol=0, atol=8 * np.finfo(float).eps))\n    if exact.size:\n        return float(fpr[exact].min())\n    # Preserve every corner of the ROC; collapsing duplicate FNR before\n    # interpolation would shortcut horizontal segments and understate BPCER.\n    right = int(np.flatnonzero(fnr < target)[0])\n    left = right - 1\n    fraction = (fnr[left] - target) / (fnr[left] - fnr[right])\n    return float(fpr[left] + fraction * (fpr[right] - fpr[left]))\n\n\ndef clopper_pearson(\n    successes: int,\n    total: int,\n    confidence: float = 0.95,\n) -> tuple[float, float] | None:\n    """CI nhị thức exact hai phía; số dòng độc lập là giả định, total=0 => None."""\n    if (\n        isinstance(successes, bool)\n        or isinstance(total, bool)\n        or not isinstance(successes, (int, np.integer))\n        or not isinstance(total, (int, np.integer))\n        or not 0 <= successes <= total\n        or not 0 < confidence < 1\n    ):\n        raise ValueError("Cần số đếm nguyên 0 <= successes <= total và confidence trong (0,1).")\n    if total == 0:\n        return None\n    interval = binomtest(successes, total).proportion_ci(\n        confidence_level=confidence, method="exact"\n    )\n    return float(interval.low), float(interval.high)\n'),
    'vericlue_ml/face/__init__.py': ('f1465bdea1efebc1fb51efb29fd258635866c541f6729e7b572e150baa375d9d', '"""Mô-đun khuôn mặt: PAD (FACE-01), deepfake (FACE-02)."""\n'),
    'vericlue_ml/face/frames.py': ('cb71c475c9e57fac7c66304c35b70b50fdd2611bdd4be03656a2f04286de9af4', '"""Đọc khung hình từ video hoặc ảnh cho `face_deepfake` và `quality` (FACE-02).\n\nVideo từ trình duyệt (webm VP8/VP9 do MediaRecorder ghi) thường KHÔNG có số khung hoặc thời lượng\nđúng trong container, nên không tin `CAP_PROP_FRAME_COUNT`: đọc tuần tự bằng `grab()` để đếm số khung\nthật, rồi đọc lại và lấy đúng các khung đều nhau. Khung được trả dần (generator) để không giữ nhiều\nkhung độ phân giải gốc trong bộ nhớ. Ảnh trả về là BGR (quy ước của OpenCV).\n\n`sample_video` trả thêm số khung ĐÃ YÊU CẦU (`min(count, số khung thật)`): khung giải mã hỏng bị\nbỏ qua nhưng vẫn tính là thiếu, để video cắt cụt không thể ra kết quả "đủ khung".\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterator\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom vericlue_ml.common.result import ErrorCode\nfrom vericlue_ml.face.pad import PadError, _decode\n\n# Chặn video quá dài (bom giải mã): chỉ đếm và lấy mẫu trong chừng này khung đầu (~1 phút ở 30 fps).\n# Phần video sau mốc này không được xem xét.\nMAX_SCAN_FRAMES = 1800\n\nFrames = Iterator[tuple[int, np.ndarray]]\n\n\ndef even_indices(total: int, count: int) -> list[int]:\n    """`count` chỉ số đều nhau trong [0, total-1] (cả hai đầu), bỏ trùng, theo thứ tự tăng."""\n    if total <= 0 or count <= 0:\n        return []\n    if total <= count:\n        return list(range(total))\n    return sorted({int(i) for i in np.linspace(0, total - 1, count).round()})\n\n\ndef sample_images(paths: list[Path], count: int) -> tuple[int, Frames]:\n    """(số ảnh yêu cầu, ảnh đều nhau); ảnh hỏng thì ném PadError(E_DECODE) khi duyệt tới."""\n    chosen = even_indices(len(paths), count)\n\n    def generate() -> Frames:\n        for i in chosen:\n            yield i, _decode(Path(paths[i]))\n\n    return len(chosen), generate()\n\n\ndef read_images(paths: list[Path], count: int) -> list[tuple[int, np.ndarray]]:\n    """Như `sample_images` nhưng trả danh sách."""\n    return list(sample_images(paths, count)[1])\n\n\ndef sample_video(path: Path, count: int) -> tuple[int, Frames]:\n    """Trả (số khung yêu cầu, generator (chỉ số khung, ảnh BGR)) cho `count` khung đều nhau.\n\n    Không mở được hoặc không có khung nào đọc được thì ném PadError(E_DECODE) ngay (lượt đếm khung\n    chạy trước khi trả). Generator đóng capture khi duyệt hết hoặc khi bị `close()`.\n    """\n    import cv2\n\n    def open_capture():\n        capture = cv2.VideoCapture(str(path))\n        if not capture.isOpened():\n            capture.release()\n            raise PadError("Không mở được video.", ErrorCode.E_DECODE)\n        return capture\n\n    try:\n        capture = open_capture()\n        try:\n            total = 0\n            while total < MAX_SCAN_FRAMES and capture.grab():\n                total += 1\n        finally:\n            capture.release()\n    except PadError:\n        raise\n    except Exception as exc:\n        raise PadError("Không đọc được video.", ErrorCode.E_DECODE) from exc\n    if total == 0:\n        raise PadError("Video không có khung hình đọc được.", ErrorCode.E_DECODE)\n\n    wanted = even_indices(total, count)\n    pending = set(wanted)\n\n    def generate() -> Frames:\n        capture = open_capture()  # lỗi mở video → PadError(E_DECODE), không phải "không có mặt"\n        try:\n            for index in range(wanted[-1] + 1):\n                if not capture.grab():\n                    break\n                if index in pending:\n                    ok, frame = capture.retrieve()\n                    if ok and frame is not None and frame.size > 0:\n                        yield index, frame\n        finally:\n            capture.release()\n\n    return len(wanted), generate()\n\n\ndef read_video_frames(path: Path, count: int) -> list[tuple[int, np.ndarray]]:\n    """Như `sample_video` nhưng trả danh sách; không giải mã nổi khung nào thì ném PadError(E_DECODE)."""\n    try:\n        frames = list(sample_video(path, count)[1])\n    except PadError:\n        raise\n    except Exception as exc:\n        raise PadError("Không đọc được video.", ErrorCode.E_DECODE) from exc\n    if not frames:\n        raise PadError("Không giải mã được khung nào của video.", ErrorCode.E_DECODE)\n    return frames\n\n\ndef middle_video_frame(path: Path) -> np.ndarray:\n    """Khung ở giữa của video (một trong ba khung đều nhau), chỉ giải mã tới khung đó."""\n    _, frames = sample_video(path, 3)\n    kept: list[np.ndarray] = []\n    try:\n        for position, (_, frame) in enumerate(frames):\n            kept = [frame]\n            if position == 1:\n                break\n    except Exception as exc:\n        raise PadError("Không đọc được video.", ErrorCode.E_DECODE) from exc\n    finally:\n        frames.close()  # type: ignore[attr-defined]\n    if not kept:\n        raise PadError("Không giải mã được khung nào của video.", ErrorCode.E_DECODE)\n    return kept[0]\n'),
    'vericlue_ml/face/gend.py': ('00cec861d808087c6ca35864773a61b925ecd735cfb3a220af0be0c873851ac7', '"""FACE-02 — phát hiện deepfake khuôn mặt bằng GenD (CLIP ViT-L/14 tinh chỉnh LayerNorm + đầu tuyến tính).\n\nLuồng: video (hoặc ảnh) → lấy N khung đều nhau → SCRFD `det_10g` → mặt lớn nhất mỗi khung → căn chỉnh\n5 điểm (tỷ lệ 1,3) → 224 px, mean/std của CLIP → GenD → softmax → `p_fake` mỗi khung. Điểm video:\n`score = max(mean, top25_mean)` của `p_fake` (điểm thô; `prob_attack` để trống chờ hiệu chỉnh, như\n`face/pad.py`). Không có mặt ở mọi khung → `not_applicable` (E_NO_FACE); quá ít khung có mặt →\n`degraded` (không bao giờ được coi là đạt); thiếu trọng số, sai SHA-256 hay lỗi bất kỳ → `error`.\n\nMã KHÔNG gọi mạng: trọng số đọc từ `MODELS_DIR/<id trong models/manifest.yaml>/<file>` và kiểm SHA-256.\nBackbone dựng từ `config.json` của `clip-vit-large-patch14` rồi nạp trọng số vision từ checkpoint GenD\n(không dùng `GenD.from_pretrained` vì nó tải CLIP từ mạng và chạy mã từ xa).\n\nNguồn tiền xử lý (chép lại, giữ giấy phép — xem `third_party/README.md`):\n- yermandy/GenD (MIT, Copyright (c) 2025 Andy), commit 387a42266dd385fbe8f7626c5d3aa03eea3bfaab:\n  `detector.py::align_face` (mốc đích, `estimateAffinePartial2D` LMEDS, `target_size=None` rồi để\n  CLIP resize về 224), `src/retinaface.py` (SCRFD: letterbox 640, `blobFromImage(1/128, mean 127.5,\n  swapRB)`, NMS 0,4), `src/hf/modeling_gend.py` (`pooler_output` → L2-normalize → Linear(1024, 2)),\n  `app/run.py` (BGR→RGB, softmax, nhãn 1 = giả). Mean/std và nội suy BICUBIC theo\n  `preprocessor_config.json` của CLIP.\n- `top25_mean` là quy ước riêng của VeriClue (ADR-0004), không có ở repo gốc.\n"""\n\nfrom __future__ import annotations\n\nimport math\nimport threading\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Protocol\n\nimport numpy as np\n\nfrom vericlue_ml.common.detector import DetectorInput\nfrom vericlue_ml.common.device import resolve_torch_device\nfrom vericlue_ml.common.result import (\n    ErrorCode,\n    Evidence,\n    EvidenceType,\n    Module,\n    ModuleResult,\n    Status,\n)\nfrom vericlue_ml.face.frames import Frames, sample_images, sample_video\nfrom vericlue_ml.face.pad import (\n    FaceBox,\n    FaceFinder,\n    PadError,\n    ScrfdFaceFinder,\n    _models_root,\n    _verified_weights,\n    shared_scrfd_session,\n)\n\nGEND_MODEL_ID = "gend-clip-l14"\nCLIP_MODEL_ID = "clip-vit-large-patch14"\nGEND_WEIGHTS = "model.safetensors"\nGEND_CONFIG = "config.json"\nCLIP_CONFIG = "config.json"\nCLIP_PREPROCESSOR = "preprocessor_config.json"\n\n# Số khung lấy mẫu: giám định 32 theo task card FACE-02; eKYC 8 là lựa chọn của wrapper vì tốc độ\n# (ngân sách detector 4 s), xem bàn giao FACE-02.\nEKYC_FRAMES = 8\nFORENSIC_FRAMES = 32\nFORENSIC_MODE = "forensic"  # `DetectorInput.metadata["mode"]`; không có thì coi là eKYC\n\n# Hậu tố của model_version: cách gộp điểm khung → video. Đổi cách gộp (mean/top25/max) thì ĐỔI hậu tố này,\n# vì calibrator được khóa theo model_version. Số khung (8 hay 32) KHÔNG nằm trong version, mà ở evidence.\nAGGREGATION_TAG = "agg-maxmeantop25-v1"\n# Số luồng CPU của torch khi GenD chạy CPU: đủ nhanh mà không chiếm hết lõi của face_pad/quality/voice.\n# `torch.set_num_threads` là cài đặt toàn tiến trình (voice cũng đặt 4 qua `cpu_threads`).\nGEND_CPU_THREADS = 4\n\nALIGN_SCALE = 1.3  # tỷ lệ ngữ cảnh quanh mặt của GenD\nINPUT_SIZE = 224\nCLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)\nCLIP_STD = (0.26862954, 0.26130258, 0.27577711)\nSCRFD_SCALE = 128.0  # GenD chia 128 (không phải 127,5 như PAD) và đảo BGR→RGB\nTOP_FRACTION = 0.25\nBATCH_SIZE = 8\n# Ít hơn chừng này khung có mặt (hoặc ít hơn số khung đã lấy, nếu số đó nhỏ hơn) thì điểm không đủ\n# tin cậy → `degraded` → review. Hằng số riêng của FACE-02, KHÔNG nằm trong thresholds.yaml.\nMIN_VALID_FRAMES = 3\nMIN_FACE_SIDE_PX = 2.0\n\n_ALIGN_TARGET = np.array(\n    [[0.34, 0.46], [0.66, 0.46], [0.5, 0.64], [0.37, 0.82], [0.63, 0.82]], dtype=np.float32\n)\n\n\nclass GendError(PadError):\n    """Lỗi hạ tầng của GenD; `code` là ErrorCode sẽ ghi vào ModuleResult."""\n\n\nclass DeepfakeClassifier(Protocol):\n    def p_fake(self, batch: np.ndarray) -> list[float]:\n        """`batch` float32 [N, 3, 224, 224] đã chuẩn hóa; trả `p_fake` (softmax nhãn 1) mỗi mặt."""\n        ...\n\n\n# ── tiền xử lý (đúng repo GenD) ──────────────────────────────────────────────────────────\ndef align_face(image: np.ndarray, landmarks: np.ndarray, scale: float = ALIGN_SCALE) -> np.ndarray:\n    """Căn mặt theo 5 mốc, `target_size=None` như `detector.py::align_face` của GenD. Trả ảnh BGR vuông."""\n    import cv2\n\n    landmarks = np.asarray(landmarks, dtype=np.float32)\n    dst = _ALIGN_TARGET.copy()\n    desired = np.linalg.norm(landmarks[:, None, :] - landmarks[None, :, :], axis=-1)\n    reference = np.linalg.norm(dst[:, None, :] - dst[None, :, :], axis=-1)\n    upper = np.triu_indices(len(dst), k=1)\n    side = int(np.round(np.mean(desired[upper] / reference[upper]) * scale))\n    if side < 1:\n        raise GendError("Mốc mặt suy biến.", ErrorCode.E_NO_FACE)\n    size = (side, side)\n    dst[:, 0] *= size[0]\n    dst[:, 1] *= size[1]\n    margin = size[0] * (scale - 1) / 2.0\n    dst += margin\n    dst *= size[0] / (size[0] + 2 * margin)\n    matrix = cv2.estimateAffinePartial2D(landmarks, dst, method=cv2.LMEDS)[0]\n    if matrix is None:\n        raise GendError("Không căn được mặt.", ErrorCode.E_NO_FACE)\n    return cv2.warpAffine(image, matrix, size, flags=cv2.INTER_LINEAR)\n\n\ndef to_tensor(aligned_bgr: np.ndarray) -> np.ndarray:\n    """BGR → RGB, BICUBIC về 224 (PIL, như CLIPImageProcessor), /255, mean/std CLIP → [3, 224, 224]."""\n    from PIL import Image\n\n    rgb = np.ascontiguousarray(aligned_bgr[:, :, ::-1])\n    resized = Image.fromarray(rgb).resize((INPUT_SIZE, INPUT_SIZE), Image.BICUBIC)\n    pixels = np.asarray(resized, dtype=np.float32) / 255.0\n    pixels = (pixels - np.array(CLIP_MEAN, dtype=np.float32)) / np.array(CLIP_STD, dtype=np.float32)\n    return np.ascontiguousarray(pixels.transpose(2, 0, 1))\n\n\n# ── GenD (torch) ─────────────────────────────────────────────────────────────────────────\nclass GendClassifier:\n    """CLIP vision → pooler_output → L2-normalize → Linear(1024, 2) → softmax (nhãn 1 = giả)."""\n\n    def __init__(self, vision, head_weight, head_bias, device) -> None:\n        self._vision, self._device = vision, device\n        self._weight, self._bias = head_weight, head_bias\n\n    def p_fake(self, batch: np.ndarray) -> list[float]:\n        import torch\n        import torch.nn.functional as functional\n\n        out: list[float] = []\n        with torch.inference_mode():\n            for start in range(0, len(batch), BATCH_SIZE):\n                pixels = torch.from_numpy(batch[start : start + BATCH_SIZE]).to(self._device)\n                features = self._vision(pixel_values=pixels).pooler_output\n                logits = functional.linear(\n                    functional.normalize(features, p=2, dim=1), self._weight, self._bias\n                )\n                out.extend(float(v) for v in logits.softmax(dim=-1)[:, 1].cpu())\n        return out\n\n\ndef _limit_cpu_threads(target: str) -> None:\n    """Chạy CPU thì giới hạn luồng torch (GPU thì không đụng)."""\n    if target != "cpu":\n        return\n    import os\n\n    import torch\n\n    torch.set_num_threads(max(1, min(GEND_CPU_THREADS, os.cpu_count() or 1)))\n\n\ndef _load_classifier(root: Path, device: str | None) -> tuple[GendClassifier, str, str]:\n    """Kiểm SHA-256 rồi dựng GenD từ trọng số cục bộ. Trả (bộ phân loại, revision GenD, revision CLIP)."""\n    from safetensors.torch import load_file\n    from transformers import CLIPVisionConfig, CLIPVisionModel\n\n    _, clip_rev = _verified_weights(root, CLIP_MODEL_ID, CLIP_CONFIG)\n    _verified_weights(root, CLIP_MODEL_ID, CLIP_PREPROCESSOR)\n    _verified_weights(root, GEND_MODEL_ID, GEND_CONFIG)\n    weights, gend_rev = _verified_weights(root, GEND_MODEL_ID, GEND_WEIGHTS)\n\n    config = CLIPVisionConfig.from_pretrained(str(root / CLIP_MODEL_ID), local_files_only=True)\n    prefix = "feature_extractor.vision_model."\n    tensors = load_file(str(weights))\n    vision_state = {\n        f"vision_model.{key[len(prefix) :]}": value\n        for key, value in tensors.items()\n        if key.startswith(prefix)\n    }\n    head_weight, head_bias = tensors.get("model.linear.weight"), tensors.get("model.linear.bias")\n    if head_weight is None or head_bias is None or tuple(head_weight.shape) != (2, 1024):\n        raise GendError("Checkpoint GenD thiếu đầu tuyến tính 1024→2.")\n    import torch\n\n    # Dựng trên thiết bị `meta` (không cấp phát/khởi tạo ngẫu nhiên 300M tham số) rồi gán thẳng tensor đã\n    # nạp (assign=True): đỉnh RAM chỉ còn ~1 bản trọng số thay vì 2, và nạp nhanh hơn.\n    with torch.device("meta"):\n        vision = CLIPVisionModel(config)\n    try:\n        vision.load_state_dict(vision_state, strict=True, assign=True)\n    except RuntimeError as exc:\n        raise GendError("Checkpoint GenD không khớp kiến trúc CLIP ViT-L/14.") from exc\n    del tensors, vision_state\n    # Buffer không lưu trong checkpoint (position_ids) vẫn ở `meta`: dựng lại đúng như CLIPVisionEmbeddings.\n    embeddings = vision.vision_model.embeddings\n    embeddings.position_ids = torch.arange(embeddings.num_positions).expand((1, -1))\n    if any(t.is_meta for t in (*vision.parameters(), *vision.buffers())):\n        raise GendError("Còn tensor chưa được nạp trọng số sau khi dựng GenD.")\n    target = resolve_torch_device(device, component="gend")\n    vision = vision.to(target).eval()\n    _limit_cpu_threads(target)\n    return (\n        GendClassifier(vision, head_weight.to(target), head_bias.to(target), target),\n        gend_rev,\n        clip_rev,\n    )\n\n\n# ── Detector ─────────────────────────────────────────────────────────────────────────────\n@dataclass(frozen=True)\nclass _FrameScores:\n    indices: list[int]\n    p_fake: list[float]\n    sampled: int\n\n\ndef video_scores(p_fake: list[float]) -> tuple[float, float]:\n    """(mean, top25_mean): trung bình, và trung bình của ceil(25%) khung nghi nhất."""\n    values = sorted(p_fake, reverse=True)\n    top = values[: max(1, math.ceil(TOP_FRACTION * len(values)))]\n    return float(np.mean(values)), float(np.mean(top))\n\n\nclass FaceDeepfakeDetector:\n    """Bộ phát hiện `face_deepfake` thật. Nạp mô hình một lần (lười hoặc qua `preload`)."""\n\n    name = Module.FACE_DEEPFAKE.value\n\n    def __init__(\n        self,\n        models_dir: Path | str | None = None,\n        *,\n        finder: FaceFinder | None = None,\n        classifier: DeepfakeClassifier | None = None,\n        num_frames: int = EKYC_FRAMES,\n    ) -> None:\n        self._models_dir = models_dir\n        self._finder, self._classifier = finder, classifier\n        self._num_frames = num_frames\n        self.version = (\n            f"gend-clip-l14+scrfd-10g+{AGGREGATION_TAG}"\n            if finder is None\n            else f"gend-clip-l14+injected+{AGGREGATION_TAG}"\n        )\n        self._lock = threading.Lock()\n        self._loaded_root: Path | None = None\n\n    def preload(self, *, models_dir: Path | str | None = None, device: str | None = None) -> None:\n        with self._lock:\n            root = _models_root(models_dir if models_dir is not None else self._models_dir)\n            if self._finder is not None and self._classifier is not None:\n                if (\n                    models_dir is not None\n                    and self._loaded_root is not None\n                    and root.resolve() != self._loaded_root\n                ):\n                    raise GendError("GenD đã nạp với MODELS_DIR khác; cần restart.")\n                return\n            session, det_rev = shared_scrfd_session(root, device)\n            classifier, gend_rev, clip_rev = _load_classifier(root, device)\n            self._finder = ScrfdFaceFinder(session, scale=SCRFD_SCALE, swap_rb=True)\n            self._classifier = classifier\n            self._loaded_root = root.resolve()\n            self.version = (\n                f"gend-clip-l14@{gend_rev[:8]}+clip@{clip_rev[:8]}"\n                f"+scrfd-10g@{det_rev[:8]}+{AGGREGATION_TAG}"\n            )\n\n    def _largest_face(self, frame: np.ndarray) -> FaceBox | None:\n        faces = [\n            f\n            for f in self._finder.find(frame)\n            if f.landmarks is not None\n            and f.x2 - f.x1 >= MIN_FACE_SIDE_PX\n            and f.y2 - f.y1 >= MIN_FACE_SIDE_PX\n        ]\n        return max(faces, key=lambda f: f.area) if faces else None\n\n    def _score_frames(self, sampled: Frames, requested: int) -> _FrameScores:\n        """Duyệt từng khung (không giữ khung gốc), chỉ giữ crop đã chuẩn hóa. `requested` là số khung\n        đã yêu cầu: khung giải mã hỏng hay không có mặt đều tính là thiếu."""\n        indices: list[int] = []\n        crops: list[np.ndarray] = []\n        for index, frame in sampled:\n            face = self._largest_face(frame)\n            if face is None:\n                continue\n            try:\n                crops.append(to_tensor(align_face(frame, np.array(face.landmarks))))\n            except GendError:\n                continue\n            indices.append(index)\n        if not crops:\n            return _FrameScores([], [], requested)\n        p_fake = self._classifier.p_fake(np.stack(crops))\n        if len(p_fake) != len(crops) or not all(math.isfinite(v) for v in p_fake):\n            raise GendError("Mô hình trả điểm không hợp lệ.")\n        return _FrameScores(indices, p_fake, requested)\n\n    def predict(self, inp: DetectorInput) -> ModuleResult:\n        started = time.perf_counter()\n\n        def result(status: Status, **fields: Any) -> ModuleResult:\n            return ModuleResult(\n                module=Module.FACE_DEEPFAKE,\n                model_version=self.version,\n                status=status,\n                latency_ms=(time.perf_counter() - started) * 1000,\n                **fields,\n            )\n\n        try:\n            count = (\n                FORENSIC_FRAMES if inp.metadata.get("mode") == FORENSIC_MODE else self._num_frames\n            )\n            if inp.video is not None:\n                self.preload()\n                requested, sampled = sample_video(Path(inp.video), count)\n            elif inp.images:\n                self.preload()\n                requested, sampled = sample_images(list(inp.images), count)\n            else:\n                raise GendError("Thiếu video hoặc ảnh mặt.", ErrorCode.E_DECODE)\n            scores = self._score_frames(sampled, requested)\n            counts = {\n                "frames_sampled": float(scores.sampled),\n                "frames_used": float(len(scores.p_fake)),\n            }\n            if not scores.p_fake:\n                return result(Status.NOT_APPLICABLE, error_code=ErrorCode.E_NO_FACE, quality=counts)\n            mean, top = video_scores(scores.p_fake)\n            enough = len(scores.p_fake) >= min(MIN_VALID_FRAMES, scores.sampled)\n            return result(\n                Status.OK if enough else Status.DEGRADED,\n                error_code=None if enough else ErrorCode.E_LOW_QUALITY,\n                score=max(mean, top),\n                quality=counts,\n                evidence=[\n                    Evidence(\n                        type=EvidenceType.TIMESERIES,\n                        name="frame_scores",\n                        value=[\n                            {"frame": i, "p_fake": round(p, 6)}\n                            for i, p in zip(scores.indices, scores.p_fake, strict=True)\n                        ],\n                        label_vi="Điểm giả mạo theo khung hình (chưa hiệu chỉnh)",\n                    ),\n                    Evidence(\n                        type=EvidenceType.SCALAR,\n                        name="mean",\n                        value=mean,\n                        label_vi="Trung bình các khung (chưa hiệu chỉnh)",\n                    ),\n                    Evidence(\n                        type=EvidenceType.SCALAR,\n                        name="top25_mean",\n                        value=top,\n                        label_vi="Trung bình 25% khung nghi nhất (chưa hiệu chỉnh)",\n                    ),\n                ],\n            )\n        except PadError as exc:\n            return result(Status.ERROR, error_code=exc.code)\n        except Exception:  # noqa: BLE001 - mọi lỗi khác đều thành \'error\', không bao giờ là đạt\n            return result(Status.ERROR, error_code=ErrorCode.E_INTERNAL)\n'),
    'vericlue_ml/face/pad.py': ('2ddebf5434c03c1d91060a55d40d72a78a7d4f380b0988a27326252d5cf186db', '"""FACE-01 — PAD (chống ảnh in / màn hình phát lại) bằng MiniFASNet V2 chạy qua onnxruntime.\n\nLuồng: ảnh → SCRFD `det_10g` tìm mặt → cắt vùng quanh mặt (hệ số 2,7) → MiniFASNet → softmax →\n`score = 1 - p_real` (điểm thô; `prob_attack` để trống chờ hiệu chỉnh). Không có mặt → `not_applicable`; hai mặt cỡ tương đương → `degraded`\n(không bao giờ được coi là đạt); thiếu trọng số, sai SHA-256 hay lỗi bất kỳ → `error`.\n\nTrọng số đọc từ `MODELS_DIR/<id trong models/manifest.yaml>/<file>` và kiểm SHA-256 theo manifest.\nMã KHÔNG gọi mạng. `prob_attack` là xác suất SAU hiệu chỉnh (architecture §3) nên ở đây luôn None;\nTrustRouter tự hiệu chỉnh từ `score` bằng calibrator của đúng `model_version` (ROUT-01).\n\nNguồn tiền xử lý và giải mã (chép lại, giữ giấy phép — xem `third_party/README.md`):\n- uniface 4.0.0 (MIT, Copyright 2025-2026 Yakhyokhuja Valikhujaev, https://github.com/yakhyo/uniface):\n  `uniface/spoofing/minifasnet.py` (cắt theo `scale`, float32 không chuẩn hóa, NCHW, softmax;\n  nhãn 1 = thật) và `uniface/detection/scrfd.py` + `uniface/common.py` (letterbox 640, chuẩn hóa\n  (x-127,5)/127,5, giải mã anchor 3 tầng stride 8/16/32 × 2 anchor, NMS 0,4, ngưỡng 0,5).\n- SCRFD: "Sample and Computation Redistribution for Efficient Face Detection"\n  (insightface). Trọng số `det_10g.onnx` là giấy phép NGHIÊN CỨU PHI THƯƠNG MẠI.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport os\nimport threading\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Protocol\n\nimport numpy as np\nimport yaml\n\nfrom vericlue_ml.common.detector import DetectorInput\nfrom vericlue_ml.common.device import check_onnx_session, onnx_providers, requested_device\nfrom vericlue_ml.common.result import (\n    ErrorCode,\n    Evidence,\n    EvidenceType,\n    Module,\n    ModuleResult,\n    Status,\n)\n\nREPO_ROOT = Path(__file__).resolve().parents[4]\n\nPAD_MODEL_ID = "minifasnet"\nDETECTOR_MODEL_ID = "scrfd-10g"\nPAD_FILE = "minifasnet_v2.onnx"\nDETECTOR_FILE = "det_10g.onnx"\n\n# Mặt thứ hai nhỏ hơn rõ rệt (diện tích < 50% mặt lớn nhất, ví dụ ảnh trên CCCD hoặc người ở xa)\n# thì bỏ qua và dùng mặt lớn nhất. Hằng số riêng của FACE-01, KHÔNG nằm trong thresholds.yaml.\nSECOND_FACE_AREA_RATIO = 0.5\n\nPAD_CROP_SCALE = 2.7  # MiniFASNet V2 (uniface DEFAULT_SCALES)\nDET_INPUT_SIZE = 640\nDET_CONFIDENCE = 0.5\nDET_NMS = 0.4\n_STRIDES = (8, 16, 32)\n_ANCHORS_PER_CELL = 2\n\n\nclass PadError(Exception):\n    """Lỗi hạ tầng của PAD; `code` là ErrorCode sẽ ghi vào ModuleResult."""\n\n    def __init__(self, message: str, code: ErrorCode = ErrorCode.E_INTERNAL) -> None:\n        super().__init__(message)\n        self.code = code\n\n\n@dataclass(frozen=True)\nclass FaceBox:\n    x1: float\n    y1: float\n    x2: float\n    y2: float\n    confidence: float\n    # 5 mốc (mắt trái, mắt phải, mũi, khóe miệng trái, khóe miệng phải) theo SCRFD; None nếu không có.\n    landmarks: tuple[tuple[float, float], ...] | None = None\n\n    @property\n    def area(self) -> float:\n        return max(0.0, self.x2 - self.x1) * max(0.0, self.y2 - self.y1)\n\n\nclass FaceFinder(Protocol):\n    def find(self, image: np.ndarray) -> list[FaceBox]: ...\n\n\nclass SpoofClassifier(Protocol):\n    def p_real(self, image: np.ndarray, box: FaceBox) -> float: ...\n\n\n# ── trọng số ─────────────────────────────────────────────────────────────────────────────\ndef _models_root(models_dir: Path | str | None) -> Path:\n    root = Path(models_dir or os.environ.get("MODELS_DIR") or REPO_ROOT / "models")\n    return root if root.is_absolute() else REPO_ROOT / root\n\n\ndef _manifest_entry(model_id: str, filename: str) -> dict[str, Any]:\n    path = Path(os.environ.get("ML_MODELS_MANIFEST") or REPO_ROOT / "models" / "manifest.yaml")\n    data = yaml.safe_load(path.read_text(encoding="utf-8"))\n    for model in data.get("models", []):\n        if model.get("id") == model_id:\n            for file in model.get("files", []):\n                if file.get("path") == filename:\n                    return {**file, "revision": model.get("source", {}).get("revision", "")}\n    raise PadError(f"Manifest không có {model_id}/{filename}.")\n\n\ndef _verified_weights(root: Path, model_id: str, filename: str) -> tuple[Path, str]:\n    """Trả (đường dẫn file, revision). Thiếu file hoặc sai SHA-256 thì ném PadError — không tải."""\n    entry = _manifest_entry(model_id, filename)\n    path = root / model_id / filename\n    if not path.is_file():\n        raise PadError(f"Thiếu trọng số {model_id}/{filename}; chạy scripts/download_models.py.")\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(1 << 20), b""):\n            digest.update(block)\n    if digest.hexdigest() != entry["sha256"]:\n        raise PadError(f"SHA-256 của {model_id}/{filename} sai.", ErrorCode.E_INTEGRITY)\n    return path, str(entry["revision"])\n\n\ndef _providers(device: str | None = None) -> list[str]:\n    """CPU luôn chạy được; `DEVICE=cuda|auto` thì ưu tiên CUDA nếu có, không có thì lùi CPU (có log)."""\n    return onnx_providers(device, component="onnx")\n\n\ndef _session(path: Path, device: str | None = None):\n    import onnxruntime as ort\n\n    options = ort.SessionOptions()\n    options.log_severity_level = 3\n    session = ort.InferenceSession(str(path), sess_options=options, providers=_providers(device))\n    check_onnx_session(session, device, component=path.name)\n    return session\n\n\n_SCRFD_SESSIONS: dict[tuple[Path, str], tuple[Any, str]] = {}\n_SCRFD_LOCK = threading.Lock()\n\n\ndef shared_scrfd_session(root: Path, device: str | None = None) -> tuple[Any, str]:\n    """Phiên onnxruntime của SCRFD dùng chung cho face_pad, face_deepfake và quality (nạp một lần).\n\n    Trả (session, revision). Thiếu file hoặc sai SHA-256 thì ném PadError — không tải.\n    """\n    key = (root.resolve(), requested_device(device))\n    with _SCRFD_LOCK:\n        if key not in _SCRFD_SESSIONS:\n            path, revision = _verified_weights(root, DETECTOR_MODEL_ID, DETECTOR_FILE)\n            _SCRFD_SESSIONS[key] = (_session(path, device), revision)\n        return _SCRFD_SESSIONS[key]\n\n\n# ── SCRFD (tìm mặt) ──────────────────────────────────────────────────────────────────────\ndef _letterbox(image: np.ndarray, size: int) -> tuple[np.ndarray, float]:\n    import cv2\n\n    height, width = image.shape[:2]\n    ratio = height / width\n    if ratio > 1.0:\n        new_h, new_w = size, int(size / ratio)\n    else:\n        new_w, new_h = size, int(size * ratio)\n    factor = new_h / height\n    canvas = np.zeros((size, size, 3), dtype=np.uint8)\n    canvas[:new_h, :new_w] = cv2.resize(image, (new_w, new_h))\n    return canvas, factor\n\n\ndef _nms(dets: np.ndarray, threshold: float) -> list[int]:\n    x1, y1, x2, y2, scores = dets.T\n    areas = (x2 - x1 + 1) * (y2 - y1 + 1)\n    order = scores.argsort()[::-1]\n    keep: list[int] = []\n    while order.size > 0:\n        i = order[0]\n        keep.append(int(i))\n        xx1 = np.maximum(x1[i], x1[order[1:]])\n        yy1 = np.maximum(y1[i], y1[order[1:]])\n        xx2 = np.minimum(x2[i], x2[order[1:]])\n        yy2 = np.minimum(y2[i], y2[order[1:]])\n        inter = np.maximum(0.0, xx2 - xx1 + 1) * np.maximum(0.0, yy2 - yy1 + 1)\n        overlap = inter / (areas[i] + areas[order[1:]] - inter)\n        order = order[np.where(overlap <= threshold)[0] + 1]\n    return keep\n\n\nclass ScrfdFaceFinder:\n    """SCRFD det_10g. Mặc định giữ đúng tiền xử lý của FACE-01 (uniface); GenD (FACE-02) dùng\n    `scale=128, swap_rb=True` như `src/retinaface.py` của repo GenD."""\n\n    def __init__(self, session, *, scale: float = 127.5, swap_rb: bool = False) -> None:\n        self._scale = scale\n        self._swap_rb = swap_rb\n        self._session = session\n        self._input = session.get_inputs()[0].name\n        self._outputs = [o.name for o in session.get_outputs()]\n        if len(self._outputs) != 9:\n            raise PadError("det_10g.onnx không có 9 đầu ra (3 tầng × điểm, hộp, mốc mặt).")\n        self._centers: dict[tuple[int, int], np.ndarray] = {}\n\n    def _anchor_centers(self, cells: int, stride: int) -> np.ndarray:\n        key = (cells, stride)\n        if key not in self._centers:\n            y, x = np.mgrid[:cells, :cells]\n            centers = (np.stack((x, y), axis=-1).astype(np.float32) * stride).reshape(-1, 2)\n            self._centers[key] = np.tile(centers[:, None, :], (1, _ANCHORS_PER_CELL, 1)).reshape(\n                -1, 2\n            )\n        return self._centers[key]\n\n    def find(self, image: np.ndarray) -> list[FaceBox]:\n        canvas, factor = _letterbox(image, DET_INPUT_SIZE)\n        if self._swap_rb:\n            canvas = np.ascontiguousarray(canvas[:, :, ::-1])\n        tensor = ((canvas.astype(np.float32) - 127.5) / self._scale).transpose(2, 0, 1)[None]\n        outputs = self._session.run(self._outputs, {self._input: tensor})\n        boxes: list[np.ndarray] = []\n        points: list[np.ndarray] = []\n        scores: list[np.ndarray] = []\n        for index, stride in enumerate(_STRIDES):\n            score = outputs[index]\n            distance = outputs[len(_STRIDES) + index] * stride\n            keep = np.where(score >= DET_CONFIDENCE)[0]\n            if keep.size == 0:\n                continue\n            centers = self._anchor_centers(DET_INPUT_SIZE // stride, stride)\n            decoded = np.stack(\n                [\n                    centers[:, 0] - distance[:, 0],\n                    centers[:, 1] - distance[:, 1],\n                    centers[:, 0] + distance[:, 2],\n                    centers[:, 1] + distance[:, 3],\n                ],\n                axis=-1,\n            )\n            offsets = outputs[2 * len(_STRIDES) + index] * stride\n            marks = np.stack(\n                [centers[:, i % 2] + offsets[:, i] for i in range(10)], axis=-1\n            ).reshape(-1, 5, 2)\n            boxes.append(decoded[keep])\n            points.append(marks[keep])\n            scores.append(score[keep])\n        if not boxes:\n            return []\n        all_scores = np.vstack(scores)\n        order = all_scores.ravel().argsort()[::-1]\n        dets = np.hstack((np.vstack(boxes) / factor, all_scores)).astype(np.float32)[order]\n        landmarks = (np.vstack(points) / factor)[order]\n        keep = _nms(dets, DET_NMS)\n        return [\n            FaceBox(\n                *(float(v) for v in dets[i]),\n                landmarks=tuple((float(x), float(y)) for x, y in landmarks[i]),\n            )\n            for i in keep\n        ]\n\n\n# ── MiniFASNet (PAD) ─────────────────────────────────────────────────────────────────────\ndef _softmax(x: np.ndarray) -> np.ndarray:\n    e = np.exp(x - np.max(x, axis=-1, keepdims=True))\n    return e / np.sum(e, axis=-1, keepdims=True)\n\n\nclass MiniFasNetClassifier:\n    def __init__(self, session, scale: float = PAD_CROP_SCALE) -> None:\n        self._session = session\n        self._input = session.get_inputs()[0].name\n        self._output = session.get_outputs()[0].name\n        height, width = session.get_inputs()[0].shape[2:4]\n        self._size = (int(width), int(height))\n        self._scale = scale\n\n    def _crop(self, image: np.ndarray, box: FaceBox) -> np.ndarray:\n        import cv2\n\n        src_h, src_w = image.shape[:2]\n        x, y = int(box.x1), int(box.y1)\n        box_w, box_h = int(box.x2 - box.x1), int(box.y2 - box.y1)\n        scale = min((src_h - 1) / box_h, (src_w - 1) / box_w, self._scale)\n        new_w, new_h = box_w * scale, box_h * scale\n        cx, cy = x + box_w / 2, y + box_h / 2\n        x1, y1 = max(0, int(cx - new_w / 2)), max(0, int(cy - new_h / 2))\n        x2, y2 = min(src_w - 1, int(cx + new_w / 2)), min(src_h - 1, int(cy + new_h / 2))\n        return cv2.resize(image[y1 : y2 + 1, x1 : x2 + 1], self._size)\n\n    def p_real(self, image: np.ndarray, box: FaceBox) -> float:\n        face = self._crop(image, box).astype(np.float32)  # BGR, không chuẩn hóa\n        tensor = np.expand_dims(np.transpose(face, (2, 0, 1)), axis=0)\n        logits = self._session.run([self._output], {self._input: tensor})[0]\n        return float(_softmax(logits)[0, 1])  # nhãn 1 = thật\n\n\n# ── Detector ─────────────────────────────────────────────────────────────────────────────\ndef _decode(path: Path) -> np.ndarray:\n    import cv2\n\n    try:\n        data = np.fromfile(str(path), dtype=np.uint8)  # đường dẫn Unicode trên Windows\n        image = cv2.imdecode(data, cv2.IMREAD_COLOR)\n    except Exception as exc:\n        raise PadError("Không đọc được ảnh.", ErrorCode.E_DECODE) from exc\n    if image is None or image.size == 0:\n        raise PadError("Không giải mã được ảnh.", ErrorCode.E_DECODE)\n    return image\n\n\nclass FacePadDetector:\n    """Bộ phát hiện `face_pad` thật. Nạp mô hình một lần (lười hoặc qua `preload`)."""\n\n    name = Module.FACE_PAD.value\n\n    def __init__(\n        self,\n        models_dir: Path | str | None = None,\n        *,\n        finder: FaceFinder | None = None,\n        classifier: SpoofClassifier | None = None,\n    ) -> None:\n        self._models_dir = models_dir\n        self._finder, self._classifier = finder, classifier\n        self.version = "minifasnet-v2+scrfd-10g" if finder is None else "minifasnet-v2+injected"\n        self._lock = threading.Lock()\n        self._loaded_root: Path | None = None\n\n    def preload(self, *, models_dir: Path | str | None = None, device: str | None = None) -> None:\n        with self._lock:\n            root = _models_root(models_dir if models_dir is not None else self._models_dir)\n            if self._finder is not None and self._classifier is not None:\n                if (\n                    models_dir is not None\n                    and self._loaded_root is not None\n                    and root.resolve() != self._loaded_root\n                ):\n                    raise PadError("PAD đã nạp với MODELS_DIR khác; cần restart.")\n                return\n            pad_path, pad_rev = _verified_weights(root, PAD_MODEL_ID, PAD_FILE)\n            det_session, det_rev = shared_scrfd_session(root, device)\n            self._classifier = MiniFasNetClassifier(_session(pad_path, device))\n            self._finder = ScrfdFaceFinder(det_session)\n            self._loaded_root = root.resolve()\n            self.version = f"minifasnet-v2@{pad_rev[:8]}+scrfd-10g@{det_rev[:8]}"\n\n    def predict(self, inp: DetectorInput) -> ModuleResult:\n        started = time.perf_counter()\n\n        def result(status: Status, **fields: Any) -> ModuleResult:\n            return ModuleResult(\n                module=Module.FACE_PAD,\n                model_version=self.version,\n                status=status,\n                latency_ms=(time.perf_counter() - started) * 1000,\n                **fields,\n            )\n\n        try:\n            if not inp.images:\n                raise PadError("Thiếu ảnh mặt.", ErrorCode.E_DECODE)\n            self.preload()\n            image = _decode(inp.images[0])\n            faces = self._finder.find(image)\n            if not faces:\n                return result(Status.NOT_APPLICABLE, error_code=ErrorCode.E_NO_FACE)\n            faces = sorted(faces, key=lambda f: f.area, reverse=True)\n            largest = faces[0]\n            if len(faces) > 1 and faces[1].area >= SECOND_FACE_AREA_RATIO * largest.area:\n                return result(\n                    Status.DEGRADED,\n                    error_code=ErrorCode.E_MULTI_FACE,\n                    quality={"faces": float(len(faces))},\n                )\n            if largest.x2 - largest.x1 < 2 or largest.y2 - largest.y1 < 2:\n                return result(Status.NOT_APPLICABLE, error_code=ErrorCode.E_NO_FACE)\n            p_real = self._classifier.p_real(image, largest)\n            if not np.isfinite(p_real):\n                raise PadError("Mô hình trả điểm không hợp lệ.")\n            height, width = image.shape[:2]\n            return result(\n                Status.OK,\n                score=1.0 - p_real,\n                quality={\n                    "faces": float(len(faces)),\n                    "face_area_ratio": largest.area / float(height * width),\n                    "det_confidence": largest.confidence,\n                },\n                evidence=[\n                    Evidence(\n                        type=EvidenceType.SCALAR,\n                        name="p_real",\n                        value=p_real,\n                        label_vi="Xác suất ảnh là mặt thật (chưa hiệu chỉnh)",\n                    )\n                ],\n            )\n        except PadError as exc:\n            return result(Status.ERROR, error_code=exc.code)\n        except Exception:  # noqa: BLE001 - mọi lỗi khác đều thành \'error\', không bao giờ là đạt\n            return result(Status.ERROR, error_code=ErrorCode.E_INTERNAL)\n'),
    'vericlue_ml/finetune/__init__.py': ('06839cc59b322f7c1b3c28bdcf429b86a8bc1f79b08d9a733a9910724628c070', '"""Hàm thuần cho tinh chỉnh GenD trên Kaggle/Colab (FACE-04).\n\nKhông chứa và không gọi công cụ tạo giả nào: nhóm không tự tạo deepfake người thật. Ở đây chỉ có chia\ntập, manifest, cổng lộ đề, chặn tập test và các bước tiền xử lý/tăng cường dùng chung. Notebook nhúng\nnguyên văn các file này (`scripts/build_gend_notebooks.py`).\n"""\n'),
    'vericlue_ml/finetune/data04.py': ('fbb3fe800f8a591b599f49bdf226a36cc31c84ff12b1423a025f36d9fa8669e3', '"""Hàm thuần dùng chung cho tinh chỉnh GenD trên Kaggle/Colab (FACE-04): chia tập, cổng lộ đề, manifest.\n\nKHÔNG có công cụ tạo giả và KHÔNG lập kế hoạch tạo giả: nhóm không tự tạo deepfake người thật (quyết\nđịnh trưởng nhóm 19/10). Mẫu giả chỉ lấy từ bộ nghiên cứu công khai có sẵn. Ở đây chỉ có: đọc từng\nfile trong ZIP từ xa bằng HTTP Range (VLR, Zenodo 19705466, CC BY 4.0 — chỉ để ĐO chặn nhầm người\nViệt), gom người bằng ArcFace (thà gộp nhầm còn hơn tách sót), chia theo nhóm 60/20/20, manifest,\nlệnh ffmpeg mã hóa lại dùng chung cho cả hai lớp, đặc trưng thô và cổng kiểm lộ đề.\n\nQuy ước:\n- Đơn vị chia tập là **nhóm**: thành phần liên thông của đồ thị "cùng người (cos ArcFace ≥ ngưỡng,\n  liên kết đơn trên mọi cặp khung)" ∪ "cùng video nguồn". Mẫu giả luôn cùng nhóm với video thật\n  nguồn của nó. Hai mẫu khác tập không bao giờ có cặp khung giống nhau ≥ ngưỡng\n  (`cross_split_max_similarity`).\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport random\nimport re\nimport struct\nimport time\nimport zlib\nfrom collections import defaultdict\nfrom collections.abc import Iterable, Sequence\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nSPLITS = ("train", "calib", "test")\nSPLIT_RATIOS = (0.6, 0.2, 0.2)\nCLIP_SECONDS = 3.0\nTARGET_FPS = 25\nCLIP_SIZE = 320\n# Cổng lộ đề: AUC của bộ phân loại chỉ dùng đặc trưng thô phải nằm trong khoảng này.\nLEAK_AUC_RANGE = (0.4, 0.6)\n# Ít hơn chừng này mẫu mỗi lớp thì AUC quá nhiễu: chỉ cảnh báo, không chặn (chế độ SMOKE).\nLEAK_MIN_PER_CLASS = 40\nLEAK_FEATURES = (\n    "bitrate_kbps",\n    "size_bytes",\n    "fps",\n    "duration_s",\n    "width",\n    "height",\n    "sharpness",\n    "brightness",\n)\nVLR_RECORD = "19705466"\nVLR_URL = "https://zenodo.org/api/records/{record}/files/{part}.zip/content"\n\n\n# ── ZIP từ xa (HTTP Range) ───────────────────────────────────────────────────────────────\nclass RemoteZipError(RuntimeError):\n    """Không đọc được ZIP từ xa (máy chủ không hỗ trợ Range, mạng lỗi, CRC sai…)."""\n\n\ndef _get_range(\n    session, url: str, start: int, end: int, retries: int = 5, allow_short: bool = False\n) -> bytes:\n    """GET bytes [start, end] (cả hai đầu), thử lại có lùi thời gian; bắt buộc trả 206.\n    `allow_short`: chấp nhận ít hơn (đoạn vượt cuối file)."""\n    last: Exception | None = None\n    for attempt in range(retries):\n        try:\n            response = session.get(url, headers={"Range": f"bytes={start}-{end}"}, timeout=120)\n            if response.status_code != 206:\n                raise RemoteZipError(f"Máy chủ trả {response.status_code}, không hỗ trợ Range.")\n            data = response.content\n            if len(data) != end - start + 1 and not (allow_short and data):\n                raise RemoteZipError("Nhận thiếu dữ liệu.")\n            return data\n        except Exception as exc:  # noqa: BLE001 - lỗi mạng thì thử lại\n            last = exc\n            time.sleep(min(60.0, 2.0**attempt))\n    raise RemoteZipError(f"Không tải được bytes {start}-{end}: {last}")\n\n\nclass HttpRangeFile(io.RawIOBase):\n    """File chỉ-đọc trên HTTP, mỗi lần `read` là một yêu cầu Range (đủ để `zipfile` đọc mục lục)."""\n\n    def __init__(self, url: str, session=None) -> None:\n        import requests\n\n        self.session = session or requests.Session()\n        head = self.session.head(url, allow_redirects=True, timeout=60)\n        head.raise_for_status()\n        self.url = head.url\n        self.size = int(head.headers["Content-Length"])\n        self.pos = 0\n        self.fetched = 0\n\n    def seekable(self) -> bool:\n        return True\n\n    def readable(self) -> bool:\n        return True\n\n    def tell(self) -> int:\n        return self.pos\n\n    def seek(self, offset: int, whence: int = 0) -> int:\n        self.pos = {0: offset, 1: self.pos + offset, 2: self.size + offset}[whence]\n        return self.pos\n\n    def readinto(self, buffer) -> int:\n        if self.pos >= self.size or len(buffer) == 0:\n            return 0\n        end = min(self.size, self.pos + len(buffer)) - 1\n        data = _get_range(self.session, self.url, self.pos, end)\n        buffer[: len(data)] = data\n        self.pos += len(data)\n        self.fetched += len(data)\n        return len(data)\n\n\n@dataclass(frozen=True)\nclass ZipMember:\n    name: str\n    header_offset: int\n    compress_size: int\n    file_size: int\n    compress_type: int\n    crc: int\n\n\ndef list_remote_zip(url: str, session=None) -> list[ZipMember]:\n    """Đọc mục lục (central directory) của ZIP từ xa; chỉ tải vài MB cuối file."""\n    import zipfile\n\n    raw = HttpRangeFile(url, session)\n    with zipfile.ZipFile(io.BufferedReader(raw, buffer_size=1 << 20)) as archive:\n        return [\n            ZipMember(\n                i.filename, i.header_offset, i.compress_size, i.file_size, i.compress_type, i.CRC\n            )\n            for i in archive.infolist()\n            if not i.is_dir()\n        ]\n\n\ndef parse_local_header(header: bytes) -> int:\n    """Độ dài phần đầu cục bộ (30 + tên + extra) của một mục ZIP."""\n    if len(header) < 30 or header[:4] != b"PK\\x03\\x04":\n        raise RemoteZipError("Sai chữ ký local header.")\n    name_len, extra_len = struct.unpack("<HH", header[26:30])\n    return 30 + name_len + extra_len\n\n\ndef decode_member(member: ZipMember, payload: bytes) -> bytes:\n    """Giải nén dữ liệu thô của một mục (stored hoặc deflate) và kiểm CRC-32."""\n    if member.compress_type == 0:\n        data = payload\n    elif member.compress_type == 8:\n        data = zlib.decompressobj(-15).decompress(payload)\n    else:\n        raise RemoteZipError(f"Kiểu nén {member.compress_type} chưa hỗ trợ.")\n    if len(data) != member.file_size or (zlib.crc32(data) & 0xFFFFFFFF) != member.crc:\n        raise RemoteZipError(f"CRC/kích thước sai: {member.name}")\n    return data\n\n\ndef fetch_member(url: str, member: ZipMember, session=None) -> bytes:\n    """Tải đúng một mục trong ZIP từ xa (1–2 yêu cầu Range). An toàn khi gọi song song (mỗi luồng một session)."""\n    if session is None:\n        import requests\n\n        session = requests.Session()\n    guess = 30 + len(member.name.encode()) + 256\n    end = member.header_offset + guess + member.compress_size - 1\n    blob = _get_range(session, url, member.header_offset, end, allow_short=True)\n    start = parse_local_header(blob[:30])\n    stop = start + member.compress_size\n    if stop > len(blob):\n        blob += _get_range(\n            session, url, member.header_offset + len(blob), member.header_offset + stop - 1\n        )\n    return decode_member(member, blob[start:stop])\n\n\n# ── VLR: tên file → nguồn ────────────────────────────────────────────────────────────────\n@dataclass(frozen=True)\nclass VlrClip:\n    member: str  # đường dẫn trong ZIP, ví dụ part4/HAVEA12/MOICH251/MOICH251_1.mp4\n    part: str\n    playlist: str\n    video: str  # thư mục video YouTube (đơn vị "cùng nguồn")\n    clip_id: str  # part4__HAVEA12__MOICH251__MOICH251_1\n    index: int\n    size: int = 0\n\n\n_VLR_RE = re.compile(r"^(part\\d+)/([^/]+)/([^/]+)/([^/]+?)_(\\d+)\\.mp4$")\n\n\ndef parse_vlr_member(name: str, size: int = 0) -> VlrClip | None:\n    """`partN/PLAYLIST/VIDEO/VIDEO_k.mp4` → VlrClip; file khác (csv, thư mục) → None."""\n    match = _VLR_RE.match(name)\n    if not match:\n        return None\n    part, playlist, video, _, index = match.groups()\n    clip_id = "__".join(name[: -len(".mp4")].split("/"))\n    return VlrClip(name, part, playlist, video, clip_id, int(index), size)\n\n\ndef plan_candidates(\n    clips: Iterable[VlrClip], videos_per_part: int, clips_per_video: int, seed: int\n) -> list[VlrClip]:\n    """Chọn clip ứng viên: xen kẽ playlist (đa dạng), trong mỗi video lấy clip cách đều nhau."""\n    by_video: dict[tuple[str, str, str], list[VlrClip]] = defaultdict(list)\n    for clip in clips:\n        by_video[(clip.part, clip.playlist, clip.video)].append(clip)\n    rng = random.Random(seed)\n    chosen: list[VlrClip] = []\n    for part in sorted({key[0] for key in by_video}):\n        playlists: dict[str, list[tuple[str, str, str]]] = defaultdict(list)\n        for key in sorted(k for k in by_video if k[0] == part):\n            playlists[key[1]].append(key)\n        for keys in playlists.values():\n            rng.shuffle(keys)\n        order = sorted(playlists)\n        rng.shuffle(order)\n        videos: list[tuple[str, str, str]] = []\n        while len(videos) < videos_per_part and any(playlists[p] for p in order):\n            for name in order:\n                if playlists[name] and len(videos) < videos_per_part:\n                    videos.append(playlists[name].pop())\n        for key in videos:\n            items = sorted(by_video[key], key=lambda c: c.index)\n            if len(items) <= clips_per_video:\n                chosen.extend(items)\n            else:\n                positions = np.linspace(0, len(items) - 1, clips_per_video).round().astype(int)\n                chosen.extend(items[int(i)] for i in sorted(set(positions.tolist())))\n    return chosen\n\n\n# ── ArcFace (gom người nói) ──────────────────────────────────────────────────────────────\n# Mốc đích chuẩn của ArcFace 112×112 (insightface `face_align.arcface_dst`).\nARCFACE_DST = np.array(\n    [\n        [38.2946, 51.6963],\n        [73.5318, 51.5014],\n        [56.0252, 71.7366],\n        [41.5493, 92.3655],\n        [70.7299, 92.2041],\n    ],\n    dtype=np.float32,\n)\n\n\ndef arcface_crop(image: np.ndarray, landmarks: np.ndarray) -> np.ndarray:\n    """Căn mặt về 112×112 theo 5 mốc (như `norm_crop` của insightface/uniface)."""\n    import cv2\n\n    matrix = cv2.estimateAffinePartial2D(\n        np.asarray(landmarks, dtype=np.float32), ARCFACE_DST, method=cv2.LMEDS\n    )[0]\n    if matrix is None:\n        raise ValueError("Không căn được mặt cho ArcFace.")\n    return cv2.warpAffine(image, matrix, (112, 112), borderValue=0.0)\n\n\ndef arcface_blob(crops: Sequence[np.ndarray]) -> np.ndarray:\n    """BGR 112×112 → NCHW float32, RGB, (x − 127,5)/127,5 (tiền xử lý của uniface ArcFace)."""\n    batch = np.stack([np.asarray(c, dtype=np.float32)[:, :, ::-1] for c in crops])\n    return np.ascontiguousarray(((batch - 127.5) / 127.5).transpose(0, 3, 1, 2))\n\n\ndef l2_normalize(vectors: np.ndarray) -> np.ndarray:\n    vectors = np.asarray(vectors, dtype=np.float32)\n    norms = np.linalg.norm(vectors, axis=-1, keepdims=True)\n    return vectors / np.maximum(norms, 1e-12)\n\n\nclass UnionFind:\n    def __init__(self, size: int) -> None:\n        self.parent = list(range(size))\n\n    def find(self, item: int) -> int:\n        while self.parent[item] != item:\n            self.parent[item] = self.parent[self.parent[item]]\n            item = self.parent[item]\n        return item\n\n    def union(self, a: int, b: int) -> None:\n        ra, rb = self.find(a), self.find(b)\n        if ra != rb:\n            self.parent[max(ra, rb)] = min(ra, rb)\n\n    def labels(self) -> list[int]:\n        """Nhãn thành phần 0..K-1, đánh số theo phần tử nhỏ nhất (ổn định)."""\n        roots: dict[int, int] = {}\n        out = []\n        for item in range(len(self.parent)):\n            root = self.find(item)\n            out.append(roots.setdefault(root, len(roots)))\n        return out\n\n\ndef _union_similar(uf: UnionFind, embeddings: Sequence[np.ndarray], threshold: float) -> None:\n    owners: list[int] = []\n    vectors: list[np.ndarray] = []\n    for owner, rows in enumerate(embeddings):\n        for row in np.atleast_2d(np.asarray(rows, dtype=np.float32)):\n            owners.append(owner)\n            vectors.append(row)\n    if not vectors:\n        return\n    matrix = l2_normalize(np.stack(vectors))\n    owner_array = np.asarray(owners)\n    for start in range(0, len(matrix), 512):\n        sims = matrix[start : start + 512] @ matrix.T\n        for left, right in zip(*np.nonzero(sims >= threshold), strict=True):\n            a, b = int(owner_array[start + int(left)]), int(owner_array[int(right)])\n            if a != b:\n                uf.union(a, b)\n\n\ndef cluster_persons(embeddings: Sequence[np.ndarray], threshold: float) -> list[int]:\n    """Nhãn người cho từng clip: liên kết đơn trên MỌI cặp khung (cos ≥ ngưỡng) rồi lấy bao đóng\n    bắc cầu. Ngưỡng thấp = gộp mạnh tay (thà gộp nhầm còn hơn tách sót). Mỗi phần tử của\n    `embeddings` là mảng [k, D] (k khung của một clip)."""\n    uf = UnionFind(len(embeddings))\n    _union_similar(uf, embeddings, threshold)\n    return uf.labels()\n\n\ndef group_components(person_of: Sequence[Any], video_of: Sequence[Any]) -> list[int]:\n    """Nhóm chia tập: clip cùng người HOẶC cùng video nguồn thì chung nhóm (bắc cầu)."""\n    if len(person_of) != len(video_of):\n        raise ValueError("person_of và video_of phải cùng độ dài.")\n    uf = UnionFind(len(person_of))\n    first: dict[tuple[str, Any], int] = {}\n    for i, (person, video) in enumerate(zip(person_of, video_of, strict=True)):\n        for key in (("p", person), ("v", video)):\n            if key in first:\n                uf.union(first[key], i)\n            else:\n                first[key] = i\n    return uf.labels()\n\n\ndef select_persons(\n    person_of: Sequence[int],\n    clip_ids: Sequence[str],\n    *,\n    max_persons: int,\n    max_clips_per_person: int,\n    min_clips_per_person: int,\n    seed: int,\n    video_of: Sequence[str] | None = None,\n) -> dict[int, list[str]]:\n    """Chọn ≤ `max_persons` người (ưu tiên người nhiều clip), mỗi người ≤ `max_clips_per_person` clip,\n    xen kẽ các video khác nhau. Trả {nhãn người: [clip_id]} theo thứ tự xác định."""\n    clips_by_person: dict[int, list[int]] = defaultdict(list)\n    for i, person in enumerate(person_of):\n        clips_by_person[int(person)].append(i)\n    rng = random.Random(seed)\n    eligible = sorted(\n        p for p, items in clips_by_person.items() if len(items) >= min_clips_per_person\n    )\n    rng.shuffle(eligible)\n    eligible.sort(key=lambda p: -min(len(clips_by_person[p]), max_clips_per_person))\n    chosen: dict[int, list[str]] = {}\n    for person in eligible[:max_persons]:\n        items = sorted(clips_by_person[person], key=lambda i: clip_ids[i])\n        if video_of is not None:\n            by_video: dict[str, list[int]] = defaultdict(list)\n            for i in items:\n                by_video[video_of[i]].append(i)\n            queues = [by_video[v] for v in sorted(by_video)]\n            items = []\n            while any(queues):\n                for queue in queues:\n                    if queue:\n                        items.append(queue.pop(0))\n        chosen[person] = [clip_ids[i] for i in items[:max_clips_per_person]]\n    return dict(sorted(chosen.items()))\n\n\ndef assign_splits(\n    group_weights: dict[str, int],\n    ratios: Sequence[float] = SPLIT_RATIOS,\n    seed: int = 0,\n    names: Sequence[str] = SPLITS,\n) -> dict[str, str]:\n    """Gán từng nhóm (trọng số = số người) vào train/calib/test, tham lam theo phần thiếu hụt\n    tương đối; nhóm lớn xếp trước. Xác định hoàn toàn theo (nhóm, seed)."""\n    if len(ratios) != len(names) or not math.isclose(sum(ratios), 1.0, abs_tol=1e-6):\n        raise ValueError("ratios phải khớp số tập và có tổng 1.")\n    order = sorted(group_weights)\n    random.Random(seed).shuffle(order)\n    order.sort(key=lambda g: -group_weights[g])\n    total = sum(group_weights.values())\n    filled = dict.fromkeys(names, 0)\n    out: dict[str, str] = {}\n    for group in order:\n        target = {n: r * total for n, r in zip(names, ratios, strict=True)}\n        best = max(\n            names,\n            key=lambda n: (target[n] - filled[n]) / target[n] if target[n] else -math.inf,\n        )\n        out[group] = best\n        filled[best] += group_weights[group]\n    return out\n\n\ndef cross_split_max_similarity(embeddings: Sequence[np.ndarray], split_of: Sequence[str]) -> float:\n    """Cos lớn nhất giữa hai khung thuộc hai tập khác nhau (phải < ngưỡng gom người)."""\n    best = -1.0\n    labelled = [\n        (l2_normalize(np.atleast_2d(e)), s) for e, s in zip(embeddings, split_of, strict=True)\n    ]\n    for name in sorted(set(split_of)):\n        inside = [e for e, s in labelled if s == name]\n        outside = [e for e, s in labelled if s != name]\n        if not inside or not outside:\n            continue\n        best = max(best, float((np.vstack(inside) @ np.vstack(outside).T).max()))\n    return best\n\n\ndef split_summary(rows: Iterable[dict[str, Any]]) -> dict[str, dict[str, int]]:\n    """Đếm người, nhóm, clip theo tập (để in phân bố)."""\n    persons: dict[str, set] = defaultdict(set)\n    groups: dict[str, set] = defaultdict(set)\n    clips: dict[str, int] = defaultdict(int)\n    for row in rows:\n        persons[row["split"]].add(row["person_id"])\n        groups[row["split"]].add(row["group_id"])\n        clips[row["split"]] += 1\n    return {\n        s: {"persons": len(persons[s]), "groups": len(groups[s]), "clips": clips[s]}\n        for s in SPLITS\n        if s in clips\n    }\n\n\n# ── manifest ─────────────────────────────────────────────────────────────────────────────\nMANIFEST_FIELDS = (\n    "sample_id",\n    "path",\n    "variant",\n    "split",\n    "label",\n    "dataset",\n    "method",\n    "person_id",\n    "group_id",\n    "source_clip",\n    "source_url",\n    "license",\n    "encode",\n    "sha256",\n)\n\n\ndef validate_manifest(rows: Sequence[dict[str, str]]) -> None:\n    """Kiểm manifest: trường đủ, tập/nhãn hợp lệ, người và nhóm không vắt qua hai tập, mẫu giả có\n    phương pháp giả; mẫu giả có video thật nguồn trong manifest thì nguồn phải cùng tập."""\n    person_split: dict[str, str] = {}\n    group_split: dict[str, str] = {}\n    real_split: dict[tuple[str, str], str] = {}\n    for row in rows:\n        missing = [f for f in MANIFEST_FIELDS if f not in row]\n        if missing:\n            raise ValueError(f"Manifest thiếu trường {missing}")\n        if row["split"] not in SPLITS:\n            raise ValueError(f"Tập lạ: {row[\'split\']}")\n        if row["label"] not in ("real", "fake"):\n            raise ValueError(f"Nhãn lạ: {row[\'label\']}")\n        for key, seen in (("person_id", person_split), ("group_id", group_split)):\n            if not row[key]:\n                continue\n            previous = seen.setdefault(row[key], row["split"])\n            if previous != row["split"]:\n                raise ValueError(f"{key} {row[key]} nằm ở hai tập {previous}/{row[\'split\']}.")\n        if row["label"] == "real" and row["source_clip"]:\n            real_split[(row["source_clip"], row["variant"])] = row["split"]\n    for row in rows:\n        if row["label"] != "fake":\n            continue\n        if not row["method"]:\n            raise ValueError(f"Mẫu giả thiếu phương pháp giả: {row[\'sample_id\']}")\n        source = real_split.get((row["source_clip"], row["variant"]))\n        if source is not None and source != row["split"]:\n            raise ValueError(f"Mẫu giả {row[\'sample_id\']} khác tập với video thật nguồn.")\n\n\ndef write_manifest(path: Path, rows: Sequence[dict[str, Any]]) -> None:\n    rows = sorted(rows, key=lambda r: (r["variant"], r["split"], r["label"], r["sample_id"]))\n    tmp = Path(str(path) + ".tmp")\n    with tmp.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(MANIFEST_FIELDS))\n        writer.writeheader()\n        for row in rows:\n            writer.writerow(\n                {k: ("" if row.get(k) is None else row.get(k)) for k in MANIFEST_FIELDS}\n            )\n    os.replace(tmp, path)\n\n\ndef read_manifest(path: Path) -> list[dict[str, str]]:\n    with Path(path).open(encoding="utf-8", newline="") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1 << 20), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\n# ── tiến độ (chạy tiếp khi phiên 12 giờ hết) ─────────────────────────────────────────────\n@dataclass\nclass Progress:\n    """Ghi danh sách việc đã xong/lỗi vào JSON (ghi nguyên tử) sau MỖI việc."""\n\n    path: Path\n    done: dict[str, dict[str, Any]] = field(default_factory=dict)\n    failed: dict[str, str] = field(default_factory=dict)\n\n    @classmethod\n    def load(cls, path: Path) -> Progress:\n        path = Path(path)\n        if path.is_file():\n            data = json.loads(path.read_text(encoding="utf-8"))\n            return cls(path, dict(data.get("done", {})), dict(data.get("failed", {})))\n        return cls(path)\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        tmp = Path(str(self.path) + ".tmp")\n        tmp.write_text(\n            json.dumps({"done": self.done, "failed": self.failed}, ensure_ascii=False, indent=1),\n            encoding="utf-8",\n        )\n        os.replace(tmp, self.path)\n\n    def is_done(self, key: str) -> bool:\n        return key in self.done\n\n    def mark_done(self, key: str, info: dict[str, Any] | None = None) -> None:\n        self.done[key] = info or {}\n        self.failed.pop(key, None)\n        self.save()\n\n    def mark_failed(self, key: str, reason: str) -> None:\n        self.failed[key] = reason[-500:]\n        self.save()\n\n\n# ── ffmpeg: một lệnh mã hóa lại dùng chung cho thật và giả ───────────────────────────────\nENCODE_PROFILES: dict[str, dict[str, Any]] = {\n    # bản trung gian gần như không mất mát\n    "mid": {"crf": 12, "size": CLIP_SIZE},\n    # bản phát hành: cả hai lớp cùng một lệnh\n    "c23": {"crf": 23, "size": CLIP_SIZE},\n    # bản nén mạnh cho cả hai lớp (giống mức c40 của DATA-05)\n    "c40": {"crf": 40, "size": CLIP_SIZE},\n}\n\n\ndef encode_cmd(\n    ffmpeg: str,\n    src: str | Path,\n    dst: str | Path,\n    *,\n    profile: str,\n    seconds: float = CLIP_SECONDS,\n    start: float = 0.0,\n    fps: int = TARGET_FPS,\n    metadata: dict[str, str] | None = None,\n) -> list[str]:\n    """Lệnh ffmpeg cố định: bỏ âm thanh, xóa metadata nguồn, cùng kích thước/fps/GOP/x264 preset."""\n    spec = ENCODE_PROFILES[profile]\n    size = int(spec["size"])\n    cmd = [ffmpeg, "-nostdin", "-y", "-loglevel", "error"]\n    if start > 0:\n        cmd += ["-ss", f"{start:.3f}"]\n    cmd += [\n        "-i",\n        str(src),\n        "-t",\n        f"{seconds:.3f}",\n        "-an",\n        "-sn",\n        "-dn",\n        "-map_metadata",\n        "-1",\n        "-vf",\n        f"scale={size}:{size}:flags=bicubic,fps={fps},format=yuv420p",\n        "-c:v",\n        "libx264",\n        "-preset",\n        "medium",\n        "-crf",\n        str(int(spec["crf"])),\n        "-g",\n        str(2 * fps),\n        "-bf",\n        "2",\n        "-fflags",\n        "+bitexact",\n        "-flags:v",\n        "+bitexact",\n    ]\n    for key, value in sorted((metadata or {}).items()):\n        cmd += ["-metadata", f"{key}={value}"]\n    cmd += ["-movflags", "+faststart", str(dst)]\n    return cmd\n\n\n# ── đặc trưng thô và cổng lộ đề ──────────────────────────────────────────────────────────\ndef video_features(path: Path, max_frames: int = 90) -> dict[str, float]:\n    """Đặc trưng thô KHÔNG nhìn nội dung mặt: bitrate, kích thước file, fps, thời lượng, khung\n    hình, độ nét (phương sai Laplacian) và độ sáng trung bình."""\n    import cv2\n\n    capture = cv2.VideoCapture(str(path))\n    try:\n        fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)\n        width = float(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0.0)\n        height = float(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0.0)\n        sharp: list[float] = []\n        bright: list[float] = []\n        frames = 0\n        while frames < 10_000:\n            ok, frame = capture.read()\n            if not ok:\n                break\n            if frames < max_frames:\n                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n                sharp.append(float(cv2.Laplacian(gray, cv2.CV_64F).var()))\n                bright.append(float(gray.mean()))\n            frames += 1\n    finally:\n        capture.release()\n    size = float(Path(path).stat().st_size)\n    duration = frames / fps if fps > 0 else 0.0\n    return {\n        "bitrate_kbps": size * 8 / 1000 / duration if duration > 0 else 0.0,\n        "size_bytes": size,\n        "fps": fps,\n        "duration_s": duration,\n        "width": width,\n        "height": height,\n        "sharpness": float(np.mean(sharp)) if sharp else 0.0,\n        "brightness": float(np.mean(bright)) if bright else 0.0,\n    }\n\n\ndef univariate_auc(values: Sequence[float], labels: Sequence[int]) -> float | None:\n    """AUC đơn biến (nhãn 1 = giả); None khi thiếu một lớp."""\n    from sklearn.metrics import roc_auc_score\n\n    y = np.asarray(labels)\n    if np.unique(y).size < 2:\n        return None\n    return float(roc_auc_score(y, np.asarray(values, dtype=float)))\n\n\ndef leak_auc(\n    features: Sequence[dict[str, float]],\n    labels: Sequence[int],\n    groups: Sequence[str],\n    seed: int = 0,\n    n_splits: int = 5,\n) -> float | None:\n    """AUC ngoài-fold của hồi quy logistic trên đặc trưng thô, chia fold THEO NHÓM (thật và giả\n    cùng clip nguồn luôn cùng fold). None khi thiếu lớp hoặc < 2 nhóm."""\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.metrics import roc_auc_score\n    from sklearn.model_selection import GroupKFold\n    from sklearn.pipeline import make_pipeline\n    from sklearn.preprocessing import StandardScaler\n\n    y = np.asarray(labels, dtype=int)\n    groups_array = np.asarray(groups)\n    n_groups = np.unique(groups_array).size\n    if np.unique(y).size < 2 or n_groups < 2:\n        return None\n    x = np.asarray([[float(f[name]) for name in LEAK_FEATURES] for f in features], dtype=float)\n    x = np.log1p(np.abs(x)) * np.sign(x)\n    # Đổi tên nhóm theo seed để thứ tự fold không phụ thuộc tên.\n    rng = np.random.default_rng(seed)\n    names = np.unique(groups_array)\n    mapping = dict(zip(names, rng.permutation(len(names)), strict=True))\n    coded = np.asarray([mapping[g] for g in groups_array])\n    scores = np.zeros(len(y), dtype=float)\n    folds = GroupKFold(n_splits=min(n_splits, n_groups))\n    for train_idx, test_idx in folds.split(x, y, coded):\n        if np.unique(y[train_idx]).size < 2:\n            scores[test_idx] = 0.5\n            continue\n        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))\n        model.fit(x[train_idx], y[train_idx])\n        scores[test_idx] = model.predict_proba(x[test_idx])[:, 1]\n    return float(roc_auc_score(y, scores))\n\n\n@dataclass(frozen=True)\nclass LeakVerdict:\n    ok: bool  # được phép đi tiếp\n    enforced: bool  # đủ mẫu để cổng có hiệu lực\n    auc: float | None\n    message_vi: str\n\n\ndef leak_verdict(\n    auc_value: float | None,\n    n_real: int,\n    n_fake: int,\n    auc_range: tuple[float, float] = LEAK_AUC_RANGE,\n    min_per_class: int = LEAK_MIN_PER_CLASS,\n) -> LeakVerdict:\n    """Cổng lộ đề. Đủ mẫu mà AUC ngoài khoảng → DỪNG. Ít mẫu (SMOKE) → chỉ cảnh báo."""\n    low, high = auc_range\n    enforced = min(n_real, n_fake) >= min_per_class\n    if auc_value is None:\n        return LeakVerdict(\n            not enforced,\n            enforced,\n            None,\n            "Không tính được AUC lộ đề (thiếu một lớp hoặc thiếu nhóm).",\n        )\n    inside = low <= auc_value <= high\n    if inside:\n        return LeakVerdict(\n            True, enforced, auc_value, f"Lộ đề ĐẠT: AUC {auc_value:.3f} nằm trong [{low}; {high}]."\n        )\n    if not enforced:\n        return LeakVerdict(\n            True,\n            False,\n            auc_value,\n            f"CẢNH BÁO: AUC lộ đề {auc_value:.3f} ngoài [{low}; {high}], nhưng mới có "\n            f"{n_real} thật / {n_fake} giả (< {min_per_class} mỗi lớp) nên chưa kết luận được.",\n        )\n    return LeakVerdict(\n        False,\n        True,\n        auc_value,\n        f"DỪNG: AUC lộ đề {auc_value:.3f} nằm ngoài [{low}; {high}]. Bộ phân loại chỉ nhìn "\n        "bitrate/kích thước/fps/thời lượng/độ nét/độ sáng đã tách được thật và giả, nên mô hình "\n        "học trên dữ liệu này có thể học mẹo thay vì dấu vết giả mạo. KHÔNG dùng để dạy; gửi "\n        "leak_report.json cho trưởng nhóm.",\n    )\n'),
    'vericlue_ml/finetune/gend_calib.py': ('9c77790595dfc926d19528ccf3fd66a90fe9f27b380c3d9132c5032fc1b10842', '"""FACE-04 (hiệu chỉnh) — hàm thuần cho notebook `ml/notebooks/calibrate_gend.ipynb`.\n\nGenD gốc KHÔNG được dạy lại. Notebook chỉ chấm điểm bằng đúng `face/gend.py` rồi fit calibrator\n(`vericlue_ml.common.calibration.fit`, ADR-0010) trên `calib_fit`:\n- lớp thật: người Việt thật FAS-VN live và video gốc RTFS-10k (VLR chỉ tham khảo, không fit);\n- lớp giả: video ghép mặt RTFS-10k (inswapper, uniface). Không tự tạo deepfake.\n\nỞ đây có: chia RTFS theo video YouTube gốc và ảnh donor (cắt cạnh để các tập không chung donor),\nkhung cắt vuông quanh mặt và lệnh ffmpeg mã hóa lại DÙNG CHUNG cho mọi nguồn, báo cáo lộ đề nguồn,\ntỉ lệ vượt ngưỡng, xác suất ngoài-fold theo nhóm. Không đọc tập test (`assert_no_test`).\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport math\nimport random\nimport re\nfrom collections import Counter, defaultdict\nfrom collections.abc import Iterable, Mapping, Sequence\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nfrom vericlue_ml.finetune.data04 import (\n    LEAK_FEATURES,\n    UnionFind,\n    assign_splits,\n    cluster_persons,\n    group_components,\n    univariate_auc,\n)\n\nCALIB_SPLITS = ("calib_fit", "calib_ltt", "test")\nCALIB_RATIOS = (0.4, 0.3, 0.3)\nEXPECTED_MODEL_VERSION = (\n    "gend-clip-l14@891ce014+clip@32bd6428+scrfd-10g@bfcc7d48+agg-maxmeantop25-v1"\n)\n# Ngưỡng tạm của thresholds.yaml (face.b, face.a) — chỉ để BÁO CÁO, không chọn ngưỡng ở đây.\nREPORT_THRESHOLDS = {"B": 0.15, "A": 0.60}\nCROP_SIZE = 320\nCROP_SCALE = 2.5  # cạnh khung vuông = 2,5 × cạnh lớn của hộp mặt (giống khung VLR 320×320)\nENCODE_CRF = 23\nENCODE_FPS = 25\nENCODE_SECONDS = 3.0\nSOURCE_LEAK_RANGE = (0.4, 0.6)\n# ArcFace MobileFaceNet (uniface, nghiên cứu phi thương mại) — cùng mục với manifest nhánh DATA-03.\n# Chỉ để gom người khi chia tập; embedding chỉ nằm trong thư mục nháp, xóa khi xong.\nARCFACE = {\n    "id": "arcface-mnet",\n    "path": "arcface_mnet.onnx",\n    "url": "https://huggingface.co/yakhyo/uniface-weights/resolve/"\n    "bfcc7d48ea8412d55de406cecf96b65b3ca720b2/arcface_mnet.onnx?download=true",\n    "sha256": "9cc6e4a75f0e2bf0b1aed94578f144d15175f357bdc05e815e5c4a02b319eb4f",\n}\nVN_SOURCES = ("vlr", "fas_vn")\n# Quyết định trưởng nhóm 20/10: fit calibrator CHỈ bằng FAS-VN live + RTFS (thật, giả). VLR chỉ tham khảo.\nFIT_SOURCES = ("fas_vn", "rtfs_real", "rtfs_fake")\nREFERENCE_SOURCES = ("vlr",)\n\n\n# ── RTFS-10k: chia theo video YouTube gốc + ảnh donor ────────────────────────────────────\n_RTFS_RE = re.compile(r"([A-Za-z0-9_-]{11})_\\d+_\\d+(?:_|$)")\n\n\ndef rtfs_youtube_id(filename: str) -> str | None:\n    """YouTube ID 11 ký tự trong tên file RTFS (bỏ tiền tố inswapper_/uniface_)."""\n    stem = Path(filename).stem\n    for prefix in ("inswapper_", "uniface_"):\n        if stem.startswith(prefix):\n            stem = stem[len(prefix) :]\n    match = _RTFS_RE.match(stem)\n    return match.group(1) if match else None\n\n\n@dataclass\nclass RtfsSplit:\n    real: dict[str, tuple[str, str]]  # tên video gốc → (tập, nhóm)\n    fake: dict[str, tuple[str, str]]  # tên video giả giữ lại → (tập, nhóm)\n    dropped: list[str]  # video giả bị bỏ vì donor thuộc tập khác\n    report: dict[str, Any] = field(default_factory=dict)\n\n\ndef split_rtfs(\n    videos: Sequence[Mapping[str, str]],\n    fakes: Sequence[Mapping[str, str]],\n    *,\n    seed: int,\n    ratios: Sequence[float] = CALIB_RATIOS,\n    names: Sequence[str] = CALIB_SPLITS,\n) -> RtfsSplit:\n    """Chia RTFS sao cho video gốc (YouTube ID) VÀ ảnh donor không vắt qua hai tập.\n\n    Đồ thị video–donor của RTFS liên thông thành MỘT khối (mỗi ảnh donor dùng cho nhiều video), nên\n    không chia nguyên khối được. Cách làm: chia video YouTube theo tỉ lệ; mỗi ảnh donor theo tập\n    chiếm đa số trong các video giả dùng nó (hòa → theo thứ tự `names`); video giả có donor khác tập\n    với video gốc thì BỎ. `videos`: dòng videos.csv (`filename`), `fakes`: dòng inswapper/uniface.csv\n    (`video`, `photo`, `filename`)."""\n    yt_of = {}\n    for row in videos:\n        yid = rtfs_youtube_id(row["filename"])\n        if yid is None:\n            raise ValueError(f"Không đọc được YouTube ID: {row[\'filename\']}")\n        yt_of[row["filename"]] = yid\n    for row in fakes:\n        if row["video"] not in yt_of:\n            raise ValueError(f"Video giả {row[\'filename\']} trỏ tới video gốc lạ {row[\'video\']}")\n    # Báo cáo khối liên thông trước khi cắt.\n    nodes = sorted({f"v:{y}" for y in yt_of.values()} | {f"p:{r[\'photo\']}" for r in fakes})\n    index = {n: i for i, n in enumerate(nodes)}\n    uf = UnionFind(len(nodes))\n    for row in fakes:\n        uf.union(index[f"v:{yt_of[row[\'video\']]}"], index[f"p:{row[\'photo\']}"])\n    labels = uf.labels()\n    comp_videos = Counter(labels[index[f"v:{y}"]] for y in set(yt_of.values()))\n    split_of_yt = assign_splits(\n        dict.fromkeys(sorted(set(yt_of.values())), 1), ratios, seed=seed, names=names\n    )\n    votes: dict[str, Counter] = defaultdict(Counter)\n    for row in fakes:\n        votes[row["photo"]][split_of_yt[yt_of[row["video"]]]] += 1\n    order = {n: i for i, n in enumerate(names)}\n    photo_split = {\n        p: sorted(c.items(), key=lambda kv: (-kv[1], order[kv[0]]))[0][0] for p, c in votes.items()\n    }\n    real = {f: (split_of_yt[y], f"rtfs:yt:{y}") for f, y in yt_of.items()}\n    fake: dict[str, tuple[str, str]] = {}\n    dropped: list[str] = []\n    for row in sorted(fakes, key=lambda r: r["filename"]):\n        yid = yt_of[row["video"]]\n        split = split_of_yt[yid]\n        if photo_split[row["photo"]] == split:\n            fake[row["filename"]] = (split, f"rtfs:yt:{yid}")\n        else:\n            dropped.append(row["filename"])\n    report = {\n        "youtube_videos": len(set(yt_of.values())),\n        "donor_photos": len(votes),\n        "components_before_cut": len(comp_videos),\n        "largest_component_video_share": max(comp_videos.values())\n        / max(1, len(set(yt_of.values()))),\n        "fakes_total": len(fakes),\n        "fakes_kept": len(fake),\n        "fakes_dropped": len(dropped),\n        "kept_by_split": dict(Counter(s for s, _ in fake.values())),\n        "real_by_split": dict(Counter(s for s, _ in real.values())),\n    }\n    check_rtfs_split(real, fake, fakes)\n    return RtfsSplit(real, fake, dropped, report)\n\n\ndef check_rtfs_split(\n    real: Mapping[str, tuple[str, str]],\n    fake: Mapping[str, tuple[str, str]],\n    fakes: Sequence[Mapping[str, str]],\n) -> None:\n    """Video giả cùng tập với video gốc; một ảnh donor chỉ ở một tập."""\n    photo_seen: dict[str, str] = {}\n    for row in fakes:\n        if row["filename"] not in fake:\n            continue\n        split = fake[row["filename"]][0]\n        if real[row["video"]][0] != split:\n            raise ValueError(f"Video giả {row[\'filename\']} khác tập với video gốc.")\n        if photo_seen.setdefault(row["photo"], split) != split:\n            raise ValueError(f"Ảnh donor {row[\'photo\']} nằm ở hai tập.")\n\n\n# ── nhóm người cho VLR / FAS-VN ──────────────────────────────────────────────────────────\ndef person_groups(\n    embeddings: Sequence[np.ndarray],\n    sources: Sequence[str],\n    videos: Sequence[str],\n    *,\n    threshold: float,\n) -> tuple[list[str], dict[str, Any]]:\n    """Nhóm chia tập = (cùng người theo ArcFace, gom lỏng, liên kết đơn) ∪ (cùng video nguồn).\n    Mẫu không có embedding (mảng rỗng) chỉ nối theo video. Trả (group_id, báo cáo phân bố)."""\n    if not (len(embeddings) == len(sources) == len(videos)):\n        raise ValueError("embeddings, sources, videos phải cùng độ dài.")\n    persons = cluster_persons(\n        [e if np.size(e) else np.zeros((0, 1), np.float32) for e in embeddings], threshold\n    )\n    groups = group_components(persons, list(videos))\n    sizes = Counter(groups)\n    group_ids = [f"vn:g{g}" for g in groups]\n    report = {\n        "samples": len(groups),\n        "persons": len(set(persons)),\n        "groups": len(sizes),\n        "largest_group_share": max(sizes.values()) / len(groups) if groups else 0.0,\n        "group_size_top5": [n for _, n in sizes.most_common(5)],\n        "by_source": dict(Counter(sources)),\n    }\n    return group_ids, report\n\n\ndef split_groups(\n    group_ids: Sequence[str], *, seed: int, ratios: Sequence[float] = CALIB_RATIOS\n) -> dict[str, str]:\n    """Gán nhóm (trọng số = số mẫu) vào calib_fit / calib_ltt / test."""\n    return assign_splits(dict(Counter(group_ids)), ratios, seed=seed, names=CALIB_SPLITS)\n\n\n# ── khung cắt vuông quanh mặt + mã hóa lại dùng chung ────────────────────────────────────\ndef square_crop_box(\n    boxes: Sequence[Sequence[float]], width: int, height: int, scale: float = CROP_SCALE\n) -> tuple[int, int, int]:\n    """(x, y, cạnh) chẵn của khung vuông quanh hợp các hộp mặt, nằm trọn trong ảnh."""\n    if not boxes:\n        raise ValueError("Không có hộp mặt.")\n    array = np.asarray(boxes, dtype=float)\n    x1, y1 = array[:, 0].min(), array[:, 1].min()\n    x2, y2 = array[:, 2].max(), array[:, 3].max()\n    side = max(x2 - x1, y2 - y1) * scale\n    side = int(min(side, width, height)) // 2 * 2\n    if side < 2:\n        raise ValueError("Khung cắt quá nhỏ.")\n    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2\n    x = int(round(min(max(cx - side / 2, 0), width - side))) // 2 * 2\n    y = int(round(min(max(cy - side / 2, 0), height - side))) // 2 * 2\n    return x, y, side\n\n\ndef crop_encode_cmd(\n    ffmpeg: str,\n    src: str | Path,\n    dst: str | Path,\n    box: tuple[int, int, int],\n    *,\n    still: bool = False,\n    size: int = CROP_SIZE,\n    crf: int = ENCODE_CRF,\n    fps: int = ENCODE_FPS,\n    seconds: float = ENCODE_SECONDS,\n) -> list[str]:\n    """Một lệnh ffmpeg cho MỌI nguồn (thật/giả, video/ảnh): cắt vuông → 320×320 → H.264 CRF 23,\n    25 fps, bỏ âm thanh và metadata. Ảnh tĩnh thành video 1 khung cùng cấu hình."""\n    x, y, side = box\n    vf = f"crop={side}:{side}:{x}:{y},scale={size}:{size}:flags=bicubic"\n    cmd = [str(ffmpeg), "-nostdin", "-y", "-loglevel", "error", "-i", str(src)]\n    if still:\n        cmd += ["-frames:v", "1"]\n    else:\n        cmd += ["-t", f"{seconds:.3f}"]\n        vf += f",fps={fps}"\n    vf += ",format=yuv420p"\n    cmd += [\n        "-an",\n        "-sn",\n        "-dn",\n        "-map_metadata",\n        "-1",\n        "-vf",\n        vf,\n        "-c:v",\n        "libx264",\n        "-preset",\n        "medium",\n        "-crf",\n        str(int(crf)),\n        "-g",\n        str(2 * fps),\n        "-bf",\n        "2",\n        "-r",\n        str(fps),\n        "-fflags",\n        "+bitexact",\n        "-flags:v",\n        "+bitexact",\n        "-movflags",\n        "+faststart",\n        str(dst),\n    ]\n    return cmd\n\n\n# ── FAS-VN: chỉ ảnh live, bỏ trùng byte và ảnh trùng với not_live ────────────────────────\ndef sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef fas_live_files(root: Path) -> tuple[list[Path], dict[str, int]]:\n    """Ảnh `**/live/*` của FAS-VN, bỏ bản trùng byte và ảnh có byte trùng một ảnh `not_live`."""\n    root = Path(root)\n    suffixes = {".jpg", ".jpeg", ".png"}\n    live = sorted(\n        p for p in root.rglob("*") if p.suffix.lower() in suffixes and p.parent.name == "live"\n    )\n    spoof = [p for p in root.rglob("*") if p.suffix.lower() in suffixes and p.parent.name != "live"]\n    spoof_hashes = {sha256_bytes(p.read_bytes()) for p in spoof}\n    seen: set[str] = set()\n    keep: list[Path] = []\n    counts = Counter()\n    for path in live:\n        digest = sha256_bytes(path.read_bytes())\n        if digest in spoof_hashes:\n            counts["live_equal_to_spoof"] += 1\n            continue\n        if digest in seen:\n            counts["duplicate"] += 1\n            continue\n        seen.add(digest)\n        keep.append(path)\n    counts["live_kept"] = len(keep)\n    counts["spoof_files_scanned"] = len(spoof)\n    return keep, dict(counts)\n\n\n# ── lộ đề nguồn ──────────────────────────────────────────────────────────────────────────\ndef source_leak_report(\n    features: Sequence[Mapping[str, float]],\n    labels: Sequence[int],\n    auc_range: tuple[float, float] = SOURCE_LEAK_RANGE,\n    feature_names: Sequence[str] = LEAK_FEATURES,\n    pair_vi: str = "thật (VLR/FAS-VN) và giả (RTFS)",\n) -> dict[str, Any]:\n    """AUC đơn biến từng đặc trưng thô (nhãn 1 = giả) giữa hai nhóm `pair_vi`. Ngoài khoảng →\n    CẢNH BÁO (không dừng): calibrator chỉ là hàm 1 chiều của điểm GenD, nhưng nếu thật và giả khác\n    nguồn thì calibrator có thể học \'nguồn\' chứ không phải \'giả\'."""\n    low, high = auc_range\n    aucs = {}\n    for name in feature_names:\n        values = [float(f[name]) for f in features]\n        if np.ptp(values) == 0:\n            aucs[name] = 0.5\n            continue\n        aucs[name] = univariate_auc(values, labels)\n    flagged = {k: v for k, v in aucs.items() if v is not None and not low <= v <= high}\n    n_pos = int(np.sum(np.asarray(labels) == 1))\n    counts = f"{len(labels) - n_pos} thật / {n_pos} giả"\n    if flagged:\n        worst = max(flagged, key=lambda k: abs(flagged[k] - 0.5))\n        message = (\n            f"CẢNH BÁO LỘ ĐỀ NGUỒN ({counts}): {len(flagged)} đặc trưng thô tách được {pair_vi} "\n            f"(mạnh nhất: {worst} AUC {flagged[worst]:.3f}, ngoài [{low}; {high}]). Calibrator có "\n            "thể học \'nguồn\' thay vì \'giả\'. Đã ghi vào card.md."\n        )\n    else:\n        message = f"Lộ đề nguồn ({counts}): mọi đặc trưng thô có AUC trong [{low}; {high}]."\n    return {"auc": aucs, "flagged": flagged, "warning": bool(flagged), "message_vi": message}\n\n\n# ── báo cáo điểm ─────────────────────────────────────────────────────────────────────────\ndef exceed_rates(\n    probabilities: Sequence[float], thresholds: Mapping[str, float] = REPORT_THRESHOLDS\n) -> dict[str, float | None]:\n    """Tỉ lệ mẫu có xác suất ≥ từng ngưỡng (None khi rỗng)."""\n    p = np.asarray(probabilities, dtype=float)\n    return {k: (float(np.mean(p >= t)) if p.size else None) for k, t in thresholds.items()}\n\n\ndef score_summary(rows: Iterable[Mapping[str, Any]], key: str = "score") -> dict[str, dict]:\n    """Phân bố điểm theo `source`/`label`: n, trung bình, p10/p50/p90."""\n    buckets: dict[str, list[float]] = defaultdict(list)\n    for row in rows:\n        buckets[f"{row[\'source\']}|{\'gia\' if int(row[\'label\']) else \'that\'}"].append(float(row[key]))\n    out = {}\n    for name, values in sorted(buckets.items()):\n        v = np.asarray(values)\n        out[name] = {\n            "n": int(v.size),\n            "mean": float(v.mean()),\n            "p10": float(np.percentile(v, 10)),\n            "p50": float(np.percentile(v, 50)),\n            "p90": float(np.percentile(v, 90)),\n        }\n    return out\n\n\ndef oof_probabilities(calibrator, scores: Sequence[float], labels, groups) -> np.ndarray:\n    """Xác suất ngoài-fold theo nhóm (GroupKFold như lúc fit, cùng phương pháp đã chọn) — để báo tỉ\n    lệ vượt ngưỡng \'sau hiệu chỉnh\' trên calib_fit mà không dùng chính mẫu đã fit."""\n    from sklearn.model_selection import GroupKFold\n\n    from vericlue_ml.common.calibration import _fit_parameters, _predict\n\n    x = np.asarray(scores, dtype=float)\n    y = np.asarray(labels, dtype=float)\n    g = np.asarray(groups)\n    out = np.empty_like(x)\n    for train, valid in GroupKFold(n_splits=calibrator.cv_folds).split(x, y, g):\n        params = _fit_parameters(calibrator.method, x[train], y[train])\n        out[valid] = _predict(calibrator.method, params, x[valid], calibrator.epsilon)\n    return out\n\n\ndef subset(rows: Sequence[Mapping[str, Any]], **where: Any) -> list[Mapping[str, Any]]:\n    """Lọc dòng theo trường = giá trị (giá trị có thể là tuple các lựa chọn)."""\n    out = []\n    for row in rows:\n        if all((row[k] in v) if isinstance(v, tuple) else row[k] == v for k, v in where.items()):\n            out.append(row)\n    return out\n\n\ndef sample_rows(items: Sequence[Any], n: int | None, seed: int) -> list[Any]:\n    """Lấy ngẫu nhiên xác định ≤ n phần tử (None = tất cả), giữ thứ tự gốc."""\n    if n is None or n >= len(items):\n        return list(items)\n    chosen = sorted(random.Random(seed).sample(range(len(items)), n))\n    return [items[i] for i in chosen]\n\n\ndef eta_text(done: int, total: int, seconds: float) -> str:\n    """\'đã x/y, còn khoảng z phút\' cho ô in tiến độ."""\n    if done <= 0:\n        return f"0/{total}"\n    left = (total - done) * seconds / done\n    return f"{done}/{total}, {seconds / 60:.1f} phút, còn khoảng {math.ceil(left / 60)} phút"\n'),
    'vericlue_ml/finetune/gend_train.py': ('1d8af6f15539d2ab9f12798c62d658728fccbf5f710b9b0ed3548fa2184d627c', '"""FACE-04 — hàm thuần cho tinh chỉnh GenD (chỉ LayerNorm + đầu tuyến tính, như bài báo GenD).\n\nNotebook `ml/notebooks/finetune_gend.ipynb` nhúng nguyên văn file này cùng `face/gend.py`, `face/pad.py`,\n`face/frames.py`, nên khung, dò mặt, căn mặt và chuẩn hóa GIỐNG HỆT hệ thống đang chạy.\n\nNguồn (chép lại, giữ giấy phép — xem `third_party/README.md`):\n- yermandy/GenD (MIT, Copyright (c) 2025 Andy), commit 387a42266dd385fbe8f7626c5d3aa03eea3bfaab:\n  `src/exp/wacv_rebuttal.py` (lớp được mở: `pre_layrnorm`, `layer_norm1`, `layer_norm2`,\n  `post_layernorm`; loss CE 1,0 + uniformity 0,5 + alignment 0,1), `src/losses/unifalign.py`\n  (`alignment`, `uniformity`), `src/config.py` (AdamW, cosine, weight decay 0).\n\nLuật: tập test KHÔNG được đọc khi dạy hay chọn checkpoint (`rows_for`, `assert_no_test`).\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable, Sequence\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\n# Tên lớp được mở khóa (đúng `unfreeze_layers` của GenD cho CLIP).\nTRAINABLE_LAYER_NAMES = ("pre_layrnorm", "layer_norm1", "layer_norm2", "post_layernorm")\nGEND_VISION_PREFIX = "feature_extractor.vision_model."\nHF_VISION_PREFIX = "vision_model."\nHEAD_WEIGHT_KEY = "model.linear.weight"\nHEAD_BIAS_KEY = "model.linear.bias"\nLOSS_WEIGHTS = {"ce": 1.0, "uniformity": 0.5, "alignment": 0.1}\nTEST_CONFIRMATION = "TOI-LA-TRUONG-NHOM-CHAY-TEST-MOT-LAN"\n\n\n# ── chặn tập test ────────────────────────────────────────────────────────────────────────\nclass TestSplitAccessError(RuntimeError):\n    """Có mã định đọc tập test khi dạy hoặc chọn checkpoint."""\n\n\ndef assert_no_test(rows: Iterable[dict[str, Any]]) -> None:\n    """Ném lỗi nếu có dòng nào thuộc tập test (theo cột `split` hoặc đường dẫn `/test/`)."""\n    for row in rows:\n        path = str(row.get("path", "")).replace("\\\\", "/")\n        if row.get("split") == "test" or "/test/" in f"/{path}":\n            raise TestSplitAccessError(\n                f"Không được dùng tập test khi dạy/chọn mô hình: {row.get(\'sample_id\') or path}"\n            )\n\n\ndef rows_for(rows: Iterable[dict[str, Any]], splits: Sequence[str]) -> list[dict[str, Any]]:\n    """Lọc manifest theo tập; xin tập test ở đây là lỗi (test chỉ đọc qua hàm test riêng)."""\n    if "test" in splits:\n        raise TestSplitAccessError("rows_for không bao giờ trả tập test.")\n    chosen = [r for r in rows if r.get("split") in splits]\n    assert_no_test(chosen)\n    return chosen\n\n\ndef check_test_unlocked(run_test: bool, confirmation: str) -> None:\n    """Hàm test chỉ chạy khi trưởng nhóm bật cờ VÀ gõ đúng câu xác nhận."""\n    if not run_test or confirmation != TEST_CONFIRMATION:\n        raise TestSplitAccessError(\n            "Hàm test đang TẮT. Chỉ trưởng nhóm bật RUN_TEST=True và gõ đúng câu xác nhận, "\n            "một lần duy nhất sau khi đã chốt mô hình."\n        )\n\n\n# ── tham số được dạy và xuất trọng số ────────────────────────────────────────────────────\ndef is_trainable(name: str) -> bool:\n    """Tham số của CLIPVisionModel có thuộc LayerNorm được mở không."""\n    return any(part in TRAINABLE_LAYER_NAMES for part in name.split("."))\n\n\ndef to_gend_key(name: str) -> str:\n    """`vision_model.x` (HF CLIPVisionModel) → `feature_extractor.vision_model.x` (checkpoint GenD)."""\n    if not name.startswith(HF_VISION_PREFIX):\n        raise ValueError(f"Tên tham số lạ: {name}")\n    return GEND_VISION_PREFIX + name[len(HF_VISION_PREFIX) :]\n\n\ndef delta_state(vision_state: dict[str, Any], head_weight, head_bias) -> dict[str, Any]:\n    """Chỉ phần được dạy (LayerNorm + đầu), đặt tên đúng như checkpoint GenD để ghép lại được."""\n    out = {to_gend_key(k): v for k, v in vision_state.items() if is_trainable(k)}\n    if not out:\n        raise ValueError("Không tìm thấy tham số LayerNorm nào.")\n    out[HEAD_WEIGHT_KEY] = head_weight\n    out[HEAD_BIAS_KEY] = head_bias\n    return out\n\n\ndef merge_delta(base: dict[str, Any], delta: dict[str, Any]) -> dict[str, Any]:\n    """Ghép delta vào checkpoint GenD gốc → state đầy đủ mà `face/gend.py` nạp được nguyên trạng.\n    Chỉ cho thay LayerNorm và đầu tuyến tính; khóa/kích thước phải khớp."""\n    merged = dict(base)\n    for key, value in delta.items():\n        allowed = key in (HEAD_WEIGHT_KEY, HEAD_BIAS_KEY) or (\n            key.startswith(GEND_VISION_PREFIX) and is_trainable(key[len("feature_extractor.") :])\n        )\n        if not allowed:\n            raise ValueError(f"Delta chứa tham số không được phép thay: {key}")\n        if key not in base:\n            raise ValueError(f"Checkpoint gốc không có {key}")\n        if tuple(base[key].shape) != tuple(value.shape):\n            raise ValueError(f"Kích thước {key} không khớp.")\n        merged[key] = value\n    return merged\n\n\n# ── tăng cường (cho CẢ HAI lớp) ──────────────────────────────────────────────────────────\n@dataclass(frozen=True)\nclass AugConfig:\n    p_flip: float = 0.5\n    p_downscale: float = 0.3\n    downscale_side: tuple[int, int] = (64, 224)\n    p_blur: float = 0.2\n    blur_sigma: tuple[float, float] = (0.1, 2.0)\n    p_jpeg: float = 0.5\n    jpeg_quality: tuple[int, int] = (30, 95)\n    crf_range: tuple[int, int] = (23, 40)\n    crf_side: tuple[int, int] = (160, 320)\n\n\ndef augment_crop(crop_bgr: np.ndarray, rng: np.random.Generator, cfg: AugConfig) -> np.ndarray:\n    """Tăng cường trên mặt đã căn (trước `to_tensor`): lật, thu nhỏ rồi để CLIP phóng lại, mờ, JPEG."""\n    import cv2\n\n    image = np.ascontiguousarray(crop_bgr)\n    if rng.random() < cfg.p_flip:\n        image = np.ascontiguousarray(image[:, ::-1])\n    if rng.random() < cfg.p_downscale:\n        side = int(rng.integers(cfg.downscale_side[0], cfg.downscale_side[1] + 1))\n        if side < max(image.shape[:2]):\n            image = cv2.resize(image, (side, side), interpolation=cv2.INTER_AREA)\n    if rng.random() < cfg.p_blur:\n        sigma = float(rng.uniform(*cfg.blur_sigma))\n        image = cv2.GaussianBlur(image, (0, 0), sigmaX=sigma)\n    if rng.random() < cfg.p_jpeg:\n        quality = int(rng.integers(cfg.jpeg_quality[0], cfg.jpeg_quality[1] + 1))\n        ok, encoded = cv2.imencode(".jpg", image, [cv2.IMWRITE_JPEG_QUALITY, quality])\n        if ok:\n            image = cv2.imdecode(encoded, cv2.IMREAD_COLOR)\n    return image\n\n\ndef crf_variant_cmd(\n    ffmpeg: str, src: str | Path, dst: str | Path, *, crf: int, side: int, fps: int = 25\n) -> list[str]:\n    """Tăng cường mức video: mã hóa lại H.264 với CRF và cỡ khung cho trước (cả hai lớp)."""\n    return [\n        ffmpeg,\n        "-nostdin",\n        "-y",\n        "-loglevel",\n        "error",\n        "-i",\n        str(src),\n        "-an",\n        "-map_metadata",\n        "-1",\n        "-vf",\n        f"scale={side}:{side}:flags=bicubic,fps={fps},format=yuv420p",\n        "-c:v",\n        "libx264",\n        "-preset",\n        "veryfast",\n        "-crf",\n        str(int(crf)),\n        str(dst),\n    ]\n\n\ndef pad_border(image: np.ndarray, ratio: float = 0.5, value: int = 128) -> np.ndarray:\n    """Thêm viền xám mỗi phía (ảnh đã cắt sát mặt thì SCRFD cần viền mới thấy mặt)."""\n    import cv2\n\n    height, width = image.shape[:2]\n    top, side = int(round(height * ratio)), int(round(width * ratio))\n    return cv2.copyMakeBorder(\n        image, top, top, side, side, cv2.BORDER_CONSTANT, value=(value, value, value)\n    )\n\n\n# ── lấy mặt đúng như face/gend.py ────────────────────────────────────────────────────────\ndef aligned_faces(\n    detector,\n    *,\n    video: Path | None = None,\n    images: Sequence[Path] | None = None,\n    count: int | None = None,\n) -> tuple[int, list[tuple[int, np.ndarray]]]:\n    """(số khung yêu cầu, [(chỉ số khung, mặt BGR đã căn)]) theo đúng `FaceDeepfakeDetector`:\n    `sample_video`/`sample_images` → mặt lớn nhất (SCRFD, chia 128, đảo RGB) → `align_face` (1,3)."""\n    from vericlue_ml.face.frames import sample_images, sample_video\n    from vericlue_ml.face.gend import EKYC_FRAMES, GendError, align_face\n\n    count = count or EKYC_FRAMES\n    if video is not None:\n        requested, frames = sample_video(Path(video), count)\n    elif images:\n        requested, frames = sample_images(list(images), count)\n    else:\n        raise ValueError("Cần video hoặc ảnh.")\n    out: list[tuple[int, np.ndarray]] = []\n    for index, frame in frames:\n        face = detector._largest_face(frame)\n        if face is None:\n            continue\n        try:\n            out.append((index, align_face(frame, np.array(face.landmarks))))\n        except GendError:\n            continue\n    return requested, out\n\n\n# ── điểm và chỉ số theo clip ─────────────────────────────────────────────────────────────\ndef clip_score(p_fake: Sequence[float]) -> float:\n    """Điểm clip đúng như `face/gend.py`: max(mean, top25_mean)."""\n    from vericlue_ml.face.gend import video_scores\n\n    mean, top = video_scores(list(p_fake))\n    return max(mean, top)\n\n\ndef clip_metrics(labels: Sequence[int], scores: Sequence[float]) -> dict[str, Any]:\n    """AUC và EER theo clip (nhãn 1 = giả) bằng `vericlue_ml.eval.metrics`."""\n    from vericlue_ml.eval.metrics import auc, eer\n\n    y = np.asarray(labels, dtype=int)\n    return {\n        "auc": auc(y, scores),\n        "eer": eer(y, scores),\n        "n_real": int((y == 0).sum()),\n        "n_fake": int((y == 1).sum()),\n    }\n\n\ndef threshold_at_eer(labels: Sequence[int], scores: Sequence[float]) -> float | None:\n    """Ngưỡng tại điểm EER trên calib (chỉ để đo chặn nhầm FAS-VN, KHÔNG phải ngưỡng sản phẩm)."""\n    from sklearn.metrics import roc_curve\n\n    y = np.asarray(labels, dtype=int)\n    if np.unique(y).size < 2:\n        return None\n    fpr, tpr, thresholds = roc_curve(y, np.asarray(scores, dtype=float))\n    index = int(np.nanargmin(np.abs((1 - tpr) - fpr)))\n    return float(min(thresholds[index], 1.0))\n\n\ndef false_block_rate(live_scores: Sequence[float], threshold: float | None) -> float | None:\n    """Tỉ lệ ảnh người thật (FAS-VN live) bị coi là giả ở ngưỡng cho trước."""\n    if threshold is None or len(live_scores) == 0:\n        return None\n    return float(np.mean(np.asarray(live_scores, dtype=float) >= threshold))\n\n\ndef select_checkpoint(history: Sequence[dict[str, Any]], max_fbr_increase: float = 0.05) -> dict:\n    """Chọn checkpoint CHỈ trên calib: AUC calib cao nhất trong các checkpoint không làm tỉ lệ chặn\n    nhầm FAS-VN live tăng quá `max_fbr_increase` so với GenD gốc (epoch 0). Hòa → EER thấp hơn\n    → epoch sớm hơn. GenD gốc cũng là ứng viên (tinh chỉnh không tốt hơn thì giữ gốc)."""\n    if not history:\n        raise ValueError("Chưa có checkpoint nào.")\n    base = next((h for h in history if h["epoch"] == 0), history[0])\n    base_fbr = base.get("fasvn_fbr")\n\n    def allowed(entry: dict) -> bool:\n        fbr = entry.get("fasvn_fbr")\n        return base_fbr is None or fbr is None or fbr <= base_fbr + max_fbr_increase\n\n    pool = [h for h in history if allowed(h) and h.get("calib_auc") is not None] or [\n        h for h in history if h.get("calib_auc") is not None\n    ]\n    if not pool:\n        return base\n    return sorted(\n        pool,\n        key=lambda h: (-h["calib_auc"], h.get("calib_eer") or 1.0, h["epoch"]),\n    )[0]\n\n\n# ── loss của GenD (torch, nạp lười) ──────────────────────────────────────────────────────\ndef alignment_loss(embeddings, labels, alpha: float = 2.0):\n    """Alignment theo nhãn trên embedding đã chuẩn hóa L2 (GenD `unifalign.alignment`)."""\n    import torch\n\n    if embeddings.size(0) < 2:\n        return torch.tensor(0.0, device=embeddings.device)\n    mask = (labels[:, None] == labels[None, :]).triu(diagonal=1)\n    pairs = torch.nonzero(mask, as_tuple=False)\n    if pairs.numel() == 0:\n        return torch.tensor(0.0, device=embeddings.device)\n    x, y = embeddings[pairs[:, 0]], embeddings[pairs[:, 1]]\n    return (x - y).norm(p=2, dim=1).pow(alpha).mean()\n\n\ndef uniformity_loss(embeddings, t: float = 2.0, clip_value: float = 1e-6):\n    """Uniformity (GenD `unifalign.uniformity`)."""\n    import torch\n\n    if embeddings.size(0) < 2:\n        return torch.tensor(0.0, device=embeddings.device)\n    return torch.pdist(embeddings, p=2).pow(2).mul(-t).exp().mean().clamp(min=clip_value).log()\n'),
}
_SRC = _Path(_os.environ.get("VC_SRC", "vc_src")).resolve()
for _name, (_digest, _text) in _EMBEDDED.items():
    if _hashlib.sha256(_text.encode("utf-8")).hexdigest() != _digest:
        raise RuntimeError(f"Code nhúng bị sửa: {_name}")
    _path = _SRC / _name
    _path.parent.mkdir(parents=True, exist_ok=True)
    _path.write_text(_text, encoding="utf-8")
if str(_SRC) not in _sys.path:
    _sys.path.insert(0, str(_SRC))
_os.environ["ML_MODELS_MANIFEST"] = str(_SRC / "models" / "manifest.yaml")
print(f"Code nhúng: {len(_EMBEDDED)} file từ commit {SOURCE_COMMIT[:12]}{' (có sửa chưa commit)' if SOURCE_DIRTY else ''} → {_SRC}")

In [ ]:
# ===================== 0. Thông tin môi trường (gửi kèm khi báo lỗi) =====================
import json as _json

import nb_common as nb

ENV_INFO = nb.environment_report()
print(_json.dumps(ENV_INFO, ensure_ascii=False, indent=1))

In [ ]:
# ===================== 1. Môi trường =====================
import csv
import json
import os
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path

import nb_common as nb

T0 = time.time()
PLATFORM = nb.detect_platform()
if PLATFORM == "colab":
    nb.mount_colab_drive()
OUT, WORK = nb.default_dirs(PLATFORM, "calibrate_gend" + ("_smoke" if SMOKE else ""))
MODELS = WORK / "models"
for module, package in (
    ("onnxruntime", "onnxruntime-gpu" if DEVICE == "cuda" else "onnxruntime"),
    ("safetensors", None),
    ("transformers", None),
    ("cv2", "opencv-python-headless"),
    ("yaml", "pyyaml"),
    ("pydantic", None),
    ("sklearn", "scikit-learn"),
):
    nb.ensure_import(module, package)
FFMPEG, _ = nb.ensure_ffmpeg(WORK)
os.environ["MODELS_DIR"] = str(MODELS)
os.environ["DEVICE"] = DEVICE
GPU = nb.gpu_info()
print(f"Nền tảng: {PLATFORM} | GPU: {GPU['name']} ({GPU['total_mb']} MB) | SMOKE={SMOKE}")
print(f"Kết quả giữ lại: {OUT}\nNháp: {WORK} | ffmpeg: {nb.ffmpeg_version(FFMPEG)}")
STEP_TIMES: dict[str, float] = {}


def step_done(name: str, started: float) -> None:
    STEP_TIMES[name] = round(time.time() - started, 1)
    nb.write_json(OUT / "step_times.json", STEP_TIMES)
    print(f"✔ {name}: {STEP_TIMES[name] / 60:.1f} phút")

In [ ]:
# ===================== 2. Trọng số (đúng revision, kiểm SHA-256) =====================
import yaml

from vericlue_ml.face.gend import FaceDeepfakeDetector
from vericlue_ml.finetune import data04 as d4
from vericlue_ml.finetune import gend_calib as gc
from vericlue_ml.finetune import gend_train as gt

started = time.time()
MANIFEST = yaml.safe_load(Path(os.environ["ML_MODELS_MANIFEST"]).read_text(encoding="utf-8"))
nb.fetch_manifest_model(MANIFEST, "gend-clip-l14", MODELS)
nb.fetch_manifest_model(
    MANIFEST, "clip-vit-large-patch14", MODELS, only=["config.json", "preprocessor_config.json"]
)
nb.fetch_manifest_model(MANIFEST, "scrfd-10g", MODELS)
nb.download(gc.ARCFACE["url"], MODELS / gc.ARCFACE["id"] / gc.ARCFACE["path"], gc.ARCFACE["sha256"])
DETECTOR = FaceDeepfakeDetector(MODELS)
DETECTOR.preload(device=DEVICE)  # kiểm SHA-256 lần nữa và dựng GenD như hệ thống thật
MODEL_VERSION = DETECTOR.version
assert MODEL_VERSION == gc.EXPECTED_MODEL_VERSION, f"Sai phiên bản mô hình: {MODEL_VERSION}"
import onnxruntime as ort

from vericlue_ml.common.device import onnx_providers

ARCFACE_SESSION = ort.InferenceSession(
    str(MODELS / gc.ARCFACE["id"] / gc.ARCFACE["path"]),
    providers=onnx_providers(DEVICE, component="arcface"),
)
print("Mô hình:", MODEL_VERSION)
step_done("2_trong_so", started)

In [ ]:
# ===================== 3. Tìm dữ liệu (Kaggle: Add Input; Colab: tải bằng Kaggle API) =====================
import requests


def kaggle_auth() -> dict:
    """Header xác thực Kaggle API (Colab: Secrets KAGGLE_USERNAME / KAGGLE_KEY)."""
    user, key = os.environ.get("KAGGLE_USERNAME"), os.environ.get("KAGGLE_KEY")
    if PLATFORM == "colab" and not key:
        from google.colab import userdata  # type: ignore[import-not-found]

        user, key = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
    if not key:
        raise RuntimeError(
            "Không thấy dữ liệu Add Input và không có KAGGLE_KEY. Kaggle: bấm Add Input (README); "
            "Colab: thêm KAGGLE_USERNAME và KAGGLE_KEY vào Secrets (biểu tượng chìa khóa)."
        )
    if key.startswith("KGAT_"):
        return {"headers": {"Authorization": f"Bearer {key}"}}
    return {"auth": (user, key)}


def kaggle_dataset_dir(repo: str, env: str) -> Path:
    """Thư mục gốc của bộ dữ liệu: biến môi trường → /kaggle/input (Add Input) → tải ZIP (Colab)."""
    if os.environ.get(env):
        return Path(os.environ[env])
    owner, slug = repo.split("/")
    base = Path("/kaggle/input")
    candidates = [base / slug, base / "datasets" / owner / slug]
    candidates += sorted(p for p in base.glob(f"*/{slug}") if p.is_dir()) if base.is_dir() else []
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    target = WORK / "datasets" / repo.replace("/", "__")
    if (target / ".done").is_file():
        return target
    archive = WORK / "datasets" / (repo.replace("/", "__") + ".zip")
    if not archive.is_file():
        print(f"Tải {repo} bằng Kaggle API (có thể mất 10–30 phút)…")
        with requests.get(
            f"https://www.kaggle.com/api/v1/datasets/download/{repo}",
            stream=True,
            timeout=600,
            **kaggle_auth(),
        ) as response:
            response.raise_for_status()
            part = archive.with_suffix(".part")
            part.parent.mkdir(parents=True, exist_ok=True)
            with part.open("wb") as handle:
                for chunk in response.iter_content(1 << 22):
                    handle.write(chunk)
            part.replace(archive)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(target)
    archive.unlink()
    (target / ".done").write_text("ok")
    return target


started = time.time()
RTFS_ROOT = kaggle_dataset_dir("stplusplus/rtfs-10k", "VC_RTFS_ROOT")
_csv = sorted(RTFS_ROOT.rglob("inswapper.csv"))
RTFS_ROOT = _csv[0].parent if _csv else RTFS_ROOT
FAS_ROOT = kaggle_dataset_dir("maihongtng/face-anti-spoofing", "VC_FAS_ROOT")
print("RTFS-10k:", RTFS_ROOT)
print("FAS-VN  :", FAS_ROOT)
for name in ("videos.csv", "inswapper.csv", "uniface.csv"):
    assert (RTFS_ROOT / name).is_file(), (
        f"Thiếu {name} trong {RTFS_ROOT} — kiểm lại Add Input RTFS-10k."
    )
step_done("3_tim_du_lieu", started)

In [ ]:
# ===================== 4. Lập danh sách mẫu (chưa xem nội dung) =====================
started = time.time()


def read_csv(path: Path) -> list[dict]:
    with open(path, encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


SAMPLES: dict[str, dict] = {}

# RTFS: chia theo video gốc + ảnh donor TRƯỚC, rồi mới lấy mẫu trong từng tập.
rt_videos = read_csv(RTFS_ROOT / "videos.csv")
rt_fakes = [
    r
    for r in read_csv(RTFS_ROOT / "inswapper.csv") + read_csv(RTFS_ROOT / "uniface.csv")
    if r.get("success", "True") == "True"
]
RTFS = gc.split_rtfs(rt_videos, rt_fakes, seed=SEED)
print("RTFS — báo cáo chia:", json.dumps(RTFS.report, ensure_ascii=False))
gen_of = {r["filename"]: r["generator"] for r in rt_fakes}
# Chỉ lấy file có thật trên đĩa (Add Input đầy đủ thì có hết; thiếu thì báo số lượng).
real_ok = [n for n in sorted(RTFS.real) if (RTFS_ROOT / "original_videos" / n).is_file()]
fake_ok = [n for n in sorted(RTFS.fake) if (RTFS_ROOT / gen_of[n] / n).is_file()]
print(
    f"RTFS file có trên đĩa: {len(real_ok)}/{len(RTFS.real)} thật, {len(fake_ok)}/{len(RTFS.fake)} giả"
)
real_names = gc.sample_rows(real_ok, N_RTFS_REAL, SEED)
fake_names = gc.sample_rows(fake_ok, N_RTFS_FAKE, SEED + 1)
for name in real_names:
    split, group = RTFS.real[name]
    SAMPLES[f"rtfs_real__{Path(name).stem}"] = dict(
        source="rtfs_real",
        label=0,
        split=split,
        group_id=group,
        method="",
        raw=str(RTFS_ROOT / "original_videos" / name),
        still=False,
        video_key=group,
    )
for name in fake_names:
    split, group = RTFS.fake[name]
    gen = gen_of[name]
    SAMPLES[f"rtfs_fake__{Path(name).stem}"] = dict(
        source="rtfs_fake",
        label=1,
        split=split,
        group_id=group,
        method=gen,
        raw=str(RTFS_ROOT / gen / name),
        still=False,
        video_key=group,
    )

# FAS-VN live: bỏ trùng byte và ảnh trùng not_live.
fas_files, fas_counts = gc.fas_live_files(FAS_ROOT)
print(f"FAS-VN live: {fas_counts} ({time.time() - started:.0f} s)")
for path in gc.sample_rows(fas_files, N_FAS_LIVE, SEED + 2):
    SAMPLES[f"fas_vn__{path.stem}"] = dict(
        source="fas_vn",
        label=0,
        split="",
        group_id="",
        method="",
        raw=str(path),
        still=True,
        video_key=f"fas:{path.stem}",
    )

# VLR: mục lục ZIP từ xa (chỉ vài MB), chọn clip xen kẽ playlist/video.
vlr_index: dict[str, tuple[str, d4.ZipMember]] = {}
for part in VLR_PARTS if USE_VLR else []:  # VLR tắt mặc định
    url = d4.VLR_URL.format(record=d4.VLR_RECORD, part=part)
    cache = WORK / "vlr_index" / f"{part}.json"  # mục lục ZIP đã đọc (chạy lại thì khỏi tải)
    if cache.is_file():
        members = [d4.ZipMember(**m) for m in json.loads(cache.read_text(encoding="utf-8"))]
    else:
        members = d4.list_remote_zip(url)
        nb.write_json(cache, [m.__dict__ for m in members])
    by_name = {m.name: m for m in members}
    clips = [c for m in members if (c := d4.parse_vlr_member(m.name, m.file_size))]
    chosen = d4.plan_candidates(clips, VLR_VIDEOS_PER_PART, VLR_CLIPS_PER_VIDEO, SEED)
    for clip in chosen:
        vlr_index[clip.clip_id] = (url, by_name[clip.member])
        SAMPLES[f"vlr__{clip.clip_id}"] = dict(
            source="vlr",
            label=0,
            split="",
            group_id="",
            method="",
            raw=str(WORK / "raw" / "vlr" / f"{clip.clip_id}.mp4"),
            still=False,
            video_key=f"vlr:{clip.part}/{clip.playlist}/{clip.video}",
        )
    print(
        f"VLR {part}: {len(clips)} clip trong ZIP, chọn {len(chosen)} ({time.time() - started:.0f} s)"
    )
print("Mẫu theo nguồn:", dict(Counter(s["source"] for s in SAMPLES.values())))
step_done("4_lap_danh_sach", started)

In [ ]:
# ===================== 5. Tải clip VLR (HTTP Range) + dò mặt + ArcFace =====================
import cv2
import numpy as np

started = time.time()
(WORK / "raw" / "vlr").mkdir(parents=True, exist_ok=True)
import threading
from concurrent.futures import ThreadPoolExecutor

_local = threading.local()


def fetch_vlr(item: tuple[str, tuple[str, d4.ZipMember]]) -> None:
    """Tải một clip VLR (HTTP Range); mỗi luồng một session."""
    clip_id, (url, member) = item
    dst = WORK / "raw" / "vlr" / f"{clip_id}.mp4"
    if dst.is_file():
        return
    if not hasattr(_local, "session"):
        _local.session = requests.Session()
    tmp = dst.with_name(dst.name + ".part")
    tmp.write_bytes(d4.fetch_member(url, member, _local.session))
    tmp.replace(dst)


with ThreadPoolExecutor(8) as pool:
    for i, _ in enumerate(pool.map(fetch_vlr, vlr_index.items()), 1):
        if i % 100 == 0 or i == len(vlr_index):
            print("Tải VLR:", gc.eta_text(i, len(vlr_index), time.time() - started))
print(f"VLR đã có {len(vlr_index)} clip ({time.time() - started:.0f} s)")


def probe(path: Path, still: bool) -> tuple[list[np.ndarray], int, int]:
    """3 khung trong 3 giây đầu (ảnh tĩnh: 1 khung) để dò mặt và cắt khung."""
    if still:
        image = cv2.imread(str(path))
        if image is None:
            return [], 0, 0
        return [image], image.shape[1], image.shape[0]
    capture = cv2.VideoCapture(str(path))
    fps = capture.get(cv2.CAP_PROP_FPS) or 25.0
    width, height = int(capture.get(3)), int(capture.get(4))
    frames = []
    for t in (0.2, 1.5, 2.8):
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(t * fps))
        ok, frame = capture.read()
        if ok and frame is not None:
            frames.append(frame)
    capture.release()
    return frames, width, height


def arcface(frame: np.ndarray, landmarks) -> np.ndarray:
    crop = d4.arcface_crop(frame, np.asarray(landmarks, dtype=np.float32))
    blob = d4.arcface_blob([crop])
    vector = ARCFACE_SESSION.run(None, {ARCFACE_SESSION.get_inputs()[0].name: blob})[0]
    return d4.l2_normalize(vector)[0]


FACES_PATH = OUT / "faces.jsonl"
EMB_DIR = WORK / "emb"  # embedding ArcFace chỉ để chia tập; XÓA ở bước 13
EMB_DIR.mkdir(parents=True, exist_ok=True)
EMB_SOURCES = (*gc.VN_SOURCES, "rtfs_real")  # rtfs_real: chỉ để ĐO trùng người giữa các tập
FACES = {}
if FACES_PATH.is_file():
    for line in FACES_PATH.read_text(encoding="utf-8").splitlines():
        row = json.loads(line)
        FACES[row["sample_id"]] = row
todo = [k for k in SAMPLES if k not in FACES]
t1 = time.time()
with FACES_PATH.open("a", encoding="utf-8") as handle:
    for i, key in enumerate(todo, 1):
        info = SAMPLES[key]
        if info["split"] == "test" and not RUN_TEST:
            # RTFS đã biết tập (chia theo metadata): không mở file test.
            row = dict(sample_id=key, n_frames=0, n_faces=None, boxes=[], width=0, height=0)
            handle.write(json.dumps(row) + "\n")
            FACES[key] = row
            continue
        frames, width, height = probe(Path(info["raw"]), info["still"])
        boxes, embeddings = [], []
        for frame in frames:
            face = DETECTOR._largest_face(frame)
            if face is None:
                continue
            boxes.append([face.x1, face.y1, face.x2, face.y2])
            if info["source"] in EMB_SOURCES:
                embeddings.append(arcface(frame, face.landmarks))
        if embeddings:
            np.save(EMB_DIR / f"{key}.npy", np.stack(embeddings))
        row = dict(
            sample_id=key,
            n_frames=len(frames),
            n_faces=len(boxes),
            boxes=boxes,
            width=width,
            height=height,
        )
        handle.write(json.dumps(row) + "\n")
        handle.flush()
        FACES[key] = row
        if i % 100 == 0 or i == len(todo):
            print("Dò mặt:", gc.eta_text(i, len(todo), time.time() - t1))
detect = Counter()
for key, info in SAMPLES.items():
    if FACES[key]["n_faces"] is not None:  # None = mẫu test RTFS chưa mở
        detect[(info["source"], FACES[key]["n_faces"] > 0)] += 1
print("SCRFD có mặt (nguồn, có mặt): số mẫu →", dict(detect))
step_done("5_do_mat", started)

In [ ]:
# ===================== 6. Chia tập theo nhóm (seed cố định) → splits.csv =====================
started = time.time()
vn_keys = [
    k for k, s in SAMPLES.items() if s["source"] in gc.VN_SOURCES and FACES[k]["n_faces"] > 0
]
vn_emb = [
    np.load(EMB_DIR / f"{k}.npy") if (EMB_DIR / f"{k}.npy").is_file() else np.zeros((0, 512))
    for k in vn_keys
]
vn_groups, VN_GROUP_REPORT = gc.person_groups(
    vn_emb,
    [SAMPLES[k]["source"] for k in vn_keys],
    [SAMPLES[k]["video_key"] for k in vn_keys],
    threshold=ARCFACE_THRESHOLD,
)
vn_split = gc.split_groups(vn_groups, seed=SEED)
for key, group in zip(vn_keys, vn_groups, strict=True):
    SAMPLES[key].update(group_id=group, split=vn_split[group])
print("Nhóm người Việt:", json.dumps(VN_GROUP_REPORT, ensure_ascii=False))
if VN_GROUP_REPORT["largest_group_share"] > 0.3:
    print("CẢNH BÁO: một nhóm người Việt chiếm > 30% mẫu — chia tập sẽ lệch; báo trưởng nhóm.")
# Nhóm của các tập không được trùng nhau (cos ArcFace giữa hai tập phải < ngưỡng).
_with_emb = [(e, SAMPLES[k]["split"]) for k, e in zip(vn_keys, vn_emb, strict=True) if e.size]
CROSS_SIM = (
    d4.cross_split_max_similarity([e for e, _ in _with_emb], [s for _, s in _with_emb])
    if len({s for _, s in _with_emb}) > 1
    else None
)
print("Cos ArcFace lớn nhất giữa hai tập:", CROSS_SIM)
# RTFS chia theo video YouTube + donor, KHÔNG theo người: đo cos ArcFace lớn nhất giữa calib_fit và
# calib_ltt của RTFS thật (không mở tập test). >= ngưỡng gom = có thể cùng một người ở hai tập.
_rt = [
    (np.load(EMB_DIR / f"{k}.npy"), s["split"])
    for k, s in SAMPLES.items()
    if s["source"] == "rtfs_real" and (EMB_DIR / f"{k}.npy").is_file()
]
RTFS_CROSS_SIM = (
    d4.cross_split_max_similarity([e for e, _ in _rt], [s for _, s in _rt])
    if len({s for _, s in _rt}) > 1
    else None
)
print("RTFS thật — cos ArcFace lớn nhất calib_fit↔calib_ltt:", RTFS_CROSS_SIM)
if RTFS_CROSS_SIM is not None and RTFS_CROSS_SIM >= ARCFACE_THRESHOLD:
    print("LƯU Ý: có thể cùng một người RTFS ở cả calib_fit và calib_ltt (đã ghi vào card.md).")

SPLIT_FIELDS = ["sample_id", "source", "label", "split", "group_id", "method", "n_faces", "raw"]
ROWS = []
for key, info in sorted(SAMPLES.items()):
    if FACES[key]["n_faces"] == 0:
        continue  # không có mặt: không vào tập nào (đã báo ở bước 5); None = test chưa mở
    ROWS.append(
        {
            **{f: info.get(f, "") for f in SPLIT_FIELDS},
            "sample_id": key,
            "n_faces": FACES[key]["n_faces"],
        }
    )
with open(OUT / "splits.csv", "w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=SPLIT_FIELDS)
    writer.writeheader()
    writer.writerows(ROWS)
dist = Counter((r["split"], r["source"]) for r in ROWS)
print("Phân bố (tập, nguồn) → số mẫu:")
for split in gc.CALIB_SPLITS:
    print(f"  {split:9s}", {s: n for (sp, s), n in sorted(dist.items()) if sp == split})
step_done("6_chia_tap", started)

In [ ]:
# ===================== 7. Cắt mặt + mã hóa lại cùng cấu hình (chỉ calib_fit, calib_ltt) =====================
started = time.time()
WORK_ROWS = gt.rows_for(ROWS, ("calib_fit", "calib_ltt"))  # ném lỗi nếu lẫn test
PROC = WORK / "proc"
t1 = time.time()
for i, row in enumerate(WORK_ROWS, 1):
    face = FACES[row["sample_id"]]
    dst = PROC / row["split"] / row["source"] / f"{row['sample_id']}.mp4"
    row["path"] = str(dst)
    if dst.is_file():
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    box = gc.square_crop_box(face["boxes"], face["width"], face["height"])
    tmp = dst.with_name(dst.stem + ".tmp.mp4")
    nb.run(
        gc.crop_encode_cmd(FFMPEG, row["raw"], tmp, box, still=SAMPLES[row["sample_id"]]["still"]),
        log=OUT / "logs" / "ffmpeg.log",
    )
    tmp.replace(dst)
    if i % 100 == 0 or i == len(WORK_ROWS):
        print("Mã hóa lại:", gc.eta_text(i, len(WORK_ROWS), time.time() - t1))
step_done("7_ma_hoa_lai", started)

In [ ]:
# ===================== 8. Kiểm lộ đề NGUỒN trên dữ liệu đã xử lý (calib_fit) =====================
started = time.time()
FIT_ROWS = gc.subset(WORK_ROWS, split="calib_fit")
for row in FIT_ROWS:
    row["features"] = d4.video_features(Path(row["path"]))
LEAK = {}
pairs = {
    "vlr_vs_rtfs_fake": (("vlr",), d4.LEAK_FEATURES, "VLR thật và RTFS giả"),
    "rtfs_real_vs_rtfs_fake": (("rtfs_real",), d4.LEAK_FEATURES, "RTFS thật và RTFS giả"),
    # ảnh tĩnh: thời lượng/kích thước file/bitrate không so được với video → chỉ độ nét, độ sáng
    "fas_vn_vs_rtfs_fake": (
        ("fas_vn",),
        ("sharpness", "brightness"),
        "FAS-VN thật (ảnh) và RTFS giả",
    ),
}
for name, (real_sources, features, pair_vi) in pairs.items():
    chosen = gc.subset(FIT_ROWS, source=(*real_sources, "rtfs_fake"))
    if len({r["label"] for r in chosen}) < 2:
        continue
    LEAK[name] = gc.source_leak_report(
        [r["features"] for r in chosen],
        [r["label"] for r in chosen],
        feature_names=features,
        pair_vi=pair_vi,
    )
    print(f"[{name}]", LEAK[name]["message_vi"])
    print("   AUC:", {k: round(v, 3) for k, v in LEAK[name]["auc"].items() if v is not None})
nb.write_json(OUT / "leak_report.json", LEAK)
SOURCE_LEAK_WARNING = any(v["warning"] for k, v in LEAK.items() if k != "rtfs_real_vs_rtfs_fake")
step_done("8_lo_de", started)

In [ ]:
# ===================== 9. Chấm điểm GenD (đúng face/gend.py, 8 khung/clip) =====================
from vericlue_ml.common.detector import DetectorInput

started = time.time()
SCORES_PATH = OUT / "scores_progress.jsonl"
SCORED = {}
if SCORES_PATH.is_file():
    for line in SCORES_PATH.read_text(encoding="utf-8").splitlines():
        row = json.loads(line)
        SCORED[row["sample_id"]] = row
todo = [r for r in WORK_ROWS if r["sample_id"] not in SCORED]
t1 = time.time()
with nb.GpuMonitor() as monitor, SCORES_PATH.open("a", encoding="utf-8") as handle:
    for i, row in enumerate(todo, 1):
        result = DETECTOR.predict(DetectorInput(video=Path(row["path"])))
        assert result.model_version == MODEL_VERSION
        out = dict(
            sample_id=row["sample_id"],
            status=str(getattr(result.status, "value", result.status)),
            score=result.score,
            frames_used=(result.quality or {}).get("frames_used"),
            error_code=str(getattr(result.error_code, "value", result.error_code or "")),
            latency_ms=round(result.latency_ms, 1),
        )
        handle.write(json.dumps(out) + "\n")
        handle.flush()
        SCORED[row["sample_id"]] = out
        if i % 100 == 0 or i == len(todo):
            print("Chấm điểm:", gc.eta_text(i, len(todo), time.time() - t1))
print(f"Đỉnh VRAM khi chấm: {monitor.peak_mb} MB")
for row in WORK_ROWS:
    row.update(SCORED[row["sample_id"]])
print(
    "Trạng thái (tập, nguồn, status):",
    dict(Counter((r["split"], r["source"], r["status"]) for r in WORK_ROWS)),
)
step_done("9_cham_diem", started)

In [ ]:
# ===================== 10. Fit calibrator trên calib_fit (ADR-0010) + báo cáo =====================
from sklearn.metrics import roc_auc_score

from vericlue_ml.common.calibration import (
    CalibFit,
    CalibrationError,
    CalibrationStore,
    fit,
    reliability_markdown,
)

started = time.time()
OK_CALIB_FIT = [r for r in gc.subset(WORK_ROWS, split="calib_fit") if r["status"] == "ok"]
OK_FIT = gc.subset(OK_CALIB_FIT, source=gc.FIT_SOURCES)  # VLR KHÔNG vào fit
OK_REF = gc.subset(OK_CALIB_FIT, source=gc.REFERENCE_SOURCES)  # chỉ tham khảo
gt.assert_no_test(OK_FIT)
DATASET = "fas_vn(live)+rtfs10k(original_videos,inswapper,uniface)"
CAL = None
try:
    CAL = fit(
        CalibFit(
            scores=[float(r["score"]) for r in OK_FIT],
            labels=[int(r["label"]) for r in OK_FIT],
            subject_ids=[r["group_id"] for r in OK_FIT],
            dataset=DATASET,
        ),
        module="face_deepfake",
        model_version=MODEL_VERSION,
        commit=SOURCE_COMMIT,
    )
except CalibrationError as exc:
    print("KHÔNG fit được calibrator:", exc)
    if not SMOKE:
        raise

REPORT = {"model_version": MODEL_VERSION, "n_calib_fit_ok": len(OK_FIT)}
REPORT["raw_score_summary"] = gc.score_summary(OK_FIT)
labels_fit = [int(r["label"]) for r in OK_FIT]
scores_fit = [float(r["score"]) for r in OK_FIT]
if len(set(labels_fit)) == 2:
    REPORT["auc_raw_all"] = roc_auc_score(labels_fit, scores_fit)
# GenD có tách NGUỒN trong lớp thật không? (VN thật vs RTFS thật; 0,5 là tốt)
real_rows = [r for r in OK_FIT if r["label"] == 0]
is_rtfs = [int(r["source"] == "rtfs_real") for r in real_rows]
if len(set(is_rtfs)) == 2:
    REPORT["auc_score_rtfs_real_vs_vn_real"] = roc_auc_score(
        is_rtfs, [float(r["score"]) for r in real_rows]
    )
if CAL is not None:
    oof = gc.oof_probabilities(CAL, scores_fit, labels_fit, [r["group_id"] for r in OK_FIT])
    insample = CAL.predict(scores_fit)
    for r, p_oof, p_in in zip(OK_FIT, oof, insample, strict=True):
        r["prob_oof"], r["prob"] = float(p_oof), float(p_in)
    REPORT["calibrator"] = {
        "method": CAL.method,
        "n_pos": CAL.n_pos,
        "n_neg": CAL.n_neg,
        "cv_folds": CAL.cv_folds,
        "cv_log_loss": CAL.cv_log_loss,
        "before": CAL.before.model_dump(),
        "after_group_oof": CAL.after.model_dump(),
    }
exceed = {}
for name, sources in (
    ("fas_vn", ("fas_vn",)),
    ("rtfs_real", ("rtfs_real",)),
    ("rtfs_fake", ("rtfs_fake",)),
):
    rows = gc.subset(OK_FIT, source=sources)
    if not rows:
        continue
    raw = np.clip([float(r["score"]) for r in rows], 1e-4, 1 - 1e-4)  # identity hiện tại
    exceed[name] = {"n": len(rows), "truoc_hieu_chinh(identity)": gc.exceed_rates(raw)}
    if CAL is not None:
        exceed[name]["sau_hieu_chinh(ngoai_fold)"] = gc.exceed_rates([r["prob_oof"] for r in rows])
        exceed[name]["sau_hieu_chinh(trong_mau)"] = gc.exceed_rates([r["prob"] for r in rows])
REPORT["exceed_rates_calib_fit"] = exceed
if OK_REF:  # VLR: tham khảo, không ảnh hưởng fit (calibrator chưa từng thấy các mẫu này)
    ref_scores = [float(r["score"]) for r in OK_REF]
    REPORT["tham_khao_vlr"] = {
        "n": len(OK_REF),
        "phan_bo_diem_tho": gc.score_summary(OK_REF),
        "truoc_hieu_chinh(identity)": gc.exceed_rates(np.clip(ref_scores, 1e-4, 1 - 1e-4)),
    }
    if CAL is not None:
        REPORT["tham_khao_vlr"]["sau_hieu_chinh"] = gc.exceed_rates(CAL.predict(ref_scores))
print(json.dumps(REPORT, ensure_ascii=False, indent=1))
print(
    "Ý nghĩa: với người Việt thật, tỉ lệ vượt B=0,15 là tỉ lệ bị đẩy sang bước đọc câu (chặn nhầm);"
)
print("với RTFS giả, đó là tỉ lệ bắt được. Ngưỡng KHÔNG được chọn ở đây (ROUT-03 dùng calib_ltt).")
step_done("10_calibrator", started)

In [ ]:
# ===================== 11. Lưu điểm calib_fit / calib_ltt (KHÔNG chọn ngưỡng) =====================
started = time.time()
SCORE_FIELDS = [
    "sample_id",
    "source",
    "label",
    "split",
    "group_id",
    "method",
    "status",
    "score",
    "frames_used",
    "error_code",
    "prob",
    "prob_oof",
]
for split in ("calib_fit", "calib_ltt"):
    rows = gc.subset(WORK_ROWS, split=split)
    if CAL is not None:
        for r in rows:
            if r["status"] == "ok" and "prob" not in r:
                r["prob"] = float(CAL.predict([float(r["score"])])[0])
    with open(OUT / f"scores_{split}.csv", "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=SCORE_FIELDS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    print(f"Đã lưu scores_{split}.csv ({len(rows)} dòng)")
step_done("11_luu_diem", started)

In [ ]:
# ===================== 12. Tập test — TẮT mặc định (chỉ trưởng nhóm bật một lần) =====================
def run_test() -> None:
    """Chấm và đo trên test bằng calibrator đã chốt. Chỉ chạy khi RUN_TEST=True VÀ câu xác nhận đúng."""
    gt.check_test_unlocked(RUN_TEST, TEST_CONFIRMATION)
    if CAL is None:
        raise RuntimeError("Chưa có calibrator.")
    test_rows = [r for r in ROWS if r["split"] == "test"]
    out_rows = []
    for row in test_rows:
        face = FACES[row["sample_id"]]
        if face["n_faces"] is None:  # mẫu test chưa mở ở bước 5 → dò mặt bây giờ
            info = SAMPLES[row["sample_id"]]
            frames, width, height = probe(Path(info["raw"]), info["still"])
            found = [DETECTOR._largest_face(f) for f in frames]
            boxes = [[b.x1, b.y1, b.x2, b.y2] for b in found if b is not None]
            if not boxes:
                continue
            face = dict(face, n_faces=len(boxes), boxes=boxes, width=width, height=height)
        dst = PROC / "test" / row["source"] / f"{row['sample_id']}.mp4"
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.is_file():
            box = gc.square_crop_box(face["boxes"], face["width"], face["height"])
            nb.run(
                gc.crop_encode_cmd(
                    FFMPEG, row["raw"], dst, box, still=SAMPLES[row["sample_id"]]["still"]
                )
            )
        result = DETECTOR.predict(DetectorInput(video=dst))
        status = str(getattr(result.status, "value", result.status))
        prob = float(CAL.predict([result.score])[0]) if status == "ok" else None
        out_rows.append({**row, "status": status, "score": result.score, "prob": prob})
    with open(OUT / "scores_test.csv", "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=SCORE_FIELDS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(out_rows)
    print("Đã chấm test:", len(out_rows))


if RUN_TEST:
    run_test()
else:
    print("Tập test: TẮT (đúng luật). Không có mẫu test nào được mã hóa hay chấm điểm.")

In [ ]:
# ===================== 13. Xuất ZIP gửi trưởng nhóm =====================
started = time.time()
EXPORT = OUT / "export"
if EXPORT.exists():
    shutil.rmtree(EXPORT)
EXPORT.mkdir(parents=True)
cal_rel = ""
if CAL is not None:
    cal_path = CalibrationStore(EXPORT / "calibrators").save(CAL)
    cal_rel = str(cal_path.relative_to(EXPORT)).replace("\\", "/")
for name in (
    "splits.csv",
    "scores_calib_fit.csv",
    "scores_calib_ltt.csv",
    "leak_report.json",
    "step_times.json",
):
    if (OUT / name).is_file():
        shutil.copy2(OUT / name, EXPORT / name)
METRICS = {
    "smoke": SMOKE,
    "seed": SEED,
    "platform": PLATFORM,
    "gpu": GPU,
    "environment": ENV_INFO,
    "source_commit": SOURCE_COMMIT,
    "source_dirty": SOURCE_DIRTY,
    "model_version": MODEL_VERSION,
    "rtfs_split": RTFS.report,
    "vn_groups": VN_GROUP_REPORT,
    "vn_cross_split_max_cos": CROSS_SIM,
    "rtfs_real_cross_split_max_cos_fit_vs_ltt": RTFS_CROSS_SIM,
    "fas_counts": fas_counts,
    "scrfd_detect": {f"{k[0]}|{'co_mat' if k[1] else 'khong_mat'}": v for k, v in detect.items()},
    "split_distribution": {f"{a}|{b}": n for (a, b), n in sorted(dist.items())},
    "source_leak_warning": SOURCE_LEAK_WARNING,
    "report": REPORT,
    "step_seconds": STEP_TIMES,
}
nb.write_json(EXPORT / "metrics.json", METRICS)
leak_lines = "\n".join(f"- {k}: {v['message_vi']}" for k, v in LEAK.items())
cal_text = (
    f"`{cal_rel}` — phương pháp **{CAL.method}**, n_pos={CAL.n_pos}, n_neg={CAL.n_neg}, "
    f"log-loss ngoài-fold {CAL.after.log_loss:.4f} (trước {CAL.before.log_loss:.4f})"
    if CAL is not None
    else "**KHÔNG fit được** (xem metrics.json)."
)
card = f"""# Calibrator GenD (face_deepfake) — FACE-04

- Mô hình: `{MODEL_VERSION}` (GenD gốc, KHÔNG dạy lại)
- Commit nguồn: `{SOURCE_COMMIT}`{" (có sửa chưa commit)" if SOURCE_DIRTY else ""} · seed {SEED} · SMOKE={SMOKE}
- Calibrator: {cal_text}
- Fit CHỈ trên `calib_fit`; `calib_ltt` chỉ lưu điểm cho ROUT-03; tập test KHÔNG được đọc.

## Dữ liệu (số mẫu có mặt, theo tập)
{chr(10).join(f"- {a} / {b}: {n}" for (a, b), n in sorted(dist.items()))}

- Thật (fit): FAS-VN live (Kaggle maihongtng/face-anti-spoofing, CC BY 4.0 — ảnh tĩnh, GenD chấm
  1 khung) + video gốc RTFS-10k (cùng nguồn với lớp giả).
- VLR: {"tham khảo (KHÔNG dùng để fit), phần " + ", ".join(VLR_PARTS) if USE_VLR else "không dùng (USE_VLR = False)"}.
- Giả: RTFS-10k inswapper + uniface (Kaggle stplusplus/rtfs-10k, CC BY-SA 4.0).
- Chia: người Việt theo ArcFace (ngưỡng {ARCFACE_THRESHOLD}) ∪ video nguồn; RTFS theo video YouTube gốc,
  ảnh donor gán một tập, bỏ {RTFS.report["fakes_dropped"]} video giả có donor khác tập
  (đồ thị video–donor ban đầu: {RTFS.report["components_before_cut"]} khối).
- Tiền xử lý: khung vuông {gc.CROP_SCALE}× quanh mặt → {gc.CROP_SIZE}×{gc.CROP_SIZE} → H.264 CRF {gc.ENCODE_CRF},
  {gc.ENCODE_FPS} fps, ≤ {gc.ENCODE_SECONDS:g} s, cho MỌI nguồn; rồi `face/gend.py` (8 khung/clip).

## Lộ đề nguồn
{leak_lines or "- (không tính được)"}
{'- **CẢNH BÁO:** thật và giả đến từ miền khác nhau; calibrator có thể học "nguồn" thay vì "giả".' if SOURCE_LEAK_WARNING else ""}

## Tỉ lệ vượt ngưỡng tạm (B=0,15; A=0,60) trên calib_fit
```
{json.dumps(exceed, ensure_ascii=False, indent=1)}
```

## Giới hạn
- RTFS chia theo video YouTube + ảnh donor, KHÔNG theo người. Cos ArcFace lớn nhất giữa RTFS thật
  ở calib_fit và calib_ltt: {RTFS_CROSS_SIM if RTFS_CROSS_SIM is None else round(RTFS_CROSS_SIM, 4)}
  (ngưỡng gom {ARCFACE_THRESHOLD}; >= ngưỡng nghĩa là có thể cùng một người ở hai tập; tập test không đo).
- **Không có deepfake người Việt**: lớp giả chỉ có RTFS-10k (người nước ngoài, bối cảnh YouTube/hội
  nghị video). Calibrator không đo được khả năng bắt deepfake người Việt.
- FAS-VN là ảnh tĩnh; eKYC thật dùng video. Người Việt thật trong lớp thật chỉ có FAS-VN.
- VLR (video bài giảng đã căn mặt, nén lại) bị GenD chấm cao và lộ đề độ nét so với RTFS nên
  KHÔNG dùng để fit; nếu bật chỉ báo tham khảo.
- `scores_calib_*.csv` có thể chứa dòng `source=vlr` (tham khảo): ROUT-03 phải lọc bỏ.
- Ngưỡng B/A là giá trị tạm của thresholds.yaml; chọn ngưỡng là việc của ROUT-03 trên calib_ltt.

## Reliability (calib_fit, ngoài-fold)
{reliability_markdown(CAL.after_reliability) if CAL is not None else "—"}
"""
(EXPORT / "card.md").write_text(card, encoding="utf-8")
sums = {
    p.relative_to(EXPORT).as_posix(): nb.sha256_file(p)
    for p in sorted(EXPORT.rglob("*"))
    if p.is_file()
}
(EXPORT / "SHA256SUMS.txt").write_text(
    "".join(f"{v}  {k}\n" for k, v in sums.items()), encoding="utf-8"
)
ZIP = OUT / f"gend_calibration_{'smoke_' if SMOKE else ''}{time.strftime('%Y%m%d_%H%M')}.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(EXPORT.rglob("*")):
        if path.is_file():
            zf.write(path, path.relative_to(EXPORT).as_posix())
step_done("13_xuat_zip", started)
# Embedding ArcFace (sinh trắc) không lưu lâu dài: xóa cùng faces.jsonl (chạy lại sẽ dò mặt lại).
shutil.rmtree(EMB_DIR, ignore_errors=True)
(OUT / "faces.jsonl").unlink(missing_ok=True)
print(f"\nXONG sau {(time.time() - T0) / 60:.1f} phút. Tải file này gửi trưởng nhóm:\n  {ZIP}")
print("SHA-256 zip:", nb.sha256_file(ZIP))